# Kaggriculture | Zero-Waste V24 Branch Laboratory

This notebook improves the uploaded rank-one tape **without replacing its farm
plan**.

The uploaded evidence shows that the strongest public policies share the same
farmer and market program and differ only in a few hand assignments and five
extra hires. V24 therefore searches narrow branches instead of introducing a
new crop portfolio or a broad reactive scheduler.


## Evidence used

- Anchor: public episode `89549522`, seat 1.
- Uploaded screen: 97 wins from 112 games and mean money about 130.2k.
- Eight public rank-one replays reproduce exactly in the official simulator.
- Five extra `HIRE` orders lose roughly 63–73 coins per game and do not change
  the hand program.
- The natural three-turn branch is concentrated at steps 333–335.
- The closest public zero-waste opponent is the main weakness: only 11/16 wins
  and a mean margin of about +215 coins.

That makes clone timing, the three-turn state branch, and end-game inventory the
highest-value low-risk experiments.


## Safe improvement principle

V24 follows baseline bootstrapping:

- retain the exact demonstrated policy in ordinary states;
- change only actions supported by replay evidence or directly observable
  inventory;
- require direct head-to-head gains;
- reject a branch if broad wins, average money, or worst-case money regress;
- package the untouched anchor when evidence is insufficient.

This is intentionally more conservative than the previous wide adaptive
wrappers.


In [1]:
from __future__ import annotations

import ast
import base64
import copy
import gzip
import hashlib
import importlib
import importlib.metadata
import io
import json
import statistics
import subprocess
import sys
import tarfile
import types
import urllib.request
from pathlib import Path

WORK = Path("/kaggle/working")
if not WORK.exists():
    WORK = Path.cwd()

ANCHOR_BLOB = 'H4sIANyXb2oC/+1dW2+UV5Z951eUeCGRmIi6+NZSHhxTSVC7MTJmSj3IiujEyaDpAAJaPZnR/Pex25S/29p7rb3P+Yoy0A/pomxXnfu39zprrX337t3vX/73xS+TN//4299f/jz5n4u3ryf/fPHu/cXkzevLN/6Y/Pr29e+Tizcv373+5WKyf7CzONiZze5P3l28eD+ZfnP37t07L39/8/rt+8nPr9/8cefO2enh0fKnw6OzRyePn06+nTz/33u/vnj7+8Xbe3+aPL/35PDp03vn9yf3/vPFq1/eXb119Y/fX7z9r4v3V/96fu/HR6fLq99ov/ru2V9/Onz86C+Hx/cuf/voZHX5f9ObnzxdLh9evf+X5fHJ48sXu4OfrH5cHp7d/OTp8vi4/e70/Pz/7k86zfzu2aPjhz9dNvbs2XUbmvY+v7daPj27btfjk9OzH++d97tw9dVPTk8ePjs6G357tFfrv94ZtvLJo6M/P3vS/eN2Qz80r9XSq5fXzRfbPHNaOmzP8eXUr3+j15bBiNZt09Mfl8snRqtuRqn9Ae22PT15tm4IaGbvLbGR89ZkD5r0/dXc9hrRn8xZf4yWh+tlh0druKzXwwSaN1V3QtOsZoi7DT86BBO63ufrV/IWGTTg5kONlX256B6fNdsfN0T8+qn49f2Ffjklh2fLU/HLQxM1M485bwNez5U5Uc0J1lr6FRo71ZZ6+0v92TMO2OAIDhq1Hgm8/7pDj2f2QxdYw2bBhq0nxljrzbw1r0ILfAa3/RyupMetv+w1aT2peP6cOeXjNRyS9fgbK6g1P+2XkQfKXJsINPr9V91vDA7r8FhRh7f/uQUdR8PdzLcx2tcv2eyK44zOKfbwMzq4iK7s1r4Cw9y0gg14v+sLcaRb66kZ3mGTzSHXxwE+2OCDgW4robN04Q86iN5DD47YipfnAa7z4aCw/U5iT7nV3cwhEFbSZAU/suh38ai62cZHJ8fHy6Ozn75fnp49On70H93VFE+k5pnmwMeolyGF4o+y8aEjpTeubGSE728a25yGMBrKDInx9f3TP5RshDOKh6cnT9Lf1iyOVidgYAN3a+vQv/ka/EzIJVjT8IMAtgP8tLQZ8nA4T8PEYrg50mnv+0/But3H7UDdap8F6W778W1Rr/MzW6Vn67PCSjhR1+DPS2d0CHUa3Wp9u9wQ6dta4er4X8a6lp9RtWutb0A/viU9q7ZW89/WjF7VcUQfVuOrgjH2MFtpYyK9p/Z4cXco1LcxwA6ec3JybKRfm4jWQ+DXtCy5qBnN6/DcZxDUM4yzYlQPJ06FOIV2TBX8zsOzyjIKH0pia7VGZOVBowKwE+2m+ur6pB8tU3DxmszskhShlZyiQaY3LPVyhKYlOJLuP2ITOUKrN7Df8L4yvZbxt6H1G/wylCQ0H9ECANA3+N9lhBLRezM56LUX2q0JsEmeUje+bia3LCky51lqRf+zR03NPvqXfcQoH2zCLYnnYQsX24W034YoHd1oLz732F27ja4AyJPQPZtD2LE7fDjjlyjQzMPfm4jYCfBaOy9xof0kxSQRsLNL5081XmfId7TjJGCHi2u0gJ3cX0X75kfsJEZlEfuGwvLgABeF5SN9VyYsN0Kd/fMv8fqXeF2I1zFgvwVR/C2D2XH6sb5KWHyB4SnzeIOxfP2sItb/PMU1hcP/eHj678I/NNadjFPjQ1XSpfRv4uamcGeuHU12pxv2rU7KlGJDl3aJ7yqatrTevGmtN1BPz04PV98tT0//ClUNQVK8EUmuG9/6MjurQb/s9cALC116K95WKPlILHZCI+58Ynv9LJ86RFtpSdH0xvzgYVejnfbp+X4b0HtomDIDj7YPzWrDd64oGWL3QDDtlRASwqhyPhi9qvVlVbpGKDF1u+bnWIh1CsP/MtZpIql03qyfVcpdrs4Dmk89WH7LiECKGndL7g28xhfJdjeQfMzHTj5QYDKm6CCWjgA1dH5FFI1kc2pYzfZF3LWSqNbZDMT3GGbWZfiVGtRWanNad/RMD+R3zj1J2TVKgolUpfM0cXOV4aP1vLX2iRq/ct8HEiA7dMjy+mCWAROZwg5LJyxKPDqq0J7+PkH5T0xDk5U30x/mkde6VWPBXMULNnKxSNpUMBgwAfOvqOFYpKhkswpUMtYw5S6gugRga750dLEDunTZsOxhd6tkD3u1LlgsGw8ZWZccfcrugIqB+GgmWZgo0CwGBh21Qm4IOcfymHxGRQxZOrEfHAXSurrPPjhUK9DAsJEBCzsh+CqkasV3gf4hRFlGQdyZBEHe/Qy/SovMPrkTwbdZAhwhjYI/5mwQfCuXINQPZQE21atgwIlLCKMl9uFd8yy61vC5T+fw7oSJp4P6omESEEfYDJ8dBXM2w1yrmo0WI8L1G5p4bvgGkhlBcWxQMHNmN7F0IMzF2EiiPZq6zzJC9uh4aWc9aYkd9SeGQm6AT9cqldvBZiBWXqYZAdpXAkGWvm0e50+Nmgrtuncnx4ePH1ZNjBiOLcbOOzkWGpa3d7gii5wlbv7SK3CvFJvZnWpqGhzUgRTX60DFzEyhtsVvwPS2kgc+U8ugMUzPcjhX4yOT/w3ZizPskgWTLEPH4PzqaJmfT/cjq64kGxQpPrayxyKxlT7IdXNZBKFrbxYaFqC5aEIN0hiJ0KemVD7CL17zZcasonUbuQyCuWByOP28Y09tOj5JlqLmpeLIygRDlss1qxeZl4ZXqhwa6wCRTMgqt6EV24lTO/VGha4CYfSIti3TqqVqHVyYmTH6IrmO0ue41Bleb70jeGI3urUb7csQE00OWBpvc9tHWiOll5z7Now6L0lYcwBATSxASJxnNWECg64fKTFTFTIoyvybh3L2LjYEB9xI7R4d//lmUW4SIzBUF9ZFpthj/5agS0J00J0y4AFk7izQ0vikjuf0mMpFfHFrtBFeOjVv+kzePlxEVu10VvnsxtMY4bMm0SX0u1XWcmoQYAsB75kwi11hVYjhIjP0/RypS42w4akBYdJftTBp5YGIOp/BnKDP/XX0bq0lTnJXa+G3qrBQtWUw47KFk46xTrd/ilYAwl/JbhAoCW4N1EbmFoki5LjgdRpFmPgjm9Ddwc4aLqyZnJEKMyKT8tFGUWKRFrMIHZD2zhcaj2YDPp9k5NHEeltvDghbFVsvhLZCD+hhBeAEoS9sc6Dd7cbc9l7GZ08H8hjsFIc8oPYgcDnAxphRHQYGAYVkPuP7hEWuI855oTPD4YwTnfikgkVgY8glq5g1nzC00Hv90zAz/TCYEs6HLKNfLZTX/T4GXRHnXhH71C7eDFSpOoY13asMAqFVWePeewzIxsobrGKdZQDN+sMONoxMtbmHlKZjvQ/QTfi54D6pBrnloD5Mx7UcpiAh4/U0Ol9GmGbaQN73FrR1aGbNm+gLVCMQ8CLEzrr+5XIJNUq9u7vdyk4ePvrBxiELSFEqRJVF4qYK6gjnz3xICr0iZfzS/jCtCcQKGEbJWQ0n9KbtgeQmTwaDu4K7k8HibEKY7gdj5QwUNB+4NwRUgil/b2pkH68a00OzJBSV9hocg61LCRZkwxDKFVNT54SKg4JV5TXl26Pt6wWFGdIkTJpKJhqdM9yARgiof/hUMeotFuynRcFxF7ABp90JUzkJDYnRRVHTmwPLhcY5Z6Uaa80neOLzLFm8S1hETpjS7B36WGGkPHZWuWFCGTqF1yPDpAiYMZylyo8GfTwpkZlotST5J3l8iSowAQHMFLynTuI9mnAMb/WrM1Qe52KUynHP26nBpqqBUM0+GYRqOruNEFX3fQJU5Kf5wZjcMYkglzHKCHLNKuE40fULQzfeXYM+iHg/JNWu5JFnQVXNK04RW1Em3QZBrKVIFJIFcCGeVXaqpuJyRBPEFXShiYN3l2EU2I+pAskb09cFmavpgJbggb4VB46utbx1BA/smFW8tanRVusBCDV8VGAPkEE8yyeVIS/QhJK0jFnRWAb1+O8G6WpZXs/SA2G/hVBaIn/N6yUV1Jfmo/ilzz8r4xoQSIJyeyDXOZPJrct47MbLT4DTAHm5E14HnRGisGk64OUFU8dAvA9UNpVNgi6orVOAVTb0SRoMNQldXswi3um6n0CXAQxS9u59IbBN4eBKhZmfBy2K6Fmh2ah2X3WGwgMe0ZQaxie6TUHS1IiBRZiFHmMHz23s5OA8w40Sohti9urbqROpc0JF6h3AxHoN3jEatjbyWhWXulIaalaPE7b+yOn+6OrFGnjb/NbibbcaWIMfeb0NZvWGYvYxULY89YsGwupLLmsdnRqG/RMHniQZQ9oiegj0pkjJIF1OloTZHbqKMc1Li6Lw8zLFHwHoDOAnUBRlXKrZrKz3ESSPemNHZVwVAVfKRuAKLl/mOVgR+flh7BIYSzVvIjsxcjuJAGWfIJjARAiCSuyOqaMTEUeENGhMM0JUdJD54NNPiB22yYKSZkP3GeLQJStjg06+KkwfWODUt5YWTlbpJsg6G0iw0ckRd7WIXtDwQBUVC6F93TMpKFsbeEbQW4AUTPlmGqNxN0z7I9ueFVpSGNo5KpShda4mK4cpNwV+HYQhRZgyTmcTlnF/FD0kGQdPXwYEWa8gayWSxuCSVBZITu+pXQ/pxcV0/Jq114UZAxW0q3SGFnALqQkcq9UyKiUpiUujf4hSOpc4JTukiil5nMPkY3iDeErN6Rebg/BCbA2CkgiZ+2b5c5lOFGJ/WwXtRfuBlq4UUDOMJ4Xtdfb8bjVfr1BVW92vvILxYKGfl4/NNe9ppEmVRDeGdbyVn5GBZzJXetS5FkOa8gb1DKZCHH5Dc1fMGZQCA58jxTh7fIPJCjwCVeQLiq/7EPTugUONdpuPYxjMtpJqQ8xaHXdOh7uYrI2QcxzJYlEpMqLmQVMju3mx+LQf3sweiNfoROPmkBuUWwtnE5Ghbi1LnmmaNEykpYpdhaH1Zx7NPcfMeWhXoAxIyb4UQX4LwCe0HYK2GO2ghrcFsq4OymfpqHFldT07jxv6zav4a7NCAnRI9PycbLj1BqFm7EF4EarYorCJRn1zNFICvHfzV7taBW2GRRVPbWRAMhDmTOunOd+wskXhIkAru3VvQFbD+tjNV0DXYWW9FAme3evRk40RZIN3f6ViFBNZdIIZQ2eZUzVAV+uVrUMq4c1MmuIxYnd1PY2XQdHpSZITxEI8H8Bi8GFdz7gsuviJgIshR7TPBkVcCEtwkLd8LGgxbRiXRRbN8g+VCGWChQdMEHS6hV/HG9Vi0S69fRQxlr4xBl2A+xeHPqbiTXBs7cG5IPMXPRRDKt8I1uWjeZB7Q3gohNIEBy6KrU410M7HqiCmBaNEtO968Uj2gZtC43B7hn2N3WEw7ib5tHyHI17pPhbHqJyEl5EGW6Uu0Yqj7iuNSFJAFp91zs0Pqdsi6lEUcCPS8WNsDOnmax/av1vByI4iAI5Xnb8IyWBENxaK3zAQsy9yVVQNrFBYhcDyzIi+9DxhunL481g8o+JHdQssK6DwjYcqeuRRrtjo87WS7xUg2z1lhUa9DWNXvQlYc15SejXHxiO8MMYPDOu6+7pLcOASlDugImEz3qCjzXZQzQgkG/pFpN8EFmzT49at1bsaqqUQsogM1ozcL0USwX4nZwB7vAYcEMQSqvuxRZEBjANVaoh0QKwCkwiCF+WIai1QkhaGS5MvHwQgtD59tAhHCvnX1UFL55vjHMYwpALsdPi0G2Pmlz/8kGXs5S0bjQWeqSM7UrUNE/7NKbG4cWFIWb0qBKDy/WcEWZ+7CZ8CEFGv42voFcvzZLb2PyLFRjYjmpaJmnAauM10S4w74hKFZIr5eV0BJlqcshAWHFMbKOOBQ6gQw5MkVf7TuoCZSqFhdpYQ1BRSSNCxQsuv9IgTe16QPAtfRNGAGP4q0VOKwnSf8+KxKFLWWPQlgg5jZpr2skxWLzb0BKHLUIrgaHroclqrbuTJQEbpyqI+5TkASC89VYH/IJLOYyQGZ6hV8ze40pcibJCT+s69wUBDru1owumCzGfsc+aT30JwLNXDe+6SiKiOGb6+PBfJXOn5HKuEo+5wMkmwd7pK1zs/MgpoUj9DHmzi/eDeZpS1e+VabkJYCguDJTibRiPR5kMkUnUp1NdN4KCy7Rnr6J3nX/TOo+qdZx9N7wxNOcdgKtKyc1m5c4epkN8IdieEIUqSFrcId0toQwW1XIqaGMMwFjWnmtUqhQCHT3tIo2+MhLEKiO2zeEbxJ9clIzJmIclerGIGZJVKs0UwCgaUEoooc9BLQ2elDDy+VKDZYAQVlSmMoauswTkSLsBHfELl7JOsOyWjkzYPPLFY6kUhRL4A+C0y9PFY1HFhk55ACnRTk+SklpbxbQL0kj6ruhVCdNqB39jQnV6J54GN2fh7nJqiGG+vlu4e9ghXDPFD6Itw1iA6Vyt7RTgTpdW1Ybk1/hbzTIti/px3GjUaNkhP0CvTCWkUaBzNHHVSZNAx6y+ETskx3En99+JTQ3Bl6o0If7Uv+exDcpl9Qy+O+HWL7ELcB78ywb+kU1BooEQ/MEAQM41FSC3cVhp5XiauRldGTUNAKBAXNZbrP9kUlDbLCEBnXxC2goqft9VQEGiBqgJsfISo32AI6iir7Xy+TagbS6k/ltlgTYAtRHdFN9SqSXZA9iSsMiu7YTQkububxdbgolL9BlcpG8kyyI1IeiMaWbOAdHkVrlkYd0J4Wl+AnK2yREgpUaBN6hoESWM9iJR2zWMhfiDGUgHaXHYdLMxxChz58EQ/iFPUIukRKbWqIVh1+aC65ZlQU5p0JUiXYhkV00Ob1VwIVRU8PMeWv2OQikEgtvcCPziiVaqJDpwjF4AKnaqnrDWXoOQoGZcNGNN0p3JdcArAoRRI/TYmVqjD2Q+0hJRloxkwZgwcx/31UAikESopdQcmzIKwIHO6H/YEQGA27WEAiMWAV2bcc+EFvXGlCymEhGsoV7rm7jp6MdWW+50f9e4Yt0ZMujNmvdYC7/7bi7uxtV8HaosFpltQusOW3vFP6uyruQlFiWkm99DcgK5ULtYh7BUskwrUV6hby5VLTNErtXBvapJrSUztRZydZwFVKZ0xgldFKiFHBbEBPTQPsfTUTxaT4uDbt+yLsR7SKFaJuJSwY5lkWrtILzAXBDaJNOxm3BcWHvc2FeJk5ntk6HtwidGlBoDYbVe9iQqMLKnG14KmI9g7Sw1ES758N8nJSOgKsrySpmP5mlRmJWWB5tAQc+ADx5yBgXYwWIihTVJwyqgGjnuUo0KoAkrLdRajJdimXC1F2bkoLh7LnmsBhDwKm1Mos7XAqK5A3ljM/zY+UYVnB/Ga9Aum8dWl3NbG7zPRUiIRBJQP6wWJA0ajvU2yX+NoILreFkBNtMIBnzffEDfMdcTFUW5aa4O5geOgmWTfbcIpgW3jbnhilqpbXLJOOxuskTh78NjvIIL7WByy4xXquR3SWFH08oXO98nT+baLuSfAGmLR4FuhjYVUNRdLuy00PeMeJMbk2SBVTydR6lypEERYX/RqaJMMXghEBWMa5QQFT+RrmJS76NMH15ip6hAoE/biEjkSy4/tlA+5eYbNKPGjNGF1J7ekldvUWmiqkRt3uGO4kghUj7XY+G0JS14EMLde8u7T8oym0NQJ/4K97Kh/YVnlIT781m9weSBLfP2LVjMrjBQB18mTukVInjmpX7JJYL5eEdU+QDItjlUbYXsaIq+UKa2ZeS7sZHpWXGKbAyb+IdzlN4hVU3UcgtTJ1guhFcMsFeFfHzqlNDF/lCuxWdkYw1XuVMOVbod89K2z8nfPwwQ+3XmQFPfgdqQ+c7KzT3dHZuop4NteXYQNoQ/b6D8HsjMBWVKCze0k6RXAaZXLvNL6Dhug6Y3mNJfShm2Uqldw/amc+En7uapdJD5yvuEyWgLYUT9+B8lwtVCJAi5NihlmfSyvOVL/AM0WUSQ7hvpl+Y1UPgVXCQkBaNbnRRZfZIZ8p3N2YiBEkgVU/oqty4nE/LSAusQCEMxpYsukLqRDioAyH0SZkVLz8iSUlnM0XpBpLjVwuBjopZUndCdAujD7jDYHK0qwY2wNLp+FiHycuDpkqjSYQwSHnBDwq/sI+7gsBSA5sslImWBnkCZzyFK/d1DgM7679ZS7hx3EqFWBq1BVYWwCCqpNWj15O8NiWYtTtpJRlhSms7L+BEZYrmkAGU6Mw0pgTFJvgsBRUg4m8/GIdJcNKZuIKGFyXJ6aRzvbw1jZwfYoWudfFK1brmjVfWg/J0VrwI1AlRv41uN74wwS6BpCjsQ/gKPCqVWUuyOus52xBbCxCglLudJnfUEsxxQhDEUgRws0GK18BR96sWJsupbqCOVjRWwRsnQYwYsG7zR39dkgve2vwQYBVIAUh2UbqWp5WNIX9KqJ8ZM3dmIP83SLhV9Wtf13NwQ60aMvhKzENOYBhCxyt9kZmanIE5TRR/iL1hFqY2CD4zDDN2FoYqSitnCw+rZgRbiSRK5W8H4CGF9v5aKxDjGxTTc+c4nkC0T4zfbUw7FCrMygTyOPzT3yWBSFcZzT/PVAvBt8DLOQIU9BU0LBJvfi/Rq72dI7hrGCT6TTVxMfhmD9BAJpmRcHMOHpCzDFu1C/yQO0p8JeYHeaAvcjwIdziuGOgv8GqoEoWF60ykUWzOsopg2SHcTydrdEWTrbHOnNNwko5b3dXoKbRqcfo95DN45ebAf1LY7fwTEGf6yjex8Bswti3oELYtlRZDMAXYABSOJwUoc1TpCry/nrPcdgrMe8opNXIalHtLyMKBpkJ0i4kxAbqSU08z0PCVwlSI3lh3CBnIkz/Eih4lXMUPFfLsl1K0EwHTaakmwNFdmtonIXOSLG4Hl6z2DsJkqpqVhPIeKaynMqvbCrFskbm8qAYOCmJ4xSycSdlrcAV8/RRFfJeaV2h+0QyBxQQIFxTfiYcmLZjdN7GlOwsV0SuBD7i8g1UHKkWbEKm8S5dNESEQbXiEzzosqwPJ0M6WJoVYU6ksb4WNI/TplHBA3qCL1LEuXbwKXmfeteltlLyu+Y4DvJxn/lipCJxjTqs6nwXFl+0Qx/osHMeZRhm1mkrY9VmIAb5uFMF1sCuO1sUGX6eZi2tQFuSgr61E3bhMEQoTaxMoSKuG1jXdbgnXnIwo4QRrbD/42ghL4sNVS6VfV+K4HgDLjNaicphhH0LktohxiLjiFyMR8ZX4KWhePmBeUYYT5CeDhQbYiludWLzsJgy1x22CjNXIuScV62aGYJEwz1SFdAC8SvyHKbhTtCT28bFhFmIQd6j1WdhCEUIc81htGoJ7pvPUg12QGukjQZ0SmI5DbEakDPnIkigSX/pslVaWlXckAY68sGNISaGDUrrqAuGW2GjxRmZ0doRTSz1nti4H6xCxPoqVVrskwid4QZ1UF4INywVtNN4ywq6qVFwZWlW/0kqut0RmL3PCo8DzjVSSwsyCIbSGPT/ml+1Z3VMgJ/MWYigOsG0tKyKkHc3YOULi5qLVlYnTR9EV5Y7MKDmwJ4RcJQJxGjWye3jo43zkbEG+uIdRdjinUlhGQsb7vdbZDrwhAt7WaH/vC26nULOXEhRlQEZavicIcvGuVsOLpvKlBDM71EIBkRniKsUC2ZmzBNI3VI05VEhc8QKoj1UGBxqhYF1mLE0C4yJ4QIQ0N85k0SsfMnFZxlbMq6281jUvPC0p7S6mI2ahTwYLhc7SJ95Pv1epCMwMI4M6X+fc5udvRcAuMsLo0vLm4MU2JFn0hxwWUc2mI0j04YF4FIDR90lvGuGK+SZPmEQBKis4idd1RHYcCAUNPQswBOq1ADmnNLspMPF7jxmEZzmI4W2K7JLAIHDJPGZ5bBDSEhnu0MhHZSjD4DXribZVav2jFRSEBupECI1IrEkPuljewZAT5Oeg8Knp1J76X4MSleJ9IHIQPF+JyX9N6d8HlJF+1QRCqwIzCe1fq3wEUzuAiiw8AmkpVGonG3r6sOT7P55JgXdJ2judw9Bn6wCu9mH4yBE+HDV+xVr4YynenPrqkwu5qot+ckWZvXGnvaTD9RtuuU2zFtteZc+NO8beQ0K0Gf7W8CoqZViznPM6LYG5PiSqCwWOSmMVa3AoEmWLSgBFXcnMas9WcFf2I5bAVNq1KuQ0Si4UIkTkicoevDswK1rghxhotMrfIbINKVEiA52AwLCBEuFyGywo9M89tncSVg0J42diMUCXuLe6oK1HSxPH5plDfglM58NN5wtoK9VmTu8UIAtSKc8CTSUsY8ixJIx8S+DgFmQedM1ctDoMytaEmjOIt1ptVs5kdHmxm5lpHDnwYur2Sv3m60WvM5oBdy08vw0UrF5X2dxssc+VPpyq11N6UBy4/1MmDiG8St4DEfKMlTJhfIE1s9IL/mrQYOx3oWEQrGhW6PZOX6psYD3l5EbnPhGgEnocHd1C95PloPxXpsrFKOfClTaUmzyjWckcQKaTuWADEjgGzPqxB2Z1OoEnAuCB/g43dWFej7YuK5yeLVkRuRj2r7KTstbTX1N4K3Eq2DwIcNG9xUdqwMgcSseoT5NJIpiI4dysg2lxCBlfB036jQp85WK27dfM1NfMOIazj/C99LhciK8cLWfrADsVllGjLlXYmuHr0yco8VrbvA5xkD/nUF6BGPEd+MNHqBUF5VWOQLGHO1GhrwZYqu0QuasYpYizYB/j0AdSr0kJjKTGyFhLOeqUjtCowB1a2nHrpSI3UzIvCVsaLzV+Lx+w10fAvUOXY/mK93qjc4cCch+HgS2XmsLOPyhx+wn7msQIYGNAFjABIABsSu0VrSK9eSla8yHwk3awzS4K9KlWbJlJex+xjBI6t3j9Rrxjovi2xJIc6QRhvR8wqKfscYhdDVnIhJbpZkDQG/RGmFOxkpCQL0T26VAQE2upIIrztAcEWgILsTGdTtYWzJsaDAB4KNiFJ++8Fnouj/3Mtvh/T8oxIpP1k9/wj8yRTOJUUqsvLYyBpCZRJKbf14n5lsxcdlSPiruoOyZ8HOeRREsmmceQqmf1hzrm1+Ghm/QWOYWFMXq2rjBvBjWRlGoDD0u5St4+3fugJytDvg/PnGuXaBC9XSoS7UzOm7pLQmwapz/hu0UGMwLc6VXAiUPlf8NHQs+sOaaPcFanN9rCZSWgMvyUwJhzR0QSgcS5G8o3kYlFJjyyilhJMECW54CcJEFk+mGIHNolbQlO4rlweXShuFPDdFKunKL+VMJ4QVHbeLvtVmVHL2KJ04CkZl3IWr6WM1zErgXDPgQzZIIVXrxeMyKntm9ZnZsw3dfAEsS65kdP23Y4mfVe4gBoyZ8MdBu6Slr/U9OsO2Aw4h/LJVEOhk37e09uaF4GboUIE86RF6mCX97rESR/VElpoH6n4dSuVBVYyvsprdHdmDT5Vm6U7JYvvV7KISTAz9DiiZdH2HMN3bNl4lM0SUeZWjwK2VCZY+Sa1C2aXRuZOMjxcoOT1CRfCYGSpEr5S6tOXOtvN8TfAcumgZR9QpQ0+QbpJzClLSZdwgYqxi4ARVTzMinXOgLtMz6uPMRMHs5zVKtk/DZGIFdaFwTkbxEPRBBoo+6G4g05pWy5FvB1ayxS8HaM0qLrIynJsuFjo4QCNcUmma8IFi3tTVfI256M7H9kR+0WimMzJMtfIVnv7ddUHpFFZiGQkZ4Y+FGz69YsqsgDvql2BGcBr6qeygzAhi4LIInoFqw/yfyuRJNtxwyRM+Z0nDw9xCv4L3ptunDyxMAmDD0fJFZaHUn5JC40kEbSwkDAJhC6/IEmzdfHwgLMfkUx+Z8w0hQ/BRdMuYhrVAH6G+N1eG5aXFFXsu5ErU4Y5qC4Ial3xv/JSVauVWOnWtuAuwoIZgihYqUuTEP3VNtliqFFo4RWXgCyhX07DLIkITcWlt/+I5OTlhPh8TG2QrfBWJQ+fh4FIAzOAshAokjlKQB7YlUJKS+7WtRkBH6NkSqjaAsbURmk2KpRBagaaKSjidspllpYzFigckjt9ltTRrkpYw51GqUIMwwki3x9iwVF7Koqdw+aWdkgSc13QngyyDBjALAzliEZWWEfdY2MGEgGnXTt+Cwq5FZhb3Lm60vvjROBNyEov04DVj02LTWDQ4rlTSji0WeF3tqLkAtl2LkcNl70962MjCzy7DREUtIUpFcN8UvTE7Hwtlf2qUKweHnu5gNijTDDq5PLfR9KGFfp+URgzNGXbDNzH6VVFB1LIvj5f1qJNKdIqVpBLnln5S6uaZWmuUcYrWL4U3QsF2BUxESRaNcG5euTN6kUSxI9QMeGcnZRDrZwMLpw1feb8xik9Q7/55Ck6G9QeyZxocBb+FOifOPiFQ84wxGKGFweY4pclCbdGQ1vGa0v+9T6Ql6EvVhvTv3BK+NTeXcBWa7V8Q2q9yrJUKDc6+Kprw7ChFRq4z4QfihHPUpcBChMVC+cE+PjxawsKwneE3f8v/Wb4qXbxrAadXd4F5hu2ZPo20D1qvkpODopbCwZ7lB7tmz7704kMvzu/cufPLxa+TF79dvHr/1eu/vbs/+fn1q19f/vbt49evLr7+053J5f/evb94M/l28vvLV1+9vP6tb367eP/V3av3796fPPh68vrt5X/vT/5+8eqry5PxaPnT4dHZo5PHT7+e/Ntk+vW/PuTtxft/vH11+elv/vjml4uLN1cvur/8/Orzzr++8//3o2CW4RcCAA=='
WRAPPER_BLOB = 'H4sIANyXb2oC/90aa2/bRvK7fgVPwAFiLauyZac5IQqg2owtVLEMSq7RCgJBS7TDmiZVknKsBPnvNzP75kNW7oXi/CGhdmdmZ2fnvdu4difnN2ezqTWwWg0L/pq3l85w1mxbzbOh607oazb5OJxN8Gs6c4e3Pzuu+xv++uiMJ1fNNsNzLi5obDT+Bf+/nUzG+P8Hx52NxqPfHRcA7cbPw6njXbujMwdW/Kqv2LeOT9Wqfat3qlbuW2+6fBmNg751dNyVbCA+/kI++tapgCd+APJNVzAFgN2uyRnMw1DjW+NifDPzbp3RxeWszF+3c6wz2O38pHPY7bytYvG4o7PY65xKFrudnskjA+U89mgxg8du5w3yeDZxHW94NhtNrvDYPE8f8LzG8Hx4PRv96ngzdzQ8JwhzCGBmrjOc3ri/edPZ0J0RjDmkw3wY30wvTRgaQhjH/Ti6Go6924n7i06sPA7QZyAEx/vgTq5mnntzxZg3xwBKfV9O3NHvEwZXGjUgRzPn47QAR2MA1fDGw+kMmHCuAeLwqMHXPJtcfRidO1ekit1Go7EK7i3vIchbyd0fbesx2LYtGPI3UT64SuLA7tNphfdWmIVxlvvxMmCgq3CZ81n8S4N8k8YWTHWQmk7IbmgAMOnneVpezhbMrCN/GyBAxslzzDDOW5zTDLSEgYG1dW0rSeFfgX/vp09ZGV1DJQjAnC8Idb7QUTVMgkMBK5I0zpbGCY1XfS2Cn7PJBQqPY7yzoiBu0axtBVEWWF+/yW2n4bOfBzsZ5zDA+tdvxLpCzz4FqxpcgzRQQdBKEkGwyvalgbBVRML4OYjzJA2DfUlpGFVnAtBLg9aTnz4GOUpfCYaNGfwUl2UgTIjLoJL3dZKFeZjEGR2RZD0D7YTl5lGYcQ3Eaa5HpIJz8GFdzjj7XGi4neAlD+JVS9oKERKL2XL4HtDFKGi7Zaz1yY9XhnwYnqF2bDl5EnmQenkYBcZ2cJE0+VyiT4AG/b7BGM4TEiIXAcouAsG5j0DoT35GVs+Gm49hvGoW8PFvGwbRipaSR7K5i8Kll4UPsQ97DPStLJNNnGcyZFFUOZvcYtDAqH3pONf8+2IymTr0rSBlgDPCmxHcNGgjvOnBTQe6Bpd74zqC6GQi1r91nHMF+61RFGrNWeGfH4dPfiS0ncuPDcJpkYuWsMs0WRcgcagEh+IvwNGJFOHgSPnywCOTtnlmbGzOgBbWwcA6kvNBBOjE0W5kBKlCJSZ3oyIIR2V+WdgvbC7frKNAmVyWpDk4SDYqTa9kckUXoBvZnxt/lfpM46qp6/a0iaNk+RisPIn2ivFq7gFixG7TVzonqashuQU1xNjlUgNde1rQzvELd83ZZ/M2py7jqbQ9bxVy646Ce/CkafjwSaQAOOIRo232jYyJb8UR+y0NF38xGSApgc9+cALsh0aBDUgS9JMdv2APRv27rKVYsg71BWwTGNSnZ/3AMeKWYt4GNBzR2LHLuNnmSZ0cEvEB7c506n7bukM5fwnXLU0A5l6ESoh8y5Sc9bdBURTKJHR+jhu6UokZcZab9Qoir7eMwM4hsib3YP0shGZ5sOZn+RAld2DzpYxRshYnuVVIYYuJYGNnAkXbk4nQO+u4Gr0+z0rWa9hBjGH5COTNpiV/uBeKVSdt6/iEjKZFY+8H1slb8GkrBvJ3mLUGkAdr/jb5HCuNp6WrIpBM7tRBC5b2whbAGr6mvVUmZ/g/g8m2OVViQ80bnl2u9w5kaLrXqlrhKYxbb9sVUwfWkW16bp3yyR6U/ZdWt4ryoUk5C/4tWj3p0Jj2+8s8fA5aZoJaLNZQVcrE3pOZMWJAJmGhm9kR/dbqB6H9WvWgRalivCEAygNEfK5Lz6Sy04LIElqUJFeuzDC0M6SXtrUF8hJ2TiQWguAWDLKLi2wFVVq8hiJHehFILwbSfLvYgcd/C8j5y8KUqqgMtmXRakUDSqpYdZjCYWWXBlPmSZvk4iiXEVqRk1G95fmrP/wl4O179Pud7KtK8r1H/8EH42FYWfglsPRTxTWPujT3yY/uYY5AfvwRVJxC2hKKJTPJbhEgWGfbEl9aRkKz9TMKrxrH1nNkoSBG1iYOyEYXz9iTarPy12jVXp6G/qqFJp7EbasY4HgIK/SMQBAUFXAKg0ev12tbvd4J/nNqV0coll4MWEXHlqP2R125hsOgrKvgxRpIAeMhDvSpA5HQAqPaMD9olsRU80MFxaDOKzENwMqdgows9xv7FgX7FiOfIclIAyR3lyRRS4fmU16erPwtIJFu2rYZuyEmg/T7erhipVJGi5BbxnYQbYEJnLdlZTfIem91zXhBcpsreS6wpm9ej4dXM9XXXZihTKyKOiFXFjURY2afRc5HF03usYms2uTJX2CT7OQHFgejhXHD/Kj2WeB2OHPcmi2eGlv8Dy5WnxpU4S2T9bazCoI1frRKrQe9nzwHphdzbsMLnY6BxpWWmb0E56YsfVLwAr4hjB/A3qII7C1dBWlW7C19VT0BAkAHxCAN4Zls6x1ZhG2TG7INGBQvOg0CsNGF9ErzNDfvLuhYps543FQZoylXxu+cIRwtFvICpQxECisAUWNLcAciW0MlZ5DHC6bXdkHOtT2u+w1lyZmPnRPNwyvBYo3bh9REr3fF/Y8SuqDDCweoqB8Cotcm6c3c4ZnUDrvQGZMHZkDNNZILZr2ySSmDQqmDVpbmPoe8z0HvcdhlsKMFSYOJ3DiSfs2xa7pxMHjtgKuPNE2gekk38Wvhu5DHI+HyLQqUIN3qUMnMqyp2l8/JqFq5BaOIj2poY3JIIVY05Vm/cpOmrGatcQos86P2NKV9qvMtyuE4JsKYiZLno5CNpZnRPDLVVaVZpN4HFVKChKOtdaR2WgQB48p2v6p21fEOacX/mrGgEvwlDaZwHfea5SxB4cN4E5j7Q181kGSNuT83PmDkW2uwrwvF5jwS5Lkt16QyK3yC9wgh2IqTbctFTVbKsbdEBoUCvkDgM81dfkq4mvOCR/jnttUqr2lrTHcQKmvpLfJnP4z8O8p8UR4GO12zPUIJFVglKRlbT94f6nCHwloNwKpWin4YYIiSGY17PYRLcNMzGXqgWrnoAdCoosTPW8wfaAypdwVM0sWNAAJUtMSaKRSiZIz8oM7HHNYeB2grH3UK8jqwWpVuBQQpjtMuUC5yXyFbriUdf73GC7SW2FCbq4qUsKofOEq/RAOb3K00eAZHGwxm6UbjxyvSw3SRoYHhqw4feWnBzJw5gyKq1tETiSF3ZQthztm8f9SVrY48Dfxsk253hbvCywXeaiEf+64wWRfswocwptJt73BXHcSqwxMP509+GENUM7oFLAHT/FSN+Zkt82KqZjQF7jYhu5Fd6HGVxVItOhayaSYBI58uR5D/QU5dnWJx0dVn3rRpoXxs6Ve8R10IKXiswjqcEWaTpvOsDzY1PebvcnaFLb5uXwVercNB2YkZuiHDmrlnA8bUzIMyRVt0fjJC8LIoyUWjw6DEc6mNuAoo1Sqqde6vVrLbJ1q7VVZQEfM8/x7vjlVuWZCKcZlbg7TP6YiSHrTdfBDVr0tNalYzew+GGPu7spxSklGI7LukcsiPoSaYm3pbTmk0TrqN//NgroUvoZR7h1+FURFspVKXoy32cQWq4Z/p+pW5hVK9hX93EDsfv9+BNGqiMydghucgfcK44UXJEv5lWPX3EVvVfa28vdinSyui/s5rBt6J1DpZ6Jaf/WgTqN6gEVLZXKjuOrYdGoJculGIOLwwnzfP3cl1U901ySd4vJlL73W8TRzS64bqrqSkdTl0f3WmM42c4r2+n6x3Xxt6+Cx1lu+DFL7CL2D30g9oDea6LZ5NxmPnbOZpr04Xxh36HN/1TJtljficpI+7sja1vaoXo++049GSuTJkCQoM4aejf1RvSN2708W5nuep12p8Q7KrwcFRquxTJC4E1y8ltJwSmswu++ja/9olCUU+bLe22e0HQ0HNDeINLIxvB4uXIOKmhMPiVrTfNftRrWLZJd61H3k5Y+/R9/UfpMFCKLgPH/TntOKFhXyj2657bcHa6GaepqVh2hNIhNSMUAtv5ZYQXcIV3puIhr1WWgw0DpXUql8S04z+5phaP2z4lVcnuj8uNerNPhE1iJj4X7/sY2C7+okMYkcFxgFesXjDXbDpxj8BxcTy9WUwAAA='
CORE_ACTIONS = {333: {'farmer': ['PASS'], 'hands': [['SOUTH'], ['SOUTH'], ['PLANT', 'MELON'], ['EAST'], ['SOUTH'], ['WEST'], ['WATER'], ['EAST'], ['PASS'], ['PLANT', 'STRAWBERRY'], ['WEST'], ['PLANT', 'STRAWBERRY']], 'market': [['SELL', 'WHEAT', 5], ['BUY_PRODUCT', 'WHEAT', 8]]}, 334: {'farmer': ['PASS'], 'hands': [['PLANT', 'STRAWBERRY'], ['PLANT', 'MELON'], ['WATER'], ['WATER'], ['WATER'], ['WATER'], ['WEST'], ['WATER'], ['PASS'], ['WATER'], ['WATER'], ['WATER']], 'market': [['SELL', 'WHEAT', 8], ['BUY_PRODUCT', 'WHEAT', 4], ['BUY_SEED', 'STRAWBERRY', 2]]}, 335: {'farmer': ['PASS'], 'hands': [['WATER'], ['WATER'], ['NORTH'], ['EAST'], ['EAST'], ['EAST'], ['WATER'], ['SOUTH'], ['PASS'], ['NORTH'], ['SOUTH'], ['SOUTH']], 'market': [['BUY_PRODUCT', 'WHEAT', 14]]}}
CANDIDATE_SPECS = {'Anchor V6': None, 'Core Hand Schedule': {'adaptive_triad': False}, 'Adaptive Triad': {'adaptive_triad': True}, 'Treasury 718': {'treasury_start': 710, 'treasury_flush': 718}, 'Terminal Cash Work': {'treasury_start': 710, 'treasury_flush': 718, 'terminal_work_start': 704}, 'Clone Quad H1': {'clone_front_run': True, 'front_run_horizon': 1, 'front_run_items': ('MELON', 'STRAWBERRY', 'MILK', 'WOOL')}, 'Clone Premium H1': {'clone_front_run': True, 'front_run_horizon': 1, 'front_run_items': ('MELON', 'STRAWBERRY', 'MILK', 'WOOL', 'FERTILIZER', 'EGG', 'CARROT')}, 'V24 Adaptive Cash': {'adaptive_triad': True, 'treasury_start': 710, 'treasury_flush': 718, 'terminal_work_start': 704}, 'V24 Clone Cash Shield': {'adaptive_triad': True, 'treasury_start': 710, 'treasury_flush': 718, 'terminal_work_start': 704, 'clone_front_run': True, 'front_run_horizon': 1, 'front_run_items': ('MELON', 'STRAWBERRY', 'MILK', 'WOOL')}, 'V24 Clone Cash H2': {'adaptive_triad': True, 'treasury_start': 710, 'treasury_flush': 718, 'terminal_work_start': 704, 'clone_front_run': True, 'front_run_horizon': 2, 'front_run_items': ('MELON', 'STRAWBERRY', 'MILK', 'WOOL')}}
PUBLIC_REPLAYS = {89548972: {'zero_seat': 0, 'seed': 1162142893, 'rewards': [90753.0, 87383.0]}, 89549522: {'zero_seat': 1, 'seed': 1654717962, 'rewards': [136336.0, 142069.0]}, 89550073: {'zero_seat': 1, 'seed': 2094275173, 'rewards': [122505.0, 132061.0]}, 89550629: {'zero_seat': 0, 'seed': 1392790561, 'rewards': [136345.0, 126607.0]}, 89551188: {'zero_seat': 0, 'seed': 1869950003, 'rewards': [125907.0, 125788.0]}, 89551747: {'zero_seat': 1, 'seed': 968994955, 'rewards': [144213.0, 145458.0]}, 89552297: {'zero_seat': 0, 'seed': 282892391, 'rewards': [118521.0, 118203.0]}, 89552851: {'zero_seat': 1, 'seed': 1082294685, 'rewards': [103973.0, 117579.0]}}
FALLBACK_BLOBS = {'Kaito V21': 'H4sIANyXb2oC/+1dW1MbyZJ+16/Q8mLYZXyQBBg7xhPBYM2YGGwcgA9xDqFQyCCPtYMlVhIzZif837eqqy/V3Vl5qaoW2IsfZkTfKuue+dWXmWtra79Mvoyv2je3H64nl+3fRpPlrP3L7R+jz5P2zUxdumt/nM8+t8c3k8Xsatzee769s/2s19lsL8ajZXvr6draWmvy+WY2X7YvZzd3rdbZyf5Bf7h/cHZ4/Pa0/bJ98feTj6P55/H8yYv2xZN3+6enTwab7SefRtOrhb6k//g8mv8xXuq/Lp68Pjzp6yfsXz+//9dw/+3hm/2jJ+rpg+Nz9b9efue033+lr7/pHx2/VT92B4Ovm+1SsT+/Pzx6NVSFn7033yzKv3hy3j89M+W8PT45e/1kUBVJF/Lu5PjV+4MzXc756/6+/tHZQsU8fd3vv9OP1cV5d3jw2/t3RV06FYmK+8VHLPn0TyN0VdLT/tGRLWI3lwuQv2uJXxfxSHVjJqGzwWrtGl9GpPHsL9jy4VKdHr/Pftb7Ibawv+ixWRHPkgWQFBobF08O9pNe4g1MqNFq864kB/zTvBTUJB2eKP39yhTcBIYgV6COsI/yIktTMCvczMBMoLdnxUpjdQyyeIQ2FzSCikLP98/6J64hHruh8pljy2JNp2JdAOeY9eXYSwI+uqF+q4/4xiYX3Efs4cIqrmh6sBPMIIpbQ6hViZZmjssYK4rV6CudI5As1BSJsE92okwKe9qWFt/YrZQ1Az1jyEEc2kzZWirpMLNdBTVKry5Jtgk6JAHnW9rH3K2xF2EM2Y1T9JNUEmnjEJLY7XQPbVLoLwGCiJuEUIQDK162bRxmAqSd+NpTPV6RtP5fqEcHx0dH/YOz4S/9k7PDo8N/51sCJaJwVQkSt2YxuTVyeKeIaTtZbUe2IiFYWFO4CrUmWCEhtCD7VN5RZjG9X50cv3NMdNcQsj5GgAH2eHKsc0X5xeriJwnXEEIWNM4iBwE0HbGdLC+ZV7mgIkKtuLj1A40zjs4Qt6KgGIie4FFTQnmMUAK4q0WtA6TrFYt/s11EmEsRaodbqE0V8FiF768KgSruttkg4Z145Qru9qOCS+m6K1VwHW1Xg3SdzRitLWhRQLgqooprCQOD8X7TtcOxdQqlln34QCnVoLYHY4LFklS7GFGZBSElq+jKNR9llo0Vserpr9dSAJGwrqBGiQwZugh0RwhUdZFWZjQ9S9dFLD+P5oV6kCpBWAlI2SWKIG3aMHU3rHRhEcV6+u2WAOEO1VJ9YQe8ZOgXVbK9Tp+eneyf/9w/OfnXakq/hwKiKMmPmvAj1PvAoF77kdVDvXxOTJgeXCyt4JBoANIltDIvNbgrXTDAkiNowSDCkanD1mawCk0YrK1lhERQhKEiMAU8MrLrbFmg8R1EA181mNKvpWUQpgzYvNKFAdSEUVutmb5kI3fFchhB6wFbs/iu1KzwKCKGUgwp200cCzKLjqvx4yVEGAN4ASuChwGz/1EpjqMUr1Y9jkPfbFZTbgg99lCaV4Ej8/TnpjVpn7JXrFTHQS/YSotbyY6vbnvX2l/TduqhMKvUuw0oegPJhm+YaEEYHNzuYCngrw5/lajGYYg0bWdEqSfIy8AV40Z6l3t2b8nm1qxiHeP7MH+9a+ZD7o1StbilSQ2SJguLOkKgj3GU+1Wo9N2HrdJXHVdLm3znUeN/1PhFGn9R6P0q/Ah83bS+z1H/7kH799BIZCtDOMQeqP2D1WKj7Q8Ed4+m/Refg82Oe9b+favNR9/lht+99idb2/fzeYvmZ/f/TE+NWxiuqK6yrMb172JD5BbV5HmEf3X/H7C+H20Af3r46m0AD6b4/ZHGX++f/JPLrW+kWQiTwFMOxCsSZtP4hN3w45Wj2jWsO6Q/g5QhUlURWqMelQNP74VhtDyXX8YJPNjI1jJ3fHwExwuTcV1wp6raOQC7XVyhz7ri3YMyXOC5As9euE2dm5849gcw3thRU6SenNZPIBQXcKkqk4eBAYciKz4MTinp0CFDznV5Z1jQ8ULdBgn0VoH2C8zk8TBXMemdxfGopligPHCkcYyEMK4abnIIyyIC5IDFcgsLhjw4xtV9nUL0wFOI3dVbILuxNHUsNqGXgbG9EgODAdwTG5DU4gsE+iO6qEJhhhqw0Tz9Bvx2sBgGG8hTgl4LtpdgO4UyWfCYcPKYMmKpMAMm2HYDbRgKSI9sxPBsNT+uSiVuKaxeQyZCQB2hZZh5LBXXUoPUOTQgaMdLtSf0AfB8ABQo2FwFyVGAV48lUtZu5Ektf87jjrxgcyB+o7H9RIQnGFEGJ+48zTud5ZsdbEdqn2r7+1QTpQV3Ny4Ii5/Gt7W4rtyNl8W1O6J4K1Ia3a77vKP3XVgblYj1Nr79UE2OfDMuNoJw/lBRmzeHR7/lbd2QGQLCf7LGhzMieJ3GFDtmSYRahBzagPI4t5OCmKB1QY8EHF2U9T5TQyia1RJv3wV3QndJIZnqK/fQ0iUN/gH2QWHWpEf7b189Gfgjb9bOJ8MfhFMJPQrYCYDRwFo5cACRprbDm048QB6SHI29T0dFlgbvLApG03mUu8xx4iJqSGYMcTzfiG2QMtLOcBEK3+jaRAUIi7IsN6h8slGy8PjkDKcVcBhh6MgqoplTQAyUGwU+P2qosaFBUjQfqBJQY4iZDqIXYcuVDQuisbkqg29T82m/RAfQh5mitu5KsSjKSQ09AA0iSvXixdKAEnRY4wI7lgtq7d4gimsVBL+z8a+YghLDAh62NZw+wBdJXIlAiOAZXp6LoSiAC8KOpBDpnkml8z8QdCEOlGsSs5rPuNn44hMrXTVzqU5gb0azY4heDsJbXDVlH/gJOnyFp650tSCrXeS85t13nWZ6DNR8+l69KKHKxjzwdZ11ge6FnlOOeCSQm8UjRrj3T/h977NVdtxfQm5wNQhgsxUiFIAa4Rcl4lFHJNfiqSeogL/Uyogc/kaO+EHu1ZQ3nsNHj+G9yJtahI8eOFDwQ2sqmEkoxQ4HiMGh4RCJID7G3GqIZoZUfXjVAkcxZzjIxndX2u5QDYiBAre/lNqKwyxECDq4XPJnsF8PFwWgYkY6jtBdw0HMHMLJCUTjUvxbLCJRU5HU+eOBSKdWq0b0ANt8+jJ4mxlSISDuN04veJBCQek8KETH+N9sY/SF6EhIZ4swtKW+NaFgiGv+BcnZ+4ZgkT05I2N1sEj5OgRgutwtvXt+r2koRBQ6MjAszwqGoSdz3tVvjWACjDFXKHz+eAbYFBEckjtxyBoEg4PdkTFxKSS+UNEjdN9QLc9SHH1dCgXumxzYDPwaeYIeS0Uv2hq0Z0hA1lJOXPsWS62EcR5GN1JHzZRxDGwtcdsaNh5xgriTAoh4iPo0MDQOaPjG7R9c/lIjSBTUc3wgB5TYFzvrhh41O9QaCgAEK1GNRkMQBXtU1JYmkhAzkWpnHasBCJx1TNOk7PKsasoPC54RpS6DHFC4sYAKae0uKfUTFJegV7NJsg/tDEIOHuC1s1Rb2MsByKzBSR3mJLWz6czQJo1Z7Vh94AVCQG1D+c3MlJY14iAFtlIO98Sy5yYqBp5nUb1AiQiNKXK15rpwoa2MRqMAgTSwrtBnoNWdqEp5iWCOInBdhqduxdO8564QuP1CSx7AxGV11F48ECpvmz2QFdTZcS+4K6TnuDbane+GidPpfTuYEzwVUNp2Nx61oxffI4oG15zgjQ+JhRo2tMNYM/1NCi0i74DzW3QevN1Mr4JmnG/2GG4I7GC8jjDakNYWe/qtDurqMrgI0LEzwUaHjRH26I3YV7DNACkIFr4NadL0OBNNLS+HRzBaAEQAh8JMuOwK2EmqLwoIJVr9aDSOcteg/NBicqNEvUK4nhG2EIVkVQ2NiDFXoNkOcRzshJS4twHVMTLCvDQAOdjocFNDftlMdhvfEuKtAGCluH1E+RpToEFA+FFoAoDRC/vc8c9AI4NhUAoXgNh4wAqL4gDMmK+Nhd2gon6wIwThwBntNUdGNHVwKwjfa6LOOPOWlYaYcSbjJlGBldpmoiB5MjWcW8OPJIWcRQjSovr2ZJh/LNSTcIBfNNeJ75kr2SLyoO1FvEQ66CoeRFiScRvhrFY5eLGmsND9Dlx2PTPJYExXYf6DXbya3QZoadshiNRD4axtPyKE94oQOl0hfDCoMIiwYbCQCkVbDhLkixtiDJP75LmBgxxCN9iRmyJiZyKuG82sggFEnDSGIzle0TUljDcGJdT1iDyGrX+wGHYkfggUpLtLQN/C6+phpUIaBuqX2UUc2UjpuZ6b/uYr3EAOXRJE1OTGOsiq8hx9XXEqFS4Em6iwmfEEWbhUhVyrMKR7+nAzcfarpQC7lF0GT9PxU5i/g6s9elQzlMLn9uFtzHDBY2KWPAQrKBG0BJKIEMsFFbcCw5jDRJwW/ikLDC4Rppw4Ph/XMIO6kUG1sjWUdNpxZx3BrkW066B0PuC8obwDiTMMjucj4dYfzFXFaVgIMkCyOMXrZ+xFExxvsLCVUUiEaRQ6r0aoHq6E8QIEOMemAyfjjAFhDHHpJoHnGpKt6RI3aa6jtSBRTaNtAmUmxWiHQn9nrzhzMMs5EPLNKkqGQ+clAOHC2IyABfzBARLyy4GaGidK2hGOUX9hlhfpsxD8tNt8ALQdTPi97wZURfto95vAVEOJfUGRJPfuA2m9J1rmakhWjO4kvOEsABOCHwmiYBzEVcLCpAFXEds0IEBZaLSi8GQG1P0o1SQA9sbGM84lwsFdIhJAELzuyXx0Na/bRiK6FGZT1SasP7xOV5QKmdxnupOKDidEdNzGyJO0yPR6XOlvMp0jP0R5WNRHPOAg3/uaNyi88uFJZhhF7hSQWqMkuiNpwtAq53Jn5oe6DE5HhodTI2PSMUBLGVxHhggPTNhOxaADT+YErpiuBpFH5w7F8UAgQmCIO8EMGBlgR/SLGCjCgZviQDPE6XIdfJMTLRJAA2LMZBhbIlAdP8gcNSoix3WVAaeg8EXV+bXkU//dfsQ+iDMpDZxKgRkALwBaCx6h4PFaH8u3zq0VB6SOzPOWLj75+T5j5BbPMuvvNWzLi9BWADPYz4G8DIxWGOega3lUfini40+etz4odiltbmCw7kPmk4pDGwohYUaYrQhd3ltdw5CxHbmpBj2ZpcEWKCc+ooBUBe04jvbALO/GqgXhVhAYC56su6pEci7jc2QZ3SNx4SWsWAno2yT0BeGPFGoD9qTLQo4a/9KdjBbK3wHWiEKnGJREgMG9IixZelRAOd/C1iwzcGKTdcaXP9i8EEV7cMPV0kCgkspA45Xyc6QABz671B9DYSRKJmjP4LGbu+VlAbzCU5sShy8wLCyadGDGnGquZ4KG8RyiYeyITXLQNQB3SeH7vODt0xRqKzitoRLP0MBmExAf3idU8/OCl4LsYt8YeRC7DESrHEoT5S8qy9vt7QsvDghBOKhDrsJ+WUL8E0lAwhJwKeWzC4b0E6JU3JEFVQkCP6kqiUjK4PJMQXJEQo06lhohqQuUtAXOR0Q4tZ97nNmAdMBSf+4469z1r7MP9E3FlMg71MjHGbgVNiLSBvHc0u2RRISpBMR7DFb56IouRAthnf1bcEWncUGH6zVaZWGbPIDhDKmSFPOKCF7dzOAlkBqGvyXd5aA33aoDPBKGkiCIGGWSPQxSGUR0BEFDCSaMtFfMwSoB2YlAXqRLs6V6+PIdtwexInUS+SsZsxE3FyT046A6gSlseNlPOSaaGGXznDpk4gXK0RRc6O+PBC5YxQgfZm+aVzB10C+6pytZBjyoKNA0ciJZotUpx3lObApGMtFQJLDUrqKWtzGrjLLCUELcUSKqs84ntic//0gfmwp109Y5lAj9yZtxFKcfEecC4liLnaqikZRUFJYGR31gnCugoR2sbidjTvLrR0Qpgk5QfJK4ENE2I3YYEc2Ym9kEHHYEShmxW4i0VZ5h1eEdGswynC2a5+iROA+4hpdlyGfdIxIrouv6450A5zBASyJgwLQwUJBtNxaIJrl5hAi/ZYiQ5k024FiNHUE+QEJhcR2ItuGKVOnHMSRtnCbxCthtVEKKkdQkEgEPh4oYATQJ/c/TYZioF6ELgWKLskYL+oQVHYlHF7Ra1sGyI4NNgiEqoY6BcjLbeJifF7d/bE3Y3RywZNHMGk5VNy6vE9qJqfHHItjyETNEOye6RBRDk+vsLgxJRLFzAwME4EAL0cgC10xyLHqsbbjsxOmRxEeZWOKCeTyEI56Eg48HX4D7xhlZmLES1KPRCLuJyL9NYP38VNkr9HblBiYljEQf2gsrKFBQchjHSZdHP0niJtbMF6x+PS6R0RH8AyRb0csam83pQ10q6ibrWCJmMhw+DsxKxF5OqKAqQX27HckPncq8RICghJJPZwiN0IslSNNFTyf6EnTv7bM0kqBuBIhneQmikBBUP3rR8Ji9uCPsRSpYLZwEMReb08l8nizBICz3oiN4o6yDo0Z2BAASEFHcbcC3+Rv3YGbw03zC832DiKMsNOO3xkOU4qmlEcuEItHHmPHSWCEcGXZGgCN3lDCVPo5JeGDHuJ6KNNMWPCSlCQsQCkXD0vEB5jiBKmnIhpnuaeXxGwXh7Nhx5Pz6MS4fC8dsQR4aCEcxIBXGAIk0bvFDeyLvOx8f5FgnMQmbJI3WxYkR4Aoi1UronukMNibxq4WtFqddHNNhnesyzB9BlDno8j3l9QZ+Uu/MqQL3DBwuQJpnKC7fT3QEIGEj8wnRZPLaMBSUd9TCsUkdawCCk/pEAyUUMcE0xWYCgqf4D7BsgEthTMHxHxGgseZ9SvoDxjpV8wv5yT/74a1XhDO5IIMQFTiDmiy8GQCNGL/mda/AVABMIWPXEaNODKcJosVSNEQIjpNGXyFGDhHNlT4chhPDuTmqLKf350KIM5Q2WCYAgmllOKELHwmD3z5hkHYAjp/dGtkG7xHIcwBzEbiDJLGAOABswseYRKd9IvywmU6NZFuhhzJOKaRS/cE7QsM5NxjULUkyFD4iFQOHCky+QXvhwl7uEeIHxZpnhBM0KzCBA3AkOHFoZMzG3NkhSPNcEMaT1Lqi8HfjYsOgLuvA89npl1hadAMDlkJ3oW5lZyVCs02vKBaBLJ5hOB4JbhX56GjS9ZhwxhMk4YKtUDxjRKxQk4Vxxkr4jATvhMZA5XFnv4AsidAQk6Wi4XC7FIxEeYwHOIcThC8qbIKEwYiT3kI9IvkZngk5aOdx90Ak5kfEpD40L96ZoIe/DGCJueN64AoYoczwHGSIxSinFGxvYlgqagRSgYFIy9qRdonll8unYYOJkkQeM7gzCZGEmIa2mDVmEsY5XElyX/VMmbUXHSdEXIgrnsdoZupHnPAx9qAfUPjtxh5k4oQiaFCSuyQSfSbK6D7HA7hQpMXoOKFnJEJGzHs/TO7+KF80BCGILUntcDiSuSJ8Csd9HQC+DEWEI0w2DsI5CQr0uKWfcHAuRFjKqmE5AU+TH8KRsnobwI25/poMWIo2z4UJlUDmD0Ea5ZPGRM65YQEc2DmAGWlS2SEKKxgxBFzFwmgoz1ACryRjEHIgPyppNI3KccxdOkYsCVhwTWOo9dmEY9cO41hqwXwE1dsEatnpMuk9TmiSQlhoviHFAfIGj2lEh51/g7oPdZ3HROUHA4Tr5hpBRDJZnsDguABPM8F2I+EpkDyFpX9nTVKmTzo4FKCwcmADwuFm8Twa5jYzkYQUSyz6gkr8QQWQhd+v+DF7BPIjvKK5yaq9vJJZA2c3HnhWxsm6Apxs7wF5xO48esQ25xFLM5W+56h7LqgFxcwclCfiyC5o4ER1/+SlsyX5McS5Fx2lb8VGOD14CTWZcIUFL8bHQWlCIR1ArTQCSB8cTpLXOlq1cpaXd4xJ5PyY8MxslMpFIV6EGyx5WE4Dw42wSBkoNklioyAcVu7QJjuPoKaRfcMIKeqM4RYd3u2IExpIwHw6lY1/OEeJeybRZcQ5BGMsu9nCOcIRl1jEEA9Dt6AaE1lt3QGqeL5rOIolSJRbRH5nR0CgWKBRAGdHAzN5M5TZHJfH5VAXSG4Mnbjdp/mDg6cJECEaAmU5TPpzbAMAXhcSR+MaBBjvgTSCwB0r1UkMn9+YI8oBTWcTBKSW+WTKwW0RXhjJThfLxCClpILAllVfPMmS3Ded6Rss4zzjwCucgxcAUT2zlBBTkgUKPovOqOtWaHNJmhsXSriL0O+69xVwLwYNb/uRhvdw8nuI4vd9Y5H3ABwx+s2OyHFypYmE+VlBqOngsBcxFl/TmUAkUCnFQmOwl/zjlohiDFEsOx6MzPHR83fhPUdTgkB0JVpQcFRSjoONJTcGNz5BHSDz2RlkC+otNGiKvz3jyOciwpodVg9oApA+YL7Z+5hn5DkiQSuRsIsuUGuZ96cUPMOVZtL2zCvMjrDICD8RykMGgyOhLuuC1MAsf09oSAYFVu94xdsS2DTO1YKMZ8U1hmLX34cwRMV1dawwDD5gjOj5ICBkdQYJdLnkpEL1Ua6jrJwrolwlXXmiAKI/OZxNCtSMnJClG5QrgBNilAjSyVnP2OtTV5ibhR/HzencjEcvFIQpYS5BvRi9yQqoB1Wt8iIBN1rfrYWTc2KHMPwDel+yun1bviw7HA3cJ1QgXAcrIUj+3aDULd0Qz3EqdiQRhBAe2tLqSXsRXIwR1YdNdeYn6PHrQQfD1YmVSxdw72wn2dQDQyE+ZyOvSIKUXMfouGe3cwSsGKmVJTbrfD/4LZHh7LvAb+HVS8gU7Xzz+K3DufNe8zaLIvZRdtN538OSju9Tzc3g7EzbTAMs95LE+RwnTPFz0DB93xtM2yyhrTKCFUD0RkgJjEh5hPoKT89KYQ0wnYHnAR03cbML7YcO+HH8lRVPwbMHaLyLw9TD3VIhjIGmM8rioIcl/YDkgs5CSASEMiojkk2hejiGN0EqocIKMoizHsmPpcgkQdN0JcbAnBFZHrA4oy02NMkdnUxWKqF2OFQoHAaNnU6ZdH/DAya4uluA1FIbQbOd6mb+uqEBQSgMIvQhrzcRACBqXmlJTDnOQRKI59WaXkCcCwU32UMB/gkchbodkmDgjk0bbBLlYuwpREhC17LsQDNj5JqWA5ZUkMG8OxnxAMjYhM1AligVhox0SVQE3mYDqhej/5jJoFlph3C83bP3oqKx29FzT6dO57CU224xn8G4K7HCPhxf+96jr/29Zp8WfeQhpaxpLvf06uJX3kMiaiLcG8lA7Aui3vna7VErTHgth0a7XF02HBE72TOIuQOhDkZlJYuZf45tSV+SBx5BAKGE4c13caVY3ZATKrnahFlhUJPD+QWIbMD0nHNShyNEgJD4bYMZdNGgLCyeJj6nm/LjhFwKGQkeCDPTMw064UC2A3EJenJvXNDokKmErHzjmNdL3P4858anFTpW02OZdYQK87CABZ9Ny+aH9IQqH3edEIVnYCDNImg2YqoYwXEcTMQWhCEF6xIjEGY/OIIko/nRMKWEnd+DFrEY0DiDf8tZqlDCDAZC4gEH3as1CHxSsRMYXQpG5CB2MFaMRwq9JBQdARBGUYljiOum3lPtRgQvQ8JeekjMSUFPnkSyI0GcM0c1F0t81lSEzee073w68/Ye09M8+sU/MF5l73vlVVInIM3mT2iIWElnzmsyW00wsxIPm9kwPY/RJZy0x/2gWKesU/wwV3g4eClg6ZFUFXauXdZQw3ctz9CzkizPQr5eQKjano97Lk4xtEwcysdaFErTF5tkJ5mt4wrQkkGFoKWiasRbOtjZWiHnfKIaQu5LcMxSdpJjLBMMw85zc5eIA5NVUC2hISjKt803BtEcZLHrCq2ngpTb5CIhIi/F6dGumDxLgFjgxCXrQLBiYntGg1g5cbrJlZVjaHtwK3sxnL8dECVEd2REp6SDdrL9v2Eb3rPqOA+NyXRkcAEIT5IYLu/M2jn6tagfCWp6RCEW1S8KKTgbpy53LX6GIfx0Bcy85RlDcwdGoCJSpcHk1UD9QC4hmZUAGk3gwMDPAdj0aHmTdJgJvd1KFuUSf45mZyveroWBYJ8QIK7zO4J0Qn6czzSzdzxeCw/F3aFDYIDbAsBXFa2pne59+9pTDf69YMK9VccUeDixUnl667MVj4P4OLEo7TWF70UASZ6FDRs+Lgy6S7OdkBsMmMqmpYYk2YieaqoziIUoU1l6oI4TcfOqKFTcAJY4QYjryR3Uo/iBRihrCwLx+JECyYPvJvmSjigLdEQFEBShMBUywQeJqccdmdDYC8b3uNdCpxrOBxVF8D5ns0RZwAmSl4jhEgFFWiXAuJCk20x/vdUc2uAHA7DUkBcl2Hf4kAjTfAfSFC7nXA4iAd7QFEXBaOwIXQWdkVShg1DXWHQhmEC3UpnLY7j5sqPi9pkwsDgvEojhSfFztCt3pIRgv2Tr5/x0StLq7Qi7ECRnguAOyf6ln+VO1Nh9CI5TxN+YOWKpkY7hsbH7kaiim10JR74pB4fNVhwy4j0bI8WxPT9oL+6gsMMRQMFxiXMvSonio6YxJzZ4SgJOBI9jPU+KbVivPxP2ehQ8t8uAbleYtSqSo/72o6P+t+Oo/43lqmrYU18SgO2bd9YXpVwKz5F07776VHIe/+h7gXniHa3P4n/yWWwuXStiSFXPZGvgZo93oWPK3nvieAK/DMCdQfWJPypiZl3vijjB/JAhtAbo7+/uNVoJf2PXIBQFkqCRC+988lC/FDaITEzaVic8Ywk2x15UDi0RXFaOqvPZaixNm+BCOhL5UWseIW7c4w42A90Zl4MAXkVRkzi5u2IlZWa4gJ9zMwORB1ZNuDsQc+KcHfWfOrKvAz5NVYTJ+WelOQKpjNhxI7m0cU9FCSgMy5jo4i2CG20lzCnlpgbmOOOen0AHSbig7JgVwpyrjrg1FLQeSVgnJCWkOEBCsMIGQezLmPUgKZd8lLUjjiTqCEtLjcjiNYiRihxfUDE6vVmlDRFH9/i50x49/b8DT39r0YrE6mQAj4I87SuFFR0wYQVA9nPypVxfGnbgJ3Fxv3S6DDWcR9yMiZPRWCCcnlrigEx584ZmFZF0qQx/lgYOoANSkNTsmECvJJIqAf6icJFLJW0a5CUgQtL2x9P1uHsulAQoGbA4q5sKqSXIjNt8pViAHmtfYcAfuI/+eb/RYMHw2kdHNCXADsZ8xo/+iYpRoTt5jqbiXBmo719YzAao76FFjQbGiZTN3sKCcuH+sDjdkCKURoCNuEFQGTm+cVjsvM880XDZCvycEkx0rDBf7c2ntAbRRDooQ2FwHnrEb1Ga8AfqX5CdxIepqPCpUdgT4gALRJRKKpkp+DoCQwf4IOGEMSSMpGzDlaSeq3e0TyBiMtV5nxlhR8CP48CegcLyw3OCB73V8eSB0VazposiieLRCDxwTdq//D4gvO6WgCsYB8KLTi7YuQ/G3LcK4cUC7lzXXXHE3WpOGIQXsQlo40IGjIhy1RCBESNWE9K3IUqKRE0Vhk6XTEDveIVMgMNRzdgBLLthyxBlEkBUHFclcRofGRiqYe94wgOZNknpWHtBPUEPDJaeSeb/Qe2lVdPTqEMIIkCV+5cHXoDy2UQp2oWLTBMJLiSzwHcfovEn0DoTg2MgLi3xxoMMyyidEYdoBoknjafo7aHNHE9UQlO2oyAFlksir/lmOWUTNCtsLPQkhGDjFIhG37mjRsYNqcy6AMFJgD57BknhOnGSLsKSxLuIA2Jjzc/wPeUfAIC3yQo4ZoY04rmNzDKYW7K0XDXXU5bvbCATC6CnUVEkiTTXEO2OZ7JAsR3BMJco49s52sDhxG50/nLLqhgVmZXIlQ3vHRBTkEBJkGqFIGf1TYRHd9uFsDIkNzYccbRHebIi4IaD4UsaE7C25CUCqAUyBRAcflCycpsGd8WgaZ2Ud1HAubJbYXRp7OzYwGIBCXWYOHkQMZ+FJqCjX1mKN1SoxCDxPyexgjL0+eF/IPIGtLNQAZ+zowaerLxzMTRaObRTg0dFvOjjzGDbxIkskB4QuYwHZSbGABQPG5QOCUzsjHuBuF0QIwDesZiRq6FiwUaKJX4EqaBrD1Io6Frxi3Se5jp94MLivhtsYb3is1UiawslxzuSK3n1uWZaGfcGwa+532XJ+jzK9G/mV9A0exTpUaRHkQJFGrTUvwO1mpy2X7b/brXVv7XESF170f577eNkvliqX92vm+aW2pROjh33zo7f7J8dl+7tZfeKFal0v7OVPfCmf3T8tn7va2v/7eGb/aMMQCqEPDg+Vw+t6fVsLSvkdb//Tl/Umk928dfj49O+vtj/9dc1/cH0S8Of90/79Tp3d6o17e1U67e7BdWq092q1KW7k13RZb9o538mQqs38g8lAqsXtrILxT6atMSWFvz09fG7rB2s3vp5/7d+IsC6qWJWl430U+8O//3v/aF+OXnGNFhem9rjP5+8f3vwenj6Lql85ZubpTpnr/xr/+Tt8PTs+KSfvGFaP7t5eNAfHpz0998UIlif2GznAlWk7p8ND/Z/MV9M+yL/5umb4+Oz14d99JPZw7/sn7zpn5wO3+yf/NY3dcork33Ybo5yBb+2zJ3h2f7Jr0qm45NX/RPV+OvJx9e7eztKw06LWu/1ehpsyv7s7u1utnfzm1td+2ZvSz9bvLq1U7q782yzvZf/9Ww3fXYjE+f0qN//px6/u63X/ZNXwxOliieCnc1vx62T/rv9w5Ph4dt/7h8dvsquvj/tDw9fHfWHR8cHakqdH5/8lt3S6sHQfDq7pD735vCteu70XE0rdfWX0fVinM3Hg3196XmrNTxSapPq/eSRHzqt9CvDn4/f//paz9cttcBcjT+2h7+Pl+uzD/+92f5jfLfZVpdGt9fLl29n0/HGi6Sek4/tyWIyXSxH08uxefRqcrlM7+p/8/Hydj5tq1tP9dfsD220rAfUzdFyOa8Xt5EJc3M9uhvrBxbp59M3J9PleirpQg0G85gaFlsb7dlc/Td7Xy/j1tv6z4WqrPVqckm9eTFIXr0YJA+aD+onLQlsAZLXLszNgW6S9I0f29fj6Xpyd6M9Vj3R/vtrXpn55M/RclyvjV0T84wS6O+viUCl1y/HC+tts0GUq2OulV6vFmQeMWWpD0JFLT6NrxxilmqhPqIfBT8xHl8tuN/Qz0IfmUz/HE+Xs/lkzP2U9UapU9MPLifX6lO6e4Bv6cvqE8kz0Ms3s8VkOZlNF5URpXsgH2hm+GRPqlsX15NFOljTAoxuoUvYUiM2LcX8HJRffzr+shxPr9bzmWU+lt3faH9Ur2Z/qUnRLpWT6Ct2RdLPlwZyXla1ze/MiBpdqt/ZzC8aV9e52jvZ6pC8ks4E65n6AmHdvEheGtSHbDEYdL/UZbKbutJDyf2kN/U90/VFR1XE/ellIm/+hbq0eglMrn3ZbN+pL+aPprJn37tTVd/SFbnLPpoUjXxwPvtLfS956uIu/86X7Dtfsu+o57CvmL/VQxdf8jE7WuaTOWq7fZiN5lfDxeR/x227jlpu82ay9nW2koc/ja4/qsesd/7xj3Y3uTW6VEvQIleTkn00efyHdmeznf1Kd9j8rvtO8R78Tnr1q7Tzza5qXVje3qjhWB0DG3oSmjoVa+nsZri4vbmZzZfWwpFe0TWfLMefX6iu1pNZ/9bfsNVfI+xy9te0vNbrK7WVXn9EF1msBvox9fTt9Hp2+cf4KhGovCwUtbVFKCmyyUau39xsr29YL1hVudAvDtr/9bLdsRsqvZsvL8vxvL4M63L1LFDF1m9mD+gb+gn1YFmAskaiH0tVEl2/T6NFomSYy2t/TKZXa5UK6H93k/H1VVJEJunl7Ha6HKrGG46mk8+q/63eS+4lwza1cLY2C8NG/87sma2iVzLx7SawplUhkiku6+tUbnNRdVuiibWsqqePqy8bqcp1M9cuzENp75gBqJYFXUa+1+eCuj9Y/9jn0Zd1VV+tj+kPJcPE3C6UsY1W/ul8czENAW7vtBhuUWrdmsq2XhScTJUNUNDS26Vd0pSWjYyZGtFT9TkzRBZxFcz842qgjr+oJzpqTTMPVYZeLm4+BosrxVgsruVj0n7Oto3tG7kZbV/MTOatyipakTldTo0WXFtK07bEpoXRrstfHXjOkMu5WgzLT+pLcWaS9XJSDv6qfqS+QJbH1qex2iP/HF3fZrpOOkavRnfZzp2o7cnQKSyCdEqrld3M6cqeY8pTmnLSaND4bbXKjfDyZTqsijmpEZ9kOo3nf2qRbuazq9tLvYXtbba7mUGfPK93f9abymLubabISittlc+jyXQy/b2dry7d52oKqAZQ/00+ZapzeXdpVJQtLXbxmtaajPbR/i/rslI5MgGKZlRvf7yejZRKnTRlsiqkwm2W9uCL9OrAXiqKPTxp+fLb6TOXs88346VR0F+aLqisPS17MBQrmKle/ud/GoGtv9c7T7dUDbeedjrqr1SWYjz/o3hga2dHPWFJspHaAemYm89ul+N0lxvO5lfj+WJd6TPqwc12MggXy/FNYf1PZ8u2jWZojUM9oBq+87xbne6mduajWlXU9ov5dtIMhcFaqCPlZc6xDycP6THx0hSuNcrtVJtWtdFbm7bcslUmEUBPTyPJC3vm6vqUdw4b39DPbyZyl/cHZVwlq1zywIZe83q1+8m9i61BMps0wc7AMmuOBzuDRGtL5l2uTxSlVrQWU8+no5sbbSIaMSqrznQ5md6OW/nV/7kdqUvLu3Zp4zaFdwfZbp0Ldq00xlE6ckFF52urtGMPE+1sNP19vJ6VtFFdCP8yi5vu2OpSl9Zbr3Rl/fLTeHyDvGbESV+sKobp0pt8e6B7qUDG6krg5aeZWRTSb5YeGF9bnzP3BR/U5UOfy9rjp5d2PYVfWowZdQF3pOShymZW7vvyMw4NrTpma/q59b1sA/2pvVWXujyoa7cTDMSeScUGWS9hUHs9HR5m/bnI1p6BXpiTci9edLYKrGj2ebScDZejuVqoFutlwGih1i4Acr54UQJ+B8USO1Yq3OU4gcWqS6y1uv5+PfugWrSMzuZbcxlU/vGlgQfStbe7uw2vvTlMyF575+O/5pPlcjx1LaLmxftaRE/7/VfIEqofs9Ta2oMV6PvHcrM6V9usTRpccNMRZ780mZYbNbu3WbpaHhg/VKpYPFsUtVjOR399GM/nd3Zx+c8fqrK07L6uylmbybXGasGTOOlJ+1in8uUBYJFl/yrdqFanqsiWxFB1g6UunWYBJVSld649Wbml5SdbMdJaqT+0qg6sGekClWB1lSXL1tjSi/AaoZ5Z159MbGE1wWsPG8xalWGvHwWQvaaPrdcGJWvK3L140U2mpHrkaP9t9WxyUO6BrHHSD+vGsd5LR0m6ICXoNrSq1WHvfAHTt4w1uWl+m/d0jcfTW1WkPklI3reNzQQifGm9rJTqkulXHihpQxqw1WrM2kJkSUC0kmtRSkS9KARDGyxrXNM+g7RGueGZqv8f1T76YXT5R9kAKJ0CgBaAvmReqRn85nKrdIRwZx8glI4aEHAqMeTTamwNipGbF6gfeKnP0rXj7FqrusuYV5Ntplu6mX4T0b9LMMb/3I4XxsTIXyxEWH5K5kn6tjFN8xdS9c3Yp2VNrjaOSpha/o0CrUoUgNqgKr2VCGO98ZP1Qm1/W9xeV7SEDeCJZIc1tazcNacPyUNI1xztH/S/h57Jx668e8qv3m8fVaaoWQo+z/4cf9b4kLJFJvbZkVoNsl1HP1MGIRP+qyZuKC3nB/tgZi0hIKd3Sjc0l1Nf7+j6W9c1MVxf/8G68XXFx33FiQ923nelbl4lmppqj4vZjbk6VVen+qreKq6S99X/7/LlMz8inFJnhIUU+r38SHD6pfTexfRuwDmzyp69mKqt4j/UkD86PvhNK9NZx4/Uc6lZrR/OTjsYEGqGgOp9QE3VhKtWkyhhrKTw91TNlqEBcHI2R/px6+5aBZ2vvGAAAOC5tEADGtqlqWZLZDNg7EXKY8u1rmTEp5tyaeCns620BTr2O9Pu0I6Vw+VmLdQ6k7uNzIPJsmem1mY2kzbTmbOZzpR61xNTuJXPlWyq1HZg7katqRvJTp3zPVqVaiZ+hfV6lpdUu/uTI7p0bGWbhlJoamup7oIPs9l1aez8pVS4uerr5cyMiqQzLBtroyJd6jZCyVcdoslx4fB2Olku1hyLd25v1maVU5icT9hEc9W3Hpu/6NqBahKq5UIgXPlQpj1ZJL2WcxbI7vyIdCVSsYw6yKqTduFYZZ0uR5IBWvfZaEhWoO3n6tfkf8fz4ejP0eR69OF6zBDYDD9CRqfSl0ljjgUXsKiJkqsXHHOUkql6jP5OltOf3x8evRqq5ffsver5zXZ64eD4+B2wltYkKar66vDXNefzWUObRQDtE+DTqQ0T2owQAQhszsLUErdp0hInx4CwUOlGrZ/e1ddXA4TDC6m23s39ScEbu3uaXMqADsdgPLAnt7pjt9mP7e4LSEUuVKfkJc2IgUwKyTm0XqYrT9orty1hxsDBUXVwQFhSQZMn63EjjNFBkkkAPlVfWLVkyHZXr4UuI+UdxBc+mbCxJYeXVO40YhdaPvZNlL5UBdXHGNfDv2bzP+qsvRgqG0eFzmYRQ894gdvJDShN4Ei6yDW5QfVYgqsjOr5r9NeBD6Gk3BwZQ2QddmPYzPg1G3FUt9LqDm3ruivUr9TMUia7/oj+3496jJDtXSirA3C8uFUPZMisTgl01kopuKUhxB/FkoEp1LYcnwaUQ3r4g/qnqwCtG1v9y9rQsc2cs5HXdbALo2AMSotmdupgnZiMJvPUeC8xWMpm+3rF70ZJBPjcbMAnJeqrt2p5Tqq+SLnPFSBqoyU9McmfraHw5k6KwlsLc9HY5cq0qtYHgGhsbaYFpq2tG6D4HtAUpY+mxzrpeWxSi2qnFdWp7mVblVMw66DH/PQ725kqWyBBfdND1nInaZp43nC6v8wJj1rmzHuF4MmNzOniIuujgSZbpUX8YH0g7eiEyGF99if4s9l5y8ULc9s6V7dOlnICjVUM+yjKPtICxhJ+dFQ+Uda9VT2am82hk+TaEu4YkMSgTMEpqwrWjl8eodQohc7XasPVcZpmNR+sh7WwszfrC7wzt+ScOTu2XYyUUkfw7mwPQNUodUZILA5IOqbN7cS47NS/nVNG7cWfoH3UKR/lE58S3UNXd63lpHmkelRpc6mQVOANBaPAw5yN/EytUN4KfnvaDvmb7o5wk3czPqouzGKjFsXtbsHc0zrbtyanWWkm09H10KJMqqbffb5r7IpP87H6zLVuju52wphIv/9TxqbtbedraPGl0rzMPrq9s1vqsoJxYdN79B1TZ3U5L7/aXaYTQULGhRkdNn0kL6kkZCK+ZtLsFU9sDFo0McP8LJPCxvPPyYcXf43HN9RsrTjnFmQt1e6xJur4i3rS0KWLw7hslpAzA7bbmmBulafyhnWmB09DeKa0CmrjwrDTbLelzfaH0WJc9Z96qm/lBD7GFC+b7aWtrjTF25a3VNYNL2BGWHbBMcsL4auuKGldsxlgXlIKSU4ESyueD+xyGz1d6EVhPv5zPF+MX2qYoWDDDKvvJu4L5r0aYubYCBJfwPl49IdjxuaTtFzSQDLpRr+PpylOodrz4+R32xs9Y0zm/u2bIHsymXcVPl0FAtGPWKt6oevo6isz/aA/3D84Ozx+e7qReBm2rMUjoZiNE8jR4mS+tMQqmgxwvU8u2x76+n27hdSly9nN3dMrtejoH2V5LvTjWZMy2fzpsxQttfwYwkXLH3SZYOYBlsZjHiWXWojB8H+AurJ6ElgCAA==', 'Kaito V20': 'H4sIANyXb2oC/+1dbVNbR7L+rl+h4oshSygkAcaueKsIlmMqGLyAl5ulKJUCsqNrLLGSyMabyn+/5/21p5/umTkCfMmXYJ23npmeme5nnu5eWVl5M/5jdN2+vfv1ZnzV/nk4Xkzbb+4+D7+M27fT4Kev7Y+z6Zf26HY8n16P2rsvtra3nvc66+35aLhob26srKy0xl9up7NF+2p6+7XVOjvZ2+8P9vbPDo6PTtuv2hd/Pvs4nH0ZzZ69bF88e793evrscr397Lfh5Hoe/hT+48tw9nm0CP918eztwUk/vKP4148ffhnsHR282zt8Fty9f3we/K+XXTnt91+Hv7/rHx4fBX/sXF7+td4uffbHDweHrwfBx88+xO/Mv3/x7Lx/ehZ/5+j45Ozts8uqSOFH3p8cv/6wfxZ+5/xtfy/8o7PJinn6tt9/H95WF+f9wf7PH97nbelUJMqv5y8pyBf+GQtdlfS0f3hYFLGbyUXI3y2IXxfxMBjGVEJjh9X61b+MTOcV31CUj5fq9PhD+md9HHwL+ybUzYp4BVkISSnduHi2vxeNkkwxqU6rzbuSHPSf8UNOXdKRidLfq0zBdUIFpQJ1lGOUfbI0BdOPxzMwFejoLF9pCgPDLB6u3UVpUP7R872z/olJxX13VDZzirIUplO+LpBzrPBm30sCr93UuNU1vrHJRY+RWF1En8u7nhyEWIn8tpDqVdDTQr30saIUOn2pc4SSBU0RD/tkx8ukKE7b0uLru5fSbsAzBiqxazela6lmwOLtyqlTenVJ0k3QIAk535Ixlm6NPQ86VOycfJy0kmg7B0hS7Kd76JPcfnEQRN0lwBB2bHjZtzG4CZR1YutP9WSfxPZ/bh7tHx8e9vfPBm/6J2cHhwf/yrYEJKJyVXESt+YxmS1yeqfw6TsV+g72IhDMrStMHy1MsFxCakG2abzhm/n0fn1y/N4w0U0qVHgZAAOK+mRY5/Lv56uLnSRSR4hZ0CSLHAXQdNR+sv7LssY5fcLVi/PbPtI5k9gMfhtKisHYCRYtBcajhy+Qu5rXNlC2Xr74NztEwF3y0DreQ23qA09N+Paa4GjibsUbJL0TL93A3XoycJGtu1QD19B3NUjX2I3e+gKLQsJVHk3cgjA0GG83XTsSXyc3asWHD8ioJq09GhPMl6Tajx6NWRJSKny68puNMSvGikTttLdrEUCkbCtpUTIqgz/B7giOpi7Ty4KuF9m6jOdn0b3UCKIvKBtBGbvgE9CndTN33b6u/ES+nj7eL1C4Q/WrtrAD/2XqL/Tl4jp9enayd/5j/+Tkl+V8/R4+4MVIfrKEn6DeBwb1Fm9ZPtQr58S42cH50kqqRAOQLrDKrMzgrnbBIL/swQomEY7UHC5sBsuwhMnWFpwQD4Yw9QnOAPeM7Bp7luh8A9HA1gxG9rX2G8CVIbtXuzCQljDrqzUzlmLkLl8OPVg9ZG/m79W6FRaf8GEUU8Z2E8eCwk/7tfj5L3jQAf4DS4KHCbf/ySj2YxQv1zz2Q99s1lJuCD22MJqXgSPL7OemLWmbby/ZqPaDXoiNFrOR7d/ctm61vaVttENpVql1HyB6A2TDN0y0AA6HdDhEBvjrg580prEbIo39DC/tJHkZvGHcyOhKz+4LspktK1/H+DbMX+uW2ZB7vTTN79e0DkmTH/OqIdTLJMb9Mkz67sM26auBq6VNvvNk8T9Z/CqLP//o/Rr8DHzdtL0vMf/uwfq3sEh0K4M7xO5o/ZPNEqPtDwR392b956+j3Y57tv5tmy1H3/WO372Op9jat4t58xZn9//MTvX7Md5QXea3Gre/8w1R+qkmzyPsm/v/gPX95APY08OX7wNYMMXvjzT+du/kn1JufSPdAlwCSzmYqEiaTWOTdsOOV85a17TtkPzpZAxBU0XpjVo0jjy9V6bRslx+BSfwZCcXlrnj40M6X5iO68IHVdXOAcT9Ykp91lXvHshxoecKPXvpPjVufurcH4S+ibOmaCM5C38SqbiIn6oyWTgYdCqy/MXklNKqDkw515WdYVHHC3UfxDFahdovOJfHwl3lpDd+TkY15RLlkZomcRLcuGq8y6H8FkiQQ35W+jFnyEPiXN3XKUSPPIXYWb4HsuPLUudyE1o5GFtLcTAEwD3YgLQenyPQ7zFElUoz1ICPZhk3YLeD+XDYSJ4S9Zizv0T7Kchl4XPC6XPKqKXiHBhn3430YRCQ7tmJkflqdlyVSt5S2rymXASHNlLLsPBYyq+nRplzbELQjpVpD+wB8nyAFMjZXSXJUURUT0GktN/gSa18zvOBvGR3MHGjvuNElCcYXpSTD56Wnc7K3Q5xILVNs+1jqsHXnIebF0TET5P7WtJQ7sa/JfU7vEQrIotux3ze0fsmvI1Kxvoivv1QXY5sM843Anf+UN6adweHP2d93ZAbQsJ/us6nKyJYncbkO2ZJhFqGHOxAWZzbaUFM0rvAmsCji7rRF1oIebcWxNszwZ3UVSik0HyVHlqapOFfID4oTLv0cO/o9bNLe+StsPPp8AflVGKPArYdYDSyVQYcQGWpbcumkwyQpyRnc+/jrMja5J35h9lyHuUhM5y4qDpSmEOcrzdSdEgFZWekCIVtdm3QAOBRluUmjU8xSuaen1wQtEKqEYeOLCObOQJiqNoo9PlRQ51NKUnefaRJgHRIWA6i52HL1akF6GypyWDb1XLaLxgAfJip6uuuFotCQWrsAagTUarnL5cGVaCjoBfcsZxTb/cuvYRWUfC7GP/yKShQC1ptazi9QyySuhGOEMFz/nsmhqICLnA7kmKke66Vzv5A0IQ4oNAkYTOfS6vx+SdWmlpmMp3I0fTmx4BRdsJbTC0VH/gpBnyJp664WZTXrgpesx67TjMjRlo+fatR1FBlfR74ms66yPBCyykHbnHkZsmIEeb9k37e+mxVnPcXyE2uBg5stlyEHFADcVEqHrVHci1fegIl/EUrI3P46znjB9yrUTSeIUZPEL0om1ogRo9UFP7QGiUzcaXY8QAxqRoGkQDx0edWA7qZMvXpVYvUYok66PS7q+13qgVAUej+11JbeZgFpKCjvwv/dI7rkaIAKGek4QjdpA5q5hBPTgCdi/i3XEaipjKpy/UBlFOrNcN7gm05fZm8LEyp4JD3m6cXPEihqHIeCNGJ42+2OPqCdySkswkcbW1sjSsYYpp/TnL2HhEssqtnZCwPFin/TgGYpnBL65HfbRoKUaWOdEzLswQ1tGTOm8atEUxAoHO5wWePZ5Bd4SEgueOHrAEYHOKB9IlLMfmF8hHBY4N6XmQ42oYUKsI3JbAZ+TZ4gu7LRM/7mvRnICBbME5M+5bIrKRxHsEwoqNm5BwTW4vfvqadR54gbqQAMhGiNh1M6QGGb8zxweU3NYJEUSMnB3JIiW2xs67rUbPBrEEAINmIajYaQBTsoawtTRQhFiLVxjZWExAY25iUSdmRedUoDoueEaUhowJQpLmAcmmLQ1IaJyovQa/mk6Qv2r50OXig185Sa+koB6KyhqR0mJHULqYzU5s057Vz7aEXCAW1jeU3C0ta1oiDCGxFAfdg2TMTFR3Ps9AoIBEpnYKrtTSEi+1lNhsFCaSRbaVeQ63uoCnlJUKoReS6TE/dSqR5z9wgcvulljyCiSsaqF1/IFTWN7skK6izbV5wl0jPMW20298ME6fTezyYEz0VWNp21x+1o+c/IgqDa0bwxobEgtQGB4w1M95QaBV5h5zfqvPgrWZGlXTjbKvHSFNgO+N1wGljelsd6bc8qKsr4CJQx86AjU47I2Lt9ThWtM9AGQgFfJuypLGeqaaWVcAjmS2AIoBTaSZMfgUdJNVXJYRSrX4YjUPhGigOzSc3SjUqIPQM+EIIyao6Gh5zrlCzneI4FAtS8tEGaGB0hHltAnKy0+mupuKyhew2uSckWwHIRknHCMUaI9DAIf0oNQHI7IV9qf4L0EhnGBThAhQbj1hhWRxAmPO1sbQbKOuHOEMQD5zhqDmY0dTArQCx16DNPPNWVIZYcCZjJlGRjdoSoiBZMTWeWyPPJMWcRSjKotqOpFt8LDWSdIJfttaJ7Zkr7BF90vY8XyJOusonEdZU3GY4q1UOnq8prAy/I5ddy0oyHNNVWf9gh29mtwFa2pYLIvVQOGtbTwjhvSKExlAIGwzKDSJsGCxEqWjLSYJscUOOYXKfPDdSySl0Q5y5ySN2puK6YWYVDSDypDEeybHKrqlhvAkooaZb9Dls7ZPFiDPxU6AgHi4FfYtvq4WXSlkYbFxmlwlkg9JLIzft3Ve6gwy2JImo6Z11klVlqX1ddSkVKQQbmbCp80R5uKhBplWYsj1tuJk8+7VgAJuMXQFP0/Cnsn6H1Hq0aKYrhc8cw9uY48LnxCxFCFZQImoJhIiQKASV9wLdmMMgT4v8lIUGl4Arp87PJ3XMqGEUUK2KFkoy7aSzDrBrGevaqZwPOW9QdCA4w5BEPoKwfmeuKk/DYpAByOJUr5++F01S32hhK1oI0jQqg1c9NI83wmQJAoy6acDJJDqgzCGu3ST4WkO6NV0TJi0NtFYUqmm0T6jKpBztUBnvbJVnjmY5O0K+aUNhOnRZARApjC1IWCBXDpKQX07U1DhRspjhmI0XFkWRPnfBT7vNJ0Db5oTf/WZAVXaMdh4FpupK7HPKJLl7H0jrPdEyl0OyEgwniIYrAJgU/AiIgn4QVw0LEwOuKrapQ4Iy12xF7sUM0HUvzQQAe2P6zHOJeHAXZAJwgtctmY+m7jX7SGBIaTZVbcLaw+u4oShlcl8YTqo6nFDRcRsjT2KR8XpcGW9YzlGeotwt6yOfcFAefS1TCqt6eJoZhsidClKrl0J3kCZMrXKmcGZ5qkvncmR8OjWYk04AWurgOpgi3LFgO8pBR57MKUIxTR2iz87tiuORQITCETeCGTQyIM7o5zFRhAE35YFmitNlOviGE80TQENizDCNLUhUJ08yh7TCc15XHXBKCp83Xd5KOfXfHEdsgzhDaehSCsIEeA7QmrOGksdrfa7eurRVEpDaM89bu/hk5/sCzc3vFbbfSm3Li9CmAzPYLoC8DIxWGOdkaLlXfikT4w/PWx8UuxS7Gxys+5D5pOrUhkpIWJBmy8OQ95bXMTC3o7TUoCWz1NkDleRHVJCqqB3H0B+c591YsyjcigJjyZN1U5Mg59I/R1YwPJoQXuDFakDfJqEvCn9EqA05kiYP2Wv+S3MxWqp+B9kihE4JKIkEg3tJWLL2qAAF39LerDBxYpNt5pc/2r1QZXsww9XaRKCaxlD6iuIcEeAgZ5faYyiCQsmA9kweu5l7XpfAy720KTh8oWFh1aQjK+ZUaz0DGsYLioaxrXbJydAAPiRFHvPC909TqK3itAYVnsHAZhMQHz8mqPtlyUtJdrFtjjyKXUaiVQajCcWL6up2W8fCqxNCgAB1KlTYrkqIfSEJSlgAl6KYXTKlnxKlkmoW1SQK/ERNUpGUyeUZQXKgoEYdS/VQ1IUq2kLXIwJB7ecWZzYkHbA0ntvGNnft22wDfaOcEtmAxvJJFLfCRmT6wF9YelGTQJpKQrynZJVPoehKtJC22R9DKDrGBQ2h12yTlX3yANSZMiUR8wokr25GeQFSI4i3xENORtMtO8EjcJQUScSQS/YwSGUU0ZEEDTWYMNNfPpVVA7KDRF4wpLlgetjyHbcufWXqBPUrBbORdxc09GOnNpElbGTVTyUumhpls5w6sPACCjQlF/r7I4ErVjEQw2xN83KmDtpl9zQVy6CVCoGmngvJgl5HgfOS3BSCYqKuSGCpX1U9X8SsUsqKwAgxZ4mozjqb3J7y+iN9birUXVujKgH7yZpx5GccmeACcKwlLlXRSEkqhKXRWR8E5wpsaofCsMOck/L2gSxF1AmKTREXkG3T44CBbMbSyiak2gGU0uOwgLJVlmnV6R2arDKcLprn7JG4DLiml2UqZt0iEytj69rjnQTn0MFKAjBg8jFSkC0zFsgWuXmCCB8zRIh5kw0EVnNHkA+QUJj/TmTbMGWqtOMYQh+nSbyCDhvVkGI0LfFEwOOhIkECTWD/WQYMg3YBW4gUW1U1WjEmouxIMrpgoWcNLDuYbJJMUUkNDFWTuYiH2UVx2+fWpMPNCU+WraxhNHX98jqpnRjpn4hgK0fMGOscDIkqh6Y02F2Zkgixcx0TBPBAC+hkRWgm1EWLtY2XHZweaWKUwRLnzOMBgXgaDj6ffIEeG2NmYcFKUM9GoxwmUH8bYP3yUtlLjHaVJiYFTqIN7UWUFMipOIzhpMtinDR5E2vuC9e+npTIaEj+QZKt8LImZnPaUJfytukGFuRMptPHkVWJxMsJSqriNLZbnuLQUeUlAIICIx9XCPUwiiVI00RPB2NJhvf2RRaJ0zASxLPsC6qUEGgcrWh4wlHcVo4iSlZLF0HMxJYMspwnCxiE5VE0JG/UDbDXzI4EQEIiijsNxDY/8ghmAT/NJj3fI0QcdakZHxsPUYunljRWCEWytwnzpYlSOAr8DIdAbi9pKm0Ck/jEjn4jFTHTljwkxYQFCoXCsLR/gNlPokoM2QjLPS09f6MinZ04j5zdOPrlY/GYLclDI+EoAaQiUBBPessf2oO673J8UOKd+CRsQhqtiROjwBVUppUyPNOYbEwTV0t7LUa/2GfAujRkWK5ByB00xZ7KRoM/qTfWVKFHhk4XoK0z5JfvpzoC0LCR5YRoWLzWDQWVHbVIfFLDGsDgpDbZQIEhppim3Exg8BR7BUsVXAtjKo7/QILGWvQpjAf0dapml/JTfvYjW69AMLmighBKnIEmi2wGUBpj173mFRglwFQydg056tRwmiJbLKIhUnCcNvsK0ByQzRUfDtOF4cwcVVHQ+wslxOlKGywTAMmyMpLUhU+EwcdPGMQBwP6rWzPb4D0CeQZgzgN3EBILwAFgEzHGEJ22yfAjZjo1Um0FqzJPKUSl/ugdoeGaGwLqlqYYihyR8oFDORbfwFG4dJS7h/xBvuYZCIIWJSYwAI6AE8dmxmwsnJ2CNM8VaTyh1eWFv+sXGyZtWQOeLy6/JLKiG1BYhO5SwyquSsRWm15SLgJdPkN3PJLcKjLtaDL0GATjKYpw0V4oXzHCV6rJ3DkTFXxmkndSOlC53TguJEvCNcVk6dN0ul0EI6GIcYfgcED4QmkTNAxGnvTmGhEpr/AM5MDB42ZFBPPDY1EfzIs3FuiRLwNcYW6/EbgKRqgwPQdMsejllEIcTUxLhTQQJQaCnrWh7JIoLldOwyYLJakiZvhgElCEGENbwhYLCeMSriTcVy1LZu16xwmZEOJK5DFbmfoJJ3zKPWgHFD7e3INCnFAFDWpql3iiz3jR7nM+gQsiLXrHCS0zEQpy3tthcvdH+cIQhCK3JNrheCRzSfgUj/saAHwdikhnmGwchDMSFLDe4jsMnAsVlrJsWE7B05SncERebwO4sTReUwBLYfdcWVCJZP4A0qicNKYKznVL4CCuASwokypOUVjBiCngyhdGgyJDAV4JcxBKID9UNBqjchJ3F+eIhYCF1DWmel9MODbtMIallqxHUL0MUMtOV0jvMUKTCGHBfEPEAbIGjzGiI66/ga5TQ2cxUeXJAOm2mTQIFJOVCUzqBXmaSfYbhKdI8hRX/l00SYUx6aQqUGnlyA6k083ydTTiy8JCElosMR8LVPgDJZCln6/EMVsk8gNR0dJi1VZRySLF2fEHnpVxsq4CJ9t9QBGx208Rsc1FxGKm0recdc8EtbCYmYHyBI7snBTHa/inrJwt5MeAcy+cpW/JTjhWXmAmg1BY8kf/OCgmFOIEaiUNgDE4kiKvdbRq6Swv6xyTzPkxiMxslMqFEC8QBgsPyzEw3AiLVIBiQxIbgnBEtUObHDxATYNjI0gpaszh5h3e7agLGmjAfFzKxj6doyY8EwwZOIcQ6LKZLZwhHH6JRQLxOHSLajGoamtOUCWLXeNRLEWh3DzzuzgDAmKBegGcDR0s5M0gt9kvj8tgLkBuDC7cbtP9zsnTFIgQhkBFAZP2HFsHgNeExGFcA4DxFkgjCdyJSp34iPn1qVEGaDqdICS1zKZSDu+LyNJIdrpcJQYtJZUEtgrt5Yss6WPThbHBOs4zD7zSNXgJENWySgmYkiJQ8Ll3Rl23QpuLytyYUMIdhn7Xva+Eez5oeFtPNLyHU99Dlb/vkWXeI3BE7xc7qsDJpRYSllcFQdPB4C9yLL6mK4FooFLEQhOwl+zzlqhyDCGWnQxGlsTo2YfwnrMlQSi6EhaU1EoUONhYcWNy41O0gXKfjUm2qNFik6bY+zOGei4qrNng9ZAuAIwBs63eJzwjzxAJbETSIbpEq3XRn1rwjDeaoe+ZNVicYVGQfsKVh0wmR2JD1hWlgUXxnpRKOiVW71jl21L4NMbVAuazkjpDvttvQxhCeV0NK4yAD+gjez4JCBUGAwJdJjlRqj4UOiqquaKqVdLVFwoA4ynhbCJQ03NBlq5TrQBJilGQpFOynonXp66yNos8j5sxuJnPXqhIUyJcgno+RlOUUI9qWuVBADcW3ltLJ2fEDmn4h4y+FA37ln5ZNgQamE+oSLiONkKY+rtOpVu6LpHjKHckSEJIq7a2edpRJBdjxvQRU53lBXrsRtDAcDVi5doF3LraSTr1yFSIL8TIK1MgJbMxOubZbdSAJSO1usJmnW8HvwUVzr4J/JZevZRM0c6jx28NwZ33WrdZlbEP+U3nfQtP2n9MtbSCs7FsMwZY7qWI8zlPmJLXoBHGvjdYtllDWxUkK6DojZQR6JHySI0VX54VYQ00nUEWAe23cLMJ7acO+Hn8VZRPwXIEMN4lYerxYakUxoDpjLo86G5FPyi5qLMQiIAgp9Ij2ZRqh0G9AakEpRUUEGctih9rkUlA0zQVxuCCEUURsDyjzTc0KdVOISsVmB0GE4qHQX2XU4bhb3zCBNNwK5BatBE0O6hm5q8ZGlCkwgCpD2WjyQAAXutKa3LKSQ6SSDyv1vUK4pwruClWBfpP4ijUHJBEA3di2mCTKJdgTwEpCU3LsgHN9FFrWg9YoiSD2XAK8gHA3ITNQJYsFQZmugQNobdZh+b5GD9hMWhR2SEeb7ccPa9o7Jb32tNJ0Dkt5ZZZzOc07gpW2IcTa997irW/1+rTqpc8pJI1zdWeXl7+ynsoRA3SvUEGYl+R9c7Wb/faYBC17JrtcnnVcFTsZMsk5gaE2hmV1Sxm9jW2NWMJDzycAEINw1se4opY3VQQKlxt3Lwwqsvp+gKgGjCec0bqsIcMEJq4bbKCLpuURcTT5Od0U3GcVEihoMADcDMty6CDALJtikvQ00fjkk6HziQU1Rvnol78jue5ND+tMrAa67LoCJXmYRELvpiWLU/pSTXe7zqhSs8gQJpV0KzHUjGK4ziaiK1IQ0q2xUcizL5zBklB97NpSoGf36MWMR/QuIB/K1mqWMIMB0LyCQfNqzUJfKLcCYIhJTNygB1MlOMRoZfA0FEAYYhK7ENcM/Ue9RtIXsakvbSQWFKCHp5EijNBnAu1WoolPm8qw+YLHDufzLzdp/I0T3HxD4xX2ftWeZXoBKTZ+gkNEStx5bwmq9U4Myv5tJkN0/MEQyIpe9x3ynUqOsV3C4Wnk5cSnh6kqohr7YpUjd+1LFPPaqo8K/l6DqlqezbhuTzFsODioBhrVSpNW2xSXGS2jitQSwZKQYuyavhbOsTVWqngfNAMJffFOWepuMgxVwlG4OeZuUvgwGQZVEtKBVX1tuXOIFuDzHdbqfVUUXIbLhIq8pKfEe2qybMAxCInLmwDYMX4jowmsXJwuimVVeJoW3Arez6Cvw0QJUV3FGSnxEk7xfHftA9v2XSehyZkOgq4ACCSxEfIu7B1hnHN2wdBTYssxKr2eSEFp3pqCteSVxjiT1fIyluWOTS3aQTKI1WaLF5NtI/kEsKqBJQ2kYrBnwOI6dH6LukIC3qbjSwUEn/OVmfLn66lgRCfEDCh89uKckJ2nM+ksrc/XosMxd3GKTDIbYHgq6rW1E73vmPtUYd/K5hwb9k5BR5OrlSZ3fp8yXrgHydWlb1G+J4HkOS5m9rIcWEyXFochNxgwlQxLdWlyIb3UlOdS1+IMqrSQw2ciptXRaH8JrDkCULSSG6nEeUPNFxZWxSIJ88UCA++m+RLGrIs4IwKJCiCMBVY4ANi6n41k9I9Z3xP+pvrVOP5oKoM3udilqgIOGHqEglCIqhMqwCMcym6LYzXW86hDX8wQEtNRVGSY8erhJvle6kt4XIu5SAC8AZTFBXa2FGGChozqVIHoSZdNCGYxLCiyuU+wnzFWXH7QhhYXReJxPC0+Dk7lNtaQrBdsfVzeTklbfO2lUNIkjNJcAeyf/G90onqewxJPWXijYUaizSdw2N9jyNoopldSWe+KSeHTVccmPFejJHy2J4dtOdXKYrpCKjkuODcCxlRctTU58QmT0nIiWBxrGdJsXUb9efKUfeC53YF0O0Sq1Z5CtTfegrUfzyB+o+sVlXDkfqaBGyPPlhfVXLJvUbSvcfqo+I89tn3HOvEG3pfxP+Us9hMtpbHlKqWxdbIzZ4fQsOUvffC8QC/dMCdSfNJrhU+q653VZxgecoQbAHax7tbaSuINzYpoSqRBEYurOvJU+OS+yA6MbGvDiJjAZtj1yuHFiSX1aPqcraayNIGXEhDIT+05gFx/R53iBnoxrwcAHhVZU2S1O7yVZRZEAJ+Lq0MBA+smgh3AHPiXJz1Hx3Z1wGfphoi5PyLyhyRVEbuuBEubdJTUQCFcRUTTbxFcqOtpDlFYWpkjTPp+Ql1kMQLKs5Zoay5ashbg6B1T8IaISklxYESQpQ2iGJf+mwHpFzKUdaOOpOoIS0t0sj8MYqRyhxfoByd1qzShoiju/LaaU+R/t9ApH9h0fLE6hQAj4o67UuFFQ0wYQVAtgvyRaEvDQfwQ1zcrpyuwAyXETd94mQYC6TLU2sCkFE0r2tVEc2Q6vBnbeIAnJACUrN9Ar2aTKoA/GXhIpNJ2jTICyBC6Pvz5XrMI+dKAtQoLM/qRim1FJVxm2+UCNAT7SsC+IOP0T/vN5osmF77cEZTAHYI5jN/9A8ahlJ3ygJN1bUy2Ng/t5wN1NhTixoGxkHJZmthSbn4eFiebogIpR5gI2kSVEGNbx4WO+8LTzRMvoK8poQQHcvd1+LmU1qDMJGOqlDoXIeeiVvUFvyhxpdkJ8lhKpQ+1Qt7Qp1gAWSpRMVMyccZGNohBoknjDFpJHUbrqb0XH2gbRIRw1LnfWGGHQU/TgJ7OgorT89JHvRW9ckCo61WTVdlEuWzEVjgmji+/D4gvO6mgivoB8LzTi7Yvg/G3GOF8HwBd6bfTXnEzWaOG4TnsQuwc6EDRlS1akBiRI/NpOxtipKiMVOVqdM1E9A6X6EQ4DA003cCy67bMoRcAoqKY2okT+ODiaEajo4HEcjYJcW59pxGAiuGyM6E9X9Yf2nZ9DR0CAESVJn/ssALWD6bqkS7cpFposCFZhbY7kMYfyK9MzU4RuLSmmg8yrH0Mhh+iGaUeNp8itYR2kJ9QgVNxYGCCCzXZF6zrXIqJmhW2FjsSQhg4+SIRt+4o3rGDVFlXYLgpECfLZOkSIM4YYiwpvAuE4DYWPcLYk/lBwDkZdgAw8zQZjwvIrMC5pauLFct9FQUO+vIxCLoaSiLJChzTdHuZC4LlduRTHPJMr6N2kaqk7jT5cutqGEoMyuolU3vHRRTEKAkTLNckLP6JiKju+1QWBlTG5vOONpDkawMuGFg+EJngraWrEQgrUChAIrDDySrtGv4UAxM60TRRQ7nymaD0WSxi3MDqwUE5jA4eVAxn5UuoGFcRYY39VGNQ2J/TlJIytCXp/+hyBvUzoISPqdHDTJZZedibLZyaqcmj4pk2ceFybbBiSxRHpD5mU/KDHSAyodNSsckJjbmvWDCLoAG0DuWMHM19Vmyk3yJ70Eq6rcHKRT1W/4XDJ6WBn3wwvKxG2JhrfKzVTJrKyXnB1IqefW+ZnqZjwbhfzM/K5L1hZfp38xfTtPsSaQnkZ5EchTpshX89+PeaX/w/uRgv99+1f6z1Q7+W4k81ZWX7e72ensl2ItOjsN/9cJ/nR2/2zs7Dv61sxn8K19sgl863c31+Pl3/cPjo+j58Kb+Tz8Ff0d/hitQeGf0cGijhDdtpo/lG0h4T/jzX63T/tHpwdnBPw/OfqnLt7mxVRSws7FZlLCz0a2K2A0eKIu4tZGLuLmxncvYjf6RyLgVvIqQcXNjN5Sx/z/vj08/nPQHZ29P+qdvjw9f56LmwnVLstVFi/on7bhUzFiw6O6073ZzsTrR54Mvvk9BttP8yz/u/dyP3rwavWU97ba15NXvD/71r71B+HB0T/T6XMTa7T+efDjafzs4fR+1pvLOcmPSR37ZOzkanJ4dn/SjJyKh19OLgcIN9k/6e+9yEQqvSNtbl7p/NtjfexO/Menc7J2n746Pz94e9NlXpje/2Tt51z85HbzbO/m5H7cpa0z64mJ3lBv4V+v9Sf/dwYd3QYenTctljsex/FCiZqV3Jt9Za531T94dHO0dFsdxNVF2xbvjJ2rvL45TQYHXW2ut/eBl/cHb45ODfx0fBR/ttH7qH/WD5aDwW7f1jw/9D+Eq0T8N9Tz46Wx2N8qFPjz4x4eD13tnB9Ht0bU3h8fHJ4OfPuydhJPhzfBmPmrtH+8d9k+DcQ8NktPs58G749fhO1fuJp8n0/9MVlqDw8AcCzSn/z74+ftOsEpdjz62B59Gi9Xpr/+73v48+rreDn4a3t0sXh1NJ6O1l1HLxx/b4/l4Ml8MJ1ej+Nbr8dUiuRr+Nxst7maTdnBpI3xb8UVrrcINwcXhYjGrf24tFWa8GM0Gi/HNaL4aLuvJNz5OZ+3Z9D/t8aS9GskbXgt6Pbox6P6Ly7V2cEvwv1ym8JnwevRQ+HD1hnrTwtuTtoV3/zacR9LGP698Hk+uVyrPh/99HY9urqNPpY0IpRvMx58mw6DVo2JDZqN50NpwNVkJNo+zD9EcDpeo/eNoaoV/vj/cOzpL/t47Oni3F65Jm3+1qo0y9FX4Xyhq8JGoq4rSr7ejYc3uG07GX4Y3lTvjH2v3Bl0VXyn3QNyii1TSy/bfQm1Pr45ugsciaQKB41vJx8NbCo8m6hJfTHv1erQYXS0Gt7Ppx0DQQIfm6+35YnSb6kjQBfO0KdHFleinonpEN97eDL+OZsGd48litXB3/Htw+2Z09+Zaqvw3o0nUwfO19g/tbk3r8wkWXZne3gb9NgkHOXrootP+Pvlm/P3INAmuhq+NP588EcgQXSspdPTIv++G17PhZBE+tri7DVpfffBucjO9+jy6HmS35m9ZXYvfkmlk2EsVHU1fFd/5Jfj7a9iAm+lwUftWdLXYT2lHrWZdE45L++/hGper2nXa8lftXunnsCMysdfC653S9UzKi2zSXLJ3RZMpumXT+KJokvH3ZCpdu6m3vdn+4VXSTcEfu9vx9fqSuPJ5OF5MB1c3wa0rTEd16I7qWneUU/tebJbb19k2N3AxvB0NbsafC82LxQ9atUOJHapF/TXzq99G13c3wQRsFZeAlenit+i3eA1ItXAw+uN2Oo8U99d5swtA+qVw0Q5W3C/BWhwtw+Hf4aqWW/p/tYQrRvpKwYLBrvrZpH1ptaJfzaa3lTvDn7L76kt/qCnB7DpfKS/iaXsuYmsqWsh3yntA4fnTt/3+e+MbIrsMvOGn4+PTvvENockWvWCr/IK4ua8yd8DUhOgqJUH2gqKHYXhL4RZWltR1Mbwmucy+IvWFDK9ILrOvSDw/05BEV5MeaZF6HM/O+W/T28H8LtDK2aIwMRfB5liel+EvgZr9+Vc00f6MbZvw6Wz+hncUt7XoYm0mJ98ST83wYvim8GL0xrK5mD5Xcv0iiza8eT3cSMt9lHz/InwwWEbTf0ePhL9FS8nfglW62GvJTWmn3c7GV+FczrorxhLKHRb/Vuuy5I3RjfEt4TIWvbB0bzY+o+vCh4Ibfx8uRuUvJT+yn0ruCe4O30h9KbIuou2j2QU6XXCTJ34orLy1VbdgzMTrbLLGGi2a5LnNjc2sA5NNKtDHYbgAh7v3evu36Wz83+mkZuLHHTeKTPEv48lqKFuwLASe2t5+6NSdrsUGbKAhX4Z/rG7mbwp1Zi1X2Y93kdUWmtDDyafRavJQZz18eaGh1+PYkQl3kviR76MPtEpaPp1dB10VvKwky0X8xGWku7m+ZQNT85tWa45Q0UkMv7HevgkEWqvdl9oE0U2RPdAj74muX2xexmtu4NyuMLd1LsMmJehB6T7CZYsm+qvsydr1wFqZLMaL0ARORibUxPj27mWsI/V2Xd3NZvE+HutAtgwUttJCB6a3j+fRfluXsuAfpQvMRTrE65mMdelHgftPvy35ZNSnkUrmP6xn2rPGPtqJ9oH04y2zuzb6I3jfePJpMB/d3MzjzpsTc6RV08v41pcto7JJFA0pGVCwKk7AKExliCpjn2wBoRrV+pXUq7LuMh0c9mu4ewRrxuJr2hHRQpnuy/k6H2wI4cJb2GtapmbJdT96W7JKx28utDrfews/dtaKlvVt0J7Eso7X5lTucufVV+S8J2MRvsv1Mb3wt+zt2U/ftQuod1GojUKnfxc1OlhZozcXf9/c6Oy2kjGJB+BqGm4CqRJmu9x4Nq+q9nhyPfpjPdfw0eTuy2gWbKHlRxNln0wX967wV9OgPyd3o4ITYJgCMn0pti96U9jIoCei3ip/Ovopm05R17XMS1vcgReFhy7DzxbXKOL26K3Rghp3w3o7UbfLdHBno+jWeJbNb6aL+erwajGeTug5Fr83RHWCgUnuNG2ludlShm7zhtVUK34i+OI4fHH4nYu8R0s9JFK3e11a01vqatL+ewGHiL95WfKpsw6oogjRHhP2Snl8s84IeyF7+LL82MY8dFdK4n0efX11M/zy6/UwbshLwtr5XrIAr9ceS+fPemWdr6z4pqH87/g274X1uAFrL1kVj/4dvzXWy4tUJ7Or84uXnVz1h7e3gV05+DgL1oDB7G5SVvzY5vWi/pkSxePZ2aza7K3EMRxFaHbmvoBdLbU8wmu0FZLcFsgZ3hMdlISnBCWwbr0IbcVPJOZ58Ez5fCdoSvyycHlqV056clQsefqHV21DQwOHZDJJ2sp6GhXnt+54Rzdc/TZNuuiiACTFC13yqcH1cDGMJkf8743w6nyV8CnyRwoLfvEtOQT1+3B8M/z1ZpTsCK2yyVNebiahbx18t77jB75LOnilq+ut+qQp7kGBVZtJUBe6vBNlz5XHhN4AheZOvAmtVYy5X4fz8NnqbYWXK2yhAoD3n+tIX7LHg2lUPzcvGjub3333In/D/Go4ixoVN+7vcetm07vJ9Wok8neBedTpFpyd69GXYPEOtsebcbBW0JBHq2rNRLOj5kFGdk7ahhCZiYQh3LX6WKRufyT1D5HQtadKrVjd3NjpBlZh4MxvBf8staLiyokFqGid0YusbwI7mRkULTvZWXR6Bh2fqMfLSWdznXIJwv/uZp9Gk6uv2aE2tIoNlrHYOo5vjF5f+W1zo9utfKaMaYR4RTClMyczeCYcGmIyJ8vWRrwNra4mrUxNtGwm5/tIqEjJU/TCmr4y2ulno9+DPWD0KjxQj786qL476NHkkcCEKWxwqUwVqzFzxdd0u+zN10Ew2aezwae74ey6vs3m5+9hC4tn/+FsSU6Odp53DbuJcYOkN9TFdDG8GSTX5ndfcp0q2PW/D2/uRolRH20n8Q8Rnhos49G/gv2jVdDUQPzCqwOJd7vhC6oYYTCNO7ubhp0x6qBIsOJWlllFvMHx4JyrpDWpNsUvtXS/irtSfREQe+TEPKxtWaXHWowPYV7YKOJQrSOTVT22Hrqx3iXLeLDM7BaWa+i00j1tmqXJ3aVpuhjNgsU90N6b8b/vxtehEwXmKcnfSSfsD+2dFzu0jj9GG9psYkYAc5V+9dKEGjRoI96PvVfdxKpbcgIwmTY1ZsPKejnYteobV9DrtZ2Q15Joqs9Gw895y6WbnXy3GwZbeHKKE3T0x/GnIr/s08301+FN7IcFbcpYaonmhtOmYmFVzoXCWwqnN/no109cAo3prFd2p/gLgUbks/RVQYwCLFOj1MXmQ5FXl5+3pG8OXtYr+FMRiymejEZGU0lpsyeAi1oe0lTW9OnyyW9ytdAU8HRhqEPjaHr7deN6NLoN/yj38EXYgMsadcLA3Eg6UGILZXfKsYlsHMpEy8KAilG+RFDRblDCquPLrf8DW93CKf5AAgA=', 'Replay Shield V15': 'H4sIANyXb2oC/+1d224bV7J991cQfJF0Rg5ESpYvMAMoNicRRrYMWR5hjiAYjE07QmRKQ9HJMYL8+2leu5tdu9aqfWlSieZhQlPN7trX3rVq1aoHzWbzzdefry4/NIb9m6vet8aod9Nv/H45+qVxM7z+dHnVf9j7vTfsN64H/Yejr8NB47Z31W98Gl4PRg+HXweDy8Hn77KbPLj8cnM9HDU+XN98e/Dg9OTgRff9wYvTw+PXbxudxvkfG596wy/94cazxvnGm4O3bzcuthsbv/QGH2/HX43/8aU3/LU/Gv/rfOOnw5Pu+Iripx/e/ef9wevDVwdHG9nVL47Psv/sLv7yttt9Of7+Vffo+HX2Yf/i4s/tRumxP7w7PHr5Pnv46bvpPfPnn2+cdd+eTp/z+vjk9KeNi2WTxg95c3L88t2L0/Fzzn7qHow/tHZUM9/+1O2+GV9WNefN4Yt/vXuTt6W1ZFH+9/wmBfvGH6dGL1v6tnt0VDSxvbBLsL9dML9q4lE2jHMLnR1W6df4NiqdN79ob8k+3aq3x+/mH6vjENvYf47n5pJ5BVsES6W5cb7x4mAyStzElDqtsu5Kdsgfpz9iVwP30O7B0mLbFiab69HLoyE8c3HX0nqa33+6nObPfH2abxuFXvbYCbiBz29/dnDaPXHNzFitLkz0fMVWvvSfU2BuSx3sHHO87iqPX7TE8Xy5j00LaZd8qtRUscunsx81Vnjs/G6lbThvIOpry0LeNY90wZDi5kotYeFp8y3CMpdNLdwz749S/5qbSj42b19xgGcXpmmp2LvokZ7NK3ab0Kv1NZTv3PCWVuZKqgbmr1fDI5n2gUMX3Yryidlx+JT2G3hKh3edv35Lr+Tjo6Pui9P3/+yenB4eHf6v8Mbwdg80W8QXdv6kykHE62RtPpYQvQGN4I9IYh+4TChMZ2wv95ptaWd5su/y9fby5PhNxVT6HVFowdywUHPFk0puZb4pBNvbttvLnueWt67yp+XWBPRu29i70oFMN9a6C7ew7y6hHy37QT2CsdbuQw5L8g7cZV3HqGO6G6ObapxmuyIwtHRoW5/uWpzp1qe7PHa11L1Ug0l/gV6SvNAVrzzhvL/aPkpv0X0fJegjoy9n7IF0Hp6zidQj1979q6CU4uPju3+4D0R3NIn7Jz7UZeHq3T85rKSAMLGNknEEEY8VHdS0Xmk7zsF/lW6edKKM7DvFc/TEKIET0K3Dz0P9R3VlYAdS3cSBtfcddd9Rd7qj4u4GcSA+IgpUR3+JxilRopV1WGSYLEo/1Qndtc0h41V0k2RSfjBcRSf5+n0E1n83XELXSap97yz+fZ1Fx0Al9hYLH0VvkaaCxXjH5YSTrhrN1P3qBM4ieiVLp5k1iwlS1JpkvuLb05ODsx+6Jyf/4Wdpwcz5yaxwmzV4i1IEqXohZmkJrRhilk7VppN2Lf1WWMsxxjJGz8X222L0U2ybYvQTRTWrt5tyk5iTdr2dFMOiv2Yf5W/sEItCGUnrGExUX+ftu0EXuQ821uw/pvYkTU5TDU5llHSd5P5lQqcOZWd0qxu+D+UzVYiScjrXM1aJ/SOJb7t2QUw5U2ptoidKws86BJw8vOTVdWXccMZKYncr67y46H3CRbyGfbfiYIzYeenMix3mW8eIn81rW8eMklrXs9WVQ9nNK94qJVxWPHOuS+jVhtCuXxD2bkdaURzi3l2+d5ddqaZr4C5Lm8dqIrROZyN/HRMX65SAVQRypcOrGOeLoCnUinN6TTUp2iHU84J5gjrI8jzgiPIxXr/AjZL9mAg8fnt8TToKaABJzSFJybxU+QbmiBLoOw9xmJjRJcm6UKBxVYEmD7tjxJyi5LdJb9i1yQZkwbzVpwkGCNjUOj/Xr+fcG9bqO87ftr9Uv0XxkpUD7b0Dfe9Amx3oGvnMPx2c/DtXg3L50AfeoqsxnGaiG1HEOY2xUA0zf3Mi1abaPWVkOx8KkE4ytcRVRCiCOcpQ3bwbpZt1+SbW7ij+s3WrjxjN95kiqwrugzmy8rBqDLTS3n10GIhEptpxFZfYEFuNnUc5UOvQd/ROHwEmswdPI1sXJSR9V/putUSSWo2LEVdm2HQm246PjyZlImJw68JlwqInfy1XykgQhVYteZTQiTa6BivlansMX0SPWvNbDUa363EUpWDa8qfYew2twazXE9Fj3XUUFDmDSIF8oF0G4tNVk0HxXzRZ6TopEcLDu/a0aTGQLYY5RKMj4Lttc8kIsPjkOi3hB5Jd65QWexTwBdJ0qEgrA9bJ/plnuRu1Y/d8KiXJoBhyf/Mdj60nEGFjFtk4oKMjFgiRVlPBJDHIHWWWtq1C9DJYKHtE5EQM9Hy8bLNOshiUH4QGi4bWxRsgJwLgCAQ1IY2TJGUkS15oRGUPazwyTgNWG9GVnOdoCdetFfFg1q+bV+L8JwihP1qzODmJKuAzTzm06o5ARwjrRLDW5dGBgLlvL7fDrM0fW+plR5fzoXXClJa540RDfHnpUSo8cV+CB1sQGIwc4F+CqWbtGPkoLUYMKz+KW64TJM6Lp38dszLjIlJ5qfyTA9XRA6WvDo/+VUG/hbNp2EmaPQjJo+0/0RXYwsk4r3psPqNGyyHJTquMwMUvTqYTclFXVKbRI6tjRXNal/19//lMEp6PDl6/3LD6WHamsyOYLNcO39XZIftW2VmRmgW633aAeMyZJAC6+pah+K30UaFYTF0eTsEsCXpOYlXbaRVIrHVBO4jbY/RywYaqJSGhvqiU5/aK6kvThhoLvSgmaLf4fvPo/FB/7YndHTF6btrBndwdnljt8HeQ5AVGVL0n3b9HUfsbtxO6CRZH0dRyp0Me5CBybqtvMnPEieRkZbPk9XBTdwk30urASebjOSSeXqwunsxfVPw6ZwFgf0dBPmnmX/ouJ3BusnZV9dXoyFAXQ9C8XfF8ZWleib6RQ8xEBHaCPDMYnBO7SXHqY/hrkpck29FVsQfxbD8foYDzTzl9vDoHQe/JHV1wSA9/DOk9ID8jvjHE0yP6O0WRD1SmARQCPW4jnk65xUKXhQbnbi5AGrhyaWtqHlGpE8Xn+ufw8yNKl8FJMKJJOk0HDwGa4eOFgXQL1jIdywhGq3XElTVSrA1pXLasJbQKQ3ie0pTdvRc/IvoUOHUqVLdbm4/9tD4fG4Xy7pRX7XDu9CKzBg/bI86dOl4ss5gpdzum5+0ahCiCYqU70kzpiC66HYAhHOMAP1waacLxBWn7fvFjH8eEJSs7InxMUDdaBFv8EjBZFd8ulc8ufSqtG6AgRvrwlch1oE+AePeQ0szvaqGnM2nalXcmaQLIaEnu3UcJmYtPdg64Bik52+EEdQJVNAAMgOAxZyML/VoaJEnz2FwULxpaIG0UeE5JEcTKjApDDhCNWrwUZGODHGzMAogjxAkAiaJLNUa4yheWt1S+kZC6sJzx2XKWOiUZA65jmBAMl+phiPXlEP5mJSpyTA3BJHkmOvI2xIQthKlwmzFI0OKJ/BDyQjoL8ZEHEVsQX7eILaWTf2bTutXW6GycLoQwU2gssfAbOgVg6bc6IWg6+3YtJYz5/D+H2yFEABZ5QoBSZEO49s3QiS+Us+iuvdJ8mb09HNJCe4hatnpeRau9CmJFcUuMBQA9WQdahQX9icyqeJJ6+ECQEeBAZLs9hPpsQ+VonHA+wWOJAvnhWRVezUXgqgmo8eowK5chHTopTjvxWMTuWm2Du4OTTXAetY5VoRNeFOAP+evI8kIKwty3cbuVpENsDQ0il1aveOD0PQrhKWNuPmqPSegTYlsMLAMOssvLO1XKgZwpo1OX5IFWeC8e8XDEUTHiQy5oCwSNQ5Ev1tdF/iKCUHWXnmmPI62Cz3yCkyK3NsexRD8apbn7SXTu+2eTIYcY7cjVydcOaygpFucUya9uns4NWRfKYaqslve8pQeniZgUwmRS7Ay9qPQzZX5LBQdkPWRABZLnm7ioiJII4grsxpqD+mpipoecPaqrs8h9EXHfaF0kTI0FWZddVsg/WtMoKQB25ZeOa/gVLcQ5PEUurKUxifMoDFJaCXwGFW0oJclWqwzF42QtTg3V3Q0hY7X+4sBcWXjLk5+0lsgcVReB5C2tEpyjOVYm6p0XOhc2NPmWikU+KIoWxqwSpH4FD4c3RBeTeCK7A3DgHKuLJwcm4X6pMTo3DcmEMMoHYrC0rEF6q2chHaEBUYwYCw12pP2MllmrEWIsfNZfcBYWSRlYgJtgILq6qCFI0fJPimGOvAaSnAINxu1rEI/2SQ8t5oRimjYqIht6XNT4bnvL9i7spN1t5MQKROcgeJQgA8JpJo4f4+EZ8VHraR8w60QckV/6SKHJy09jXTGwIceGv6mcsWQJkwWHnjAWIslcFmHIvhHIWBXfL1JulWU0RBZMWuBHGkGkEkyXZjXVx+XeuTZgFE45a86lSbzRVEuMLc4F4GyKxMdGJkIqD6OtyfR605vCyhHLGBcCFr0RNYkIvciJdFrwWAPg7lE2AmX7mxHgyPIy602AQ43QtUzqyomUkBrPkjoAyKlRSkkGL6TUxAgStTUga3l7ROeBIeu5mkHXbPFPOwBgBchtFKWZxAB56qh83vco4c8EIhCjF4WX53DJurRTAOWVkiTZgOO8E+qwBfHV1N5oir2GnFIIU/rO+zjneJ5h44L9HdpSpdGsTj6P3EGVT7dn7WwiSqCuHg/UDBHX5AlGcD/VOjT1wmMCx0e2zpo5DmvthIgO8gPDoJzOayToHMgKGMZrkeZm5nmJY7WIcEDqGkbVK4uR9ujbURKejSCMTkWGpe5ikdZkgBngfASZyAoisRsCKoCoanKgmgliujhddTAarcms5GA6yLA1ohRNM+SWCXCYBA3ToB6Bi9Ewq194oxWfadZqM1SzpY3KCZvtxcj/dB2/1gADO6hWWy0fy3wSutYYAcsvFqYpSOcixdBWRzWjuHMWaelQDDBsrGDWuqWJ0jBaZaEsY1HA9Eydy5TVct0xvD0te0au79CYUdqY4F7Lv1K1gTVHQpwecJ4IHkOGhhlfJcYTQCAx6nBBRSo9pQNVmefnEXJARXBS9ruWQJalf8KdL0V2IXIy/FgMDnXG0Aag7ELAOuAR4zT9r/sdEC8hoFcyqTK94CBcCjrDDQ1cGJYHexq+pYCrlEIAzzBRDGXL4T6VuKe9CnJXV2cYOK3TgGSJMbr0dhQgWkL5RI093kIRntFVD1n32VppEsUOMGABUCmj3nx04jyki2EIUNKopBhyYbRAf3UIsfyglKEaNDjx1OCnXKp9SUDMkcbY8sKW9u9CQbdWa62YV+qr+e4lOJoUjvhq2ivKb8ROP0BCVwQ0IVKJJ20uZoG4SP0frpAGSrPVkM0oOp9EDT/R4wirEqC7cabYqrkoA5/pF4V4hVz+KGu/dIm/Uj+aNl0POpae/FwDEmPIv/JNIQ2eQyCKzA+GnjOqpTlGxmT4BeCY6GgspONAsrpbtHgbRBNC+FXz1YP9IiAG6IIZuXxd/16W149fn3uCISBb3QolwDxPJrEV5UrKYRzVw+JxhgBpQB72dVMjJdfVI9ajG60L+Cn8KHALmuzFpdcLcvAo9V9XUDOQ0djsK6mjRQZN11ymFfDOpt2TpJyIqBmpJ7YRk0fHm9x1/wC+0tpxr/Q900oXz4C6hXIRGxKI8pQWs41lVPRK6dyFCbvhUlx5NYNpIWGPooiCXLXxYPe4PrwKuWe6XheUUN9ZHVylqoZL8qTzAW9Tr1TjkO6sJmOw2rqWsyfEd4KJ97eSzE75bSwypGrMInT1G6K9+xY5TDEemC8jVV13uY4Q/jHhWMlAOhnO1pEtHgPzVhWDUXBEpzLJ9VnnVVANTIIKiNSWwFSNTZfqegigYHqekrWaDBIVTQXYIwJ3q7cPVDOBZBC4InRNa1P4iRQNQzuKhCjCyroKaBdpgRq0zmyRTgFaDACvBPQNiRWbNAcByIJmeEwWIBgbx0lNkgUMO8rpLC6U+SodFIBGT6IYEeB4ybPXETmR9BSQvH6UBGR+d0GVfNygaCjCjLhDfBty9CPvZlS+dTXcL6D2BJJzJD6RQcZOR+WEJkdrFw146vCU8pNEPKkZYP/UjDQp6N6T6Ol23qHavZTZdgE6JEB6dJ1RJUymdiAuaBtYGZ7kh7qY8EQvgfto44TceFq66MzG6/AHj6QxEbWb9M5mbx/cDpNbD2uJABKIZd+pR0xLl6RGR0O4o1CTDWBGtsEAsVZ5WNRJlbhknwx+8SgkzDoOlHWRJzpK63IZSHu1cRW0gKggqg4AZlq1C3xQIxfbDZcS43W0PEk47TjKtjqsinoZKFf5oEhemuoGWTNrgV8dHKKSDVl9GOmdHBytwMwsvuCABLswOa40cBGRWQnmuBjQc1HmlLdvGMmMznm2YNaaa5CMRwkjqGCWubp+ee8PghXhto0YrWQJychSkzRTTCzCwBffDuOPzR+OiG9AN52HzZfRO8D/2TPX29uriaBUTpvckbLsFCrSI7ldqu27cVPpIKK0bN3TmhGXYBnlCNwltck7SVEmrXrynruZAuNNhDyquzzqjLZ9/D16gBJ/kmaxQ3QB7Z8xNYejMLciiFjRkrlpqFuOA4KsNy4ltrF4pMe5guJqieF412EHl+mL4Z+ThztvoSuETpyBipmBeA/KfxMPlzocKvoSXGXPqCgiDO2IzDEJVEO+dXQeXdtOhmLpWFbZFyVsmowYZcjx0P1WB8btD3FBF00XTKWygmS/NpnrDwhnoNId68kGeaJExV/eiwbMtNT1D+BbC80WNO94xSUoDChQUpxgM2yXhq2mEkIDgSAHfwvbH0/IDevQESAXoIe6jqIiMJte0C1C5EXFrT1UsN0dLkpj4a71q1gQkRWEqEu8nxaQNEwzl0DmJGJ3u6dWWupS65FURE9TCl/CqxhK0+Oa0ac6+Uwpgaa7xWcioCaUVeUrjrpTHz3NxGECksm0YxJtjOjjsUmKS5ep5rAVA43MpW6kfsfnjMU9N9sguQB5alDc0UyXA2XcXBaBjB9EmeCTE+NRghBzCSwQQsc3uAEg6wBkA+GNSsO0OcQ/GjcIucEWyRY18BzZ40HTnagQ6PZcdYHodCiWThEy6O6LQCXn7YBy8r5KX1iKKkWun9Thbkkdi2QmdpG9EBZMJ+L1dPT6UAT3LgnWQqboMRMeV57knDpkpgGL4yvdx3TtAZ0YQq5oILyk/c34OLSNVUYSTxiOF1GAKqUIGrreIwQNzopFSIwgEFSTlJUsIBTkF3Lq1X6V0WKkwQEGKVL81zfc5d71z2Nre5CO2qkYU22zfNPj5DrjcTLudu9Yxh1J/1klQgUO2q6WsVXyvD2jtCl41lqGdhIYV7GqDkEha3ILV2stUntazsESp6PDATGFjwlBHP+QtiP4psGKRISJLzEVQMRwQ1alM5I423U2U8VlS0aSgX3GFcD01C9PXDALBOV917N4QqagB5qxBHVH3Odwd+A2LerAJacR/Bm+pJlXRl6XhWkhiTSyOEybBAYXNaoqJKnqpkluUQjT8aoLxutjW74FUE4NSUs6jRmQqKwFW6meBvAOZj7zjDYdFUwGpOGyP46v3ZPYgXTaIEF2zdqic04JaeIkhkJK1vRVPboCcFdUBtAxBlYbweRxlvY0JABzCf/JCPQY1cZQMapU4b2pA1ov0D1HVe2siJyC2tQlGD4tapntDnsK7UkshvdUYk+pmk+6hLgj4e+eIPVXJkhJ8goCCsOW5K0dfyrtWiSQxLmp4OBFk9+DQDUWCZT8D8K7JvDZpcyemJgai+GceQJslgQ2b6KYLALvD5QhxmJ61hg9QhBOKy00Pc/VBwxxYmfO872ntjjnGFvJPXasxHFEE3Puuj7i3Um4YogN6uQXOeEF97ySzrOpIEPvBFoQdtUyCurQarYgLE4JZz33KZkyDGAVGhqp+aix5KYB/GyQW0MKZInEd7qhnDc+pBAXLnJtNIZp4diMtXWcDPISaWIIEpCkfGyQnep7LkkRBwSb9IREW75bYhgSi3rROxRKyE4LogIisyj+RNtuUPsuz6k5WLJvlC0yZ77qxDZRQIpuPaUKTlfDdRyrgYlYdCapdnkZ3XoiAVlPpCHeN6f/PY2b/heSWVEnwCX5AVCXiXEtVw5vWRpUhItJIGW90C0S0GKhO30eYznoWPiWw96WlxcOoUkT0pIyFxJxzPMzjS+8ZDNe8lMcLwZR/4HoWApqTXbyB+gWzOUECwNCdSnEpfDYwyGMQ95KrzZlK5enp4uhJEMgMRecLwngOEQ/c42TmpZSR+U9CaU3ZXdEeeGGCDaLOmXBMr2pdZBMOfIAndDQVfmyaFmTvGYzTZ5KPflpwpZrR0YbgD+PRcNgkdQU6GnEjaoDiBBPRdIGBN4AcvigFj4axEblljvxdDXvMvLuifTWYCCGAlgXjNUo2apQGA68SaFKmHftCfF2OazmIu5CM51TxTnRKzCaVY5jYasMOvkl0erzoHal9PaOqFXVwsSsOWa5dG04NevxfTm+u5ccSAcCQnMDvUMFCcCrcgscE5KEs0SpYbsYVhoF8SiEQzHdK1ZaHU4SBFKf4tFLVNCSIBGAjgRJoRPFA/33GchWnr+ukjh+BskZ7+wHcxEzBydLO45L7D58JuYTqVKpdFEZdFY4FAuRhKaXSKsHkeBwbkDxvDqf9UlC1ICIQqbwMlmaiZg0Ujk3uQGc5+52wpdPy7EdwoJ9UDoNbUaGyoW+xa68StNJWxdNPARYkC745ZObapAxshyPkJZdaoao2AB3vTYKuKRFXu3+FIHLo2Q3b4iKwkIMykbiwY8CGVIz3bC8HMrOs9TpSE3WI2rzGIbCkE0eAvOJ6xKlbC4fisVfJ4vxQD1FwM+TO2LRpij8AA57knXSd3jsaT8y9PTkPiswrD7fWmhS6W3wU2Zat5RAL80pnz+2nCc+r34KwlNDCDiwNgcNlqQu0oeSiUC2nrF0mjFB0JF4FoAREJekqCsWgJmJA4TpiNLKSUKlEk+K3sCUjd2mg9dJNZ6BQr9HQD0xeUocJzIpEyu2Q18uGIYjxgTnjZBgERofnsQDhOYdVbJNPAWC4JYY2EW8OwRwgQCOP5VHrgIqvAUdUocRKEeWKQ5q4dmFEM+8ysmlzpelEo/R8sUJ3XgFBGDlemqTX0ZwHLjBk1Zq4PAkJ0Y56qI4prYfESlF5t8ZXb0BK72hpUAhr4snGqw0ZNPLH3P6FRD3KcEzjmTEAIlJRpaTTG8VKxjWAlq1ieJ+tYJWj+6T/e5asp88yYlifyqCQ7FuVqikLrrIpiw2Q9nDSFgNozapIzdRih5aG8ZnKohkJ9xocUx0PYl68BudFsrp28g7pXDnyMCNKTEstxJlVpmmV/r0N7dnb8ooYEkI9RQOdAoGSdwLnwpvmF8ReVzk6nIEXcFQqT7YPzdQSLBvhSEz5Nemr5IenCYqLSChWR7vl5w0h/gggN8gZyZ2a+tti7IIWVMSOJKxgDSYDMfLUkfkM9AFAmGYiXDNFYGfhOQdZpdAtM245buTiLLrb19I5IFlVv0xHTQDTPTfCnTigZcstMOiqC8yb1sgF1jRGA9V0ZAmgzwMBU3zLjmBJBNDywICsFjubtb2guB0bQpTu5H009MpTN0n6a17kp6JQcUm6fm+tFaVo+cjsC7ROQxJPDUQ/KLW6jvzrkyCU/NQUFSCOkxolZu+5vESFDk/ClJkrLDolC5SG1wPuMZ4+ihNxu7V1oGyifw6y2wjynPVAa05UqpA2WxiCSGcJAmqhh4q4tiGika+SX1t66leL+3EICFctcrI3qGvZlcecHaQ02ysKFEtNXCHkymCuDSWqSKcb/DtIihhHHpphuJ7GmesFil2Xa8ZqWfpU9UnERyxD+0VMtFbNMlLUFzZ8nYESqRQuK1PT/vRXkh8O0RbrR1HdZzImJYSzZR3AuhlR7IzX6DQkrQnyCgFrDpKLYoHdURpKsynY0sAzFoMhfbkNyawPb+ywn+ymv4kXG7eF8cS1dDL8ultBrLCRj+Jq5T+t8n4W1cQywRYiTk3GBbxFjxNle0HE70MwSUteyFisp4bQ6g61b4kpFjqUXx5L1+6GsqWSi2xG064o0AtQBoJKIoHlLpceYlSS85s2FwSb8cA7fmAiHx+TyjuAQbLVt3AmQrB0CqA07GakvdndPbbmVkXBWSyARtBHVJUQz0JG8Et0V11QYtvDyQNq8lmhK5uGrekogW6yIgqjpQMjsEV1C2sNK/6Zd5lrpDiDJOp5i6flSpZSd9JZAdS1aSJW38EZPwI6nmg4gqCSswKQNKMFj39M1ZJxxAKQRXUYVXBfc0Ft2HfEg9Gww1sPYESoUOTVX2sF2MPjvWMtc3kFqhDKUIS5RwyG8Ti0Qni3+kKD3rzIxvorqkb18goyX/7RPU+X8KVAzuwIlUGT2GF0BX2waSJcNewKqKAH+EgUZ43XdeuBuwKrFcU2DG56taDI7Zecoxg7TFqjHwIWRGFxK2V0AjFJussTcGyQLIl/vrpQH3YOlYgECtH/qAAsSVbJVHqDP3Ozhu7aB8RK4+YikIrh8v2mQIr7pilZw2YdizPHeBPIP8qtdIp0CfVc8I8ELlUa0H6BJNDoaqyDlwZizYKQXoi1kFy6WJrMekMO7J2GF2B3i0H7dHjyGVeEt81FhOMWS9CoC4UNm2BpCBBbe6dD0Uk0UE+DDqx0nak3B8/NowBW4hGOgq0XnxJ2FSiExgooIwaNOWBXEj6QoX6dQpdyIgd6qmFQVO9feHdVOmToj1JDxyXFRdgpKPc3mK8nFVvYNZhBAyv+lIAHGcH+2lXe+wj91xhxKUc7r6DNch5Xcut9LJA9JuwoPOZVcQQGWtPkbVVR7B85M4ieOzg6yrso08EG3qxRCTPS2zQGKQLC/KHdNt8U/PtNVhBwuAtcaEQaqfHYsFUKV4uz8HjTWTYWwh9c9e25f2itAjcWPI+YthGrFMCbUT5EzHMMyRmkN1Vl2U68hDXLGCAXPpZOIotj53xEPbowmit9DylZ/yMjWCT+1GuPvXqv92L5LYynzy6T5pM6UyyBPf3rBF8aQzd4xraAE/KQmvH2qw0n4KoCPcmrY9JFw8OXpwe/vvg9PD4daPTaI56N/33v7XaN71hfzDKPu01H7w56b46fPcq+/Nmczxvm9vN8Uwd/2espZz9N5+j2T+6P/6Y/f/p8auD0+Psw4uDk5Pj0+xD7iU3tx68f3X8spvd8fX1oP/gwcf+p8b7z/3R5sftX/vftrN/9r5ejTrjP249e9DI/nf5qXF5ezm4HfUGH/rZZR8vP4y2njWG/dHX4aDx8bvxjws/3Zr8aPbX7G+90WhYvvnW7Kk3w+tPl1f9zeufb2ePGo/QbWdiTvbldnPy7+b2+cVW43qY9dnkopvO5WC0mV90c9X71h82t3cmF+1szY2+6g82JzfYet5+NrOn+XXw6+D690FzctH1zU1ncsV56+HN9OaT+dAZ/3T6gJub7ebku4IV0yd8yXroW+fT1XVvVLh28m3Vlsktnnd2G9l/p7/8vvN0Z2dh1njsm6WLO539ycX7OzvPO5NfPO88LvxiMVHkn02fUf7BnnLp8/3CpV9a7WZxFJvXo1+yHp4NW+/D6PK3/uZs9PJJks/mTqfZu7pqzu93OvzaFy+atHp+1ex+86+dP8iXyPJPG5eDbJlMfr1d6J8t960yK29Hw+vBZ/JW25Ne3J500BZh4WQRW25dWjz/7F3dzlfop8zK0fvh18Fkzo+H4HqwfTvq38z6/2Z4+Vtv1C+sndk3ze0//pxMxj/+nFx4+0v/4/Sq2QXbzfFXy5ddDz/2h9lCuLwdbU6fNlnnzekGVlkMs+U2/dXW953WYjJNfzy5qHc17Pc+fuvMHvEp+/3kB+Memf5y2pbqpjP56/bYmK3JlF08K3vUdE1N/nW+c5ENwfjt3czvVHjy+fSq1sVFZ/bNpFHzb7NV+4/x1jL9d/uisIRvP2R99PUq67qZ8Zl9497/R+v52JZsD37RfT+eCMev327lzy41sXTR+fTXF45eLZsf2hmFL1sXY1Nmr5XyU0rtLHTV4ju2s7KJ9aE/28bz+Thv42KeTaZoduHy1Pvwy/Xk97PtftyHl6P+l+3/jr6Nbc/NGX97u1no7t5vvcur3s/ZJvKl93+bO9tj+8aTe2L45B7zfflhcfxnf9la3Cd7UufL5WBzcb/xs7eKczP79/c7z2aWfte7uekPPm5ujp83bVP1if+T/WR73o7Zs+a/v70ejjaH/d+yFdDvjHfLrUXL328X2z77QWmZLK+7n7OG/bq4YPqXuYXn0/mwuOds9U7X6Pl8iC4601+dP8vOvMUNabaWJztS73N/MB3ZD9eDT5efiyeGz1fXP/euGpODxnT5ZJN90qPlN/f46/xduV1dSQ9bW8XV1unsPJvcdPKw8l9ak0k+Pdtc3k4ON7NrS+eMQns7H65vvn33sd+/GX/YrC7PfG+bv/MmN9x6Nvu9sikLvfb/cweF0zUMAgA=', 'Scenario V14': 'H4sIANyXb2oC/9V9bVMbudLod/+KeXzqPtdOxsQ2kLAUpIqA94RagrmGbHaPyzU12EOYE+Pxztgh7Bb//apbb62XGduEnJetUyd4JLVarVaru9Vq/S04SfL0azIJbvLsLljcJsF8eT1Nx8Ev8efP0ySYZYvkOsu+BPXLcTKL8zRrHd3HeRL0xtksu2MVLzJW/aFe+1tw/RBcpNMv9/Hsc/BLercf3C4W82L/1av7+/utLwhva5zdvRpnk+TVXNR8xRpCWZ6Ol9PFMk9ahewoho5aieioNceOavV6/RdaPzhu7wbJtzkbx10yWwT36eI2uImLRZIHd+mkVSRxkc2C2ySfQLV4VqTZbKtWu7pNi2CeJ0WSf00KBmXn/xbBTXqzSJJZK56ld/GU/Zyx/1/E+eeEAb5NGUHyOC1SNkAgVTpjndwlkzReJDXsQFRFYibp59tFsMiCRTILOMAiuE4W96yDYBI/FEGRfMWiCVTZgpHVath0nE2nyXjBEC2C9G6e5YtgkvyxTGrix128uK3VGOVOsrs4nbH6s2IRzxZFrXY86F9cBofBX7WA/Vf/9L53dFXfFz/xU5EkE/al0w71t5s0LxbsY5d8u4u/RQxN9nXH+vqQJlMA8Zp8z2afM0YX9vVnNs6ElOTpPLGATNn0RPMpwxj6FCWP/J/68dFg0Pfj3F0f5+0SnHc2wXm7FOddE+er/oejq74X510fzntenPc2wvkqX3pQ3itDufOTifLl1eDo07veYPB7CXv48DaYRiPufH4K5gYQE/U9E/UPvbP+uRfrvU2Q7j4HW6+F9WOtdnR++uHojKzMv/f7lz1zEOMM8d02SF8s8uUYBB0rqR/3+xd1UjjPswkrhaLe3/9e9wyezgDKq6/xFJCzxn5rzJdaiv1PXgx3yjG8OLq8+jjolSD54fTsl/qK9UCw7PqxfG1x8vte78KL5+5T8fzU75/58Hztx3N7FZ6MAS4G/ZOPx1fAAQ0qm02pZ8oTd6kaK0D80DNP6UvHUP+5N7g6PTv9R2/AvjRrtQ9Hg196V+420WCCDWeXUeuPfFEPg/bWHvyaZp/xR7fdtMV0Yxva7Jq1KIA3qo0Sk43XrEYX+5mmsyTOseKO0ey1amaIqkYHoHfaVhcUUoe0lbKCjYzV2W570GR/b5MWQE5WH6tvdz0YurRAsgNqMKxOt2sOoww1nCDArI0D2i3BTHdDZhE6a5fS0PjdRPa7fG/oBe+OfulxcnL2kSwgu7o4/cc/jiJohHU4X2m+tKu/G3w8P34fXV5wjjBhhsYMyia/Hw3Oo8ur/qCHLTi3ysLT4150POgdfdAo0EUQKIQsrHtX0fHRzxyiXFGKiz70+1fvT3uVIBWpjwYfeoPLiC8UjqEcjARMyWEOkNF70LvsDX7tRT8Pjo6vTvvn7lpjrLFnr6X21u6uvVbYt7ZvIbC53bOYnFXdM7iY9bFr8ihr1TX5D3nGw2DtLdjB2FA+9H/tabHVqJ/3B1fvgbnCoNURBGPk6V0CTVodVqA+XvY/yqq6Zu8Ia4qKTBqdHZ2fRBcDNunYDWNsztltlEXtphBX0WmbFUNpu3bVvzo6i06OfocW221W4zcxV1F/cMJmDmvi5/cMOv7s1o4Zt0Xve4MT9nOn9uH0RP7odGpXR4O/s9byw24N/op6v10cnV+y+YO+2Pc3/PPPp+e8e96Wb+7RxcfB8fujy150xkYoS/dk6eVZH6X/X/XzT7jVBvVz4NQ3wD7yC6oE7UdU46MPp79ptsFGf6mp7rRDzUo7IeGhrtwXEfpf/kodtXl+WqeSCWnX7u6xhmhFjHl6bM2cArfvGd/eMbKIKaE1j37jEwPz1PuNcXcEvMC+sbk6unwfiVUEH3bbtbPT//fx9OQIllPENu9zoGa3y0Rb74Qt+4uj49Or3zmH1BioX3tn0XH/Ena5va127eceq8XkzfEvim044/UveufyU4MN7acm26tP+wMGLHrXP/+opWarA1pjO2IMuCWWTBs1ZfoF6kS7+neXzfCu+rUNtoP6tQOrTPy9u8/g82aPaNsdZ7Ob9PMyj8EODJmpmo6ZJhqivZhnywVYoUzRYIYvM/smyU0QjW8+N8bYKgy+JA8hMxpvYmYlN/exh/Qm4KUBs3zPs1myr7SWPGF60UzWl7XB0gW7cpwosJN0LMGRdrx0i5m+DaPfGqnDCuPFIvcjWBMjKG7jedKYxXdJGDDVapmIvvBvNg9MuWoAiYKbaRYvGrxKU+IL7YLDQ7X/OXhifac222f9NYMXpS3Y5u60Aat8C4oEXi5abHf3t2IlnXlFs067omG70dlqBy8D0twYr6DtXZx/SZh5EucMMFNdi0Z2XYRBukjuBJWvY2bhBItbxl2fb+fLRRhcJ9PsPrqZyb/usq+sRnzN/sGv/C/4yiaHy98hABwhvPGyWGR3sKwa0FeQ5cFfj01kkzrHhm0D7ItRgPgVboEaPsCHUvxgDFfX4ezB++dgYWwMKPzTbIYVFWFr0iSgdQvgXgqSE2c5G9c1paqB81rcS1SnRK3qR5Cb9yNpX90Pr6X60dNU3ey0zSqrzVbWVasTRFASsTZ8CtLZ12S2yHK2jtnsHoJEEYykJwJZSv0irKVrSBazviCrqW+K5awvZq3kj2U6Ta/zdHknUGfMV8n4StTJsQQHFIpedPHdfJoulhNgdI0gkxAwwuAVSqZO0voplEJMLx3KTQqelGjY/CUB/8IFQBAKWhpVDiyZFokfTbI4y9HUa3k1mi0fmhqApmGLomwsUewfqi4arLfZRIpwxWOL7H4WTZI7tstFbGsDX40jp9g3hhLAcMQKuHZAr4SPbd7zOAFDnTVo40QzMChVia6LNAy6bEyacXegMnT0lukYbVlDf+uIbx0ihorbbA794I8bhgGK8HTmk38wzgohV1/Optn4SzKJACiIw+FIcDSrPhyRPVi4LQrWMxp5CIhvoQ0yk3LsDB3ZQsNQ2L8EdQzqTpNZQ9ZrAsE6fLyv6WwK0r7EtnIGs/mcSYLZIvrK9IfraRIVy/l8SmYxDG6zPP0zmx12xDDm0/gBp8g7p7zUntbNmID73yOYFNDAuXUkfHChMo7Q4RUqy0h4lvhOs8gW8dSY3XQ2Sb4xVSTOkarJbHmXMG0tcRGCKnwO3elD4cMAAY35SM1pYfoSU/WEEiK7zrN76BLgCnZKp4nRgwkE2kAVaMTamoVS38gWVOOD6o6+V4kYgdXwtpD8h2cU3hp4FsI65oP6wghTR+arXzBV/aq+Rptxns15G+jNaVAylnw5BSmHiBEFxv4PXbvJhNXUHYpv6FRuVrWKOMfC/zMKSVBCFeeLC7hZFPhBxZ8TAaJFwXrrghNSrBCNLfq4o+UsXRT2EvFMI4J4ywQn0Bj6ZoIPSDUUDtGRn5p6uTBxAjC8tZIp6wGAvpTyQINH5/o60MV+cpfOGrylduSPQo6/At90x4k4SLYkMsLkKV7AuUpX4nzix1Hh913Ul1RhVuYalGAkgC1MtAoDceJg7iteEM5A2b5knAo8lrZvDnXFkbHTI2JyT5ikQqrETOkVa1DUi6+LBlApHraZ7Gohxa7h7yaoRaqwQwrZ38RiZEsApV/jOotztl+mf4IRA1+oSnobT29Aj1F1glevgi5XENhspxMmuAvl5EIlFpu0wE8l/yLKM5aWl+h2/jZKtRZzjfiW2uUaQ76ZjcdJgXsrSKLlfJporOdZkYLDwNgu5EfgcgsW7X8o6zEKj/SP9mgU/A+TwWf94196J3XX7iL4sM50B8P9zkjOExjkSbHA+WpI0GFQOWdSZ2ScrXCtnHBC6y/Jw+E0vruexOJQfj9oaDbUCPDCpvyDjVz9yXjQMoCub4oIJV9ULJI5Lmym0xTZMh8nChTHnX8EhgKe5b8QYkB+IyPjEuRxA6I26T8gv1VtNl8S/KEcnc0yQzjhuqzzfYxw/SFqdpxaWPbHMkENHwMMGkMOWCxlZjJBSAXTmPjnfWTPR1ImbX9hjvMYCQSpMRovcwEGC7bmGePYm0XD0E1VJXdEOII8ib8YPM013AlTwCYPwNfopTYbzb4hSQVoRk8mUaCB/NDBDw+mABaaUKMdHBwCBGh2QCkIe4Mq7JiFHvXCqyZBJww50MU5iYV+Vgw50NGQ9zxC5UcsvDVhc4gAgLWWY/XUwKmT1YCaRh0+VfF8njAjjVXSkoqzKmqLEv1q7tPzz9sSThG4ihooZgSnebjHquzjHkeAluEkv1FSSMBKZt0xC3wmvCig0zvmJ/tWanEIU/Eg2Haos+caPQqS39hRNhXfJzxWSLa4TXLTDFEwf5T5gX2uZX9Y+DN1pVjeuVqJ3/yofa+VUGYhiMMUp75lLRnlQjCCJwzVBU3lco8mVjYKNBAx/7wOry8O9+GgAOvDv8q/AJ0WrBbp2O9K8HkQ5PRzcHfp9Asa5xBSBqpRNm3wLv6X6X/kBDh0DmRD+zj1UTG9NdVv0aEAM2AcEgNpeV9QBMKEYuOsmE67BDwTw7sIghPzLR4Uuc275c33rObb7bLe1erV7tCvbJ1FYAdEqO42xCfDWyUdB+iqEBX4DIEq404cd5EYy5+vc+ldQw8OASNL0rIVaFgjDQ2oulOfHg+SJBIm0DhbzhZFAz6JkfIvoA3wKvtsn+TWFEZTMryFTfJYe4ojo9KJYR5XEelRZsth2CTg69tW4fsQGg1lA7YVg2/M8H5hNcftZZJHbxubyPsnULLUG/VjtgHh3AP/ehlTGBPHy0M+MkBQANgCzisaTbs7nAHeCCmPX8qpP02/MuuCybuoGGd5wp2Nsk9wKWP1UK9//C3tDO79ESSVvdaIczWQjgkZIzZ6/m2AH8eQjUD0JU9khuL3CA1lBsDzGQHOspzhz7TRSSRh839fiT7KGgq3u94S+Llie6vdAc9CuW9eQBK8y9bmBA6ndfvOHmtukp7Dxe9ydujsOmN4YWP3SnUkeeCPZTzJY9ZBduO1MMVss0UC+pthKFd4B+TZYv28DqsENDqsjQ67+mUd3BSN+ics/GYW9urKchRx5hO5ToppZspOqdhVSMKVppzc8YWKoEEpVUCSiMKFyJKRVDCSCSBRbWTzqjAAVlWi9aBga2sQ1r36miIN0b1+jj73S/z7khHJEEOqgbAwJO4r5NE4mU4JPrL7h1BuMVog8lG4DvJvodpedGXWuul1l2NVYp0BLQ0WbACjNQ0GBPtGVtnQqS5dFwH39GlXhuwGpqyJ44C/uH7FZtPVnpFS0rJrSDghkOobcZHyagVbasRQ50uXo0DDmoZyVKMQmRIbE2BFAvcZkDOHHGHENOK94l4MLYb72MFIzyHhK92HCsjRwdhiedX3VVehWQO0/oIVV3aPve+PRrqt9nwiw28l3xZANtmJoSlhjZCiLFc/3A0RdqQ4fIa1GQZKXRzH83icLh42sjDBicKqkVC4ViAPA9gePkZyV+/OTPZiLU1OqW9wAHoHrlZ3eTXfjs/XvThX4+dpvNqjaSW70Xb7JCAi/iyjGeCURkbx1ZQLn8BQoXml7WXcn+f43KpJYgKpiMLpOSztzjbVnBX4Xbabxxo1507u0F9mDLa06xwcELDz9X+thaWWD7UG/TVsC9Ffy7QaSyqR0GC3xmMFAUwVA3eyCrXcaHqX5nmWR0TMujSD0whcMUNk6BFbb1aPosAVui9pY74MvM1FUdUsC0zh6loB9998mIJ3xYK9hZEWTNNugpmrYl9bwqax/SU2PQ4OhT7kR0p4TX0cmOU2yu7WV7VCZcxFUBply3pAWXgQ7JUuZaQIkF/TgXjeI2mkw8YmhXGIp4uehqEBHM6iNKTCkKbiBg3A2dbOU9HbWyLKuMbYDgPKX+Q4Q11ywQPPjUBJlgqJ2JW8sdYS4a5aoIMYImGlAwMPTX0mlT8L+3Bo+uQhJMaIjfFtDlTP4n0OoT44239SFQwPMHCf7NTkL4qK2dfI2vVEgKm3tSmIyOkS4LXvWYKOSeqVdaAMeAvweMN/tkwJ4q+ipresUtP91FoBV4WjIglrJdD09ElwxIY3vCeiXBlI2TTBq3NaP5J+E65ksMnRThP+iW6cGxlR1AgLgz8ZrXAhVBhp1gpfrdGBFkzBNKWJXaS56qEQEsepHPqXm4hlGi8Ya0XSApsyEW20Hu5bvYy0F5GeBNM2esn4zGNJWxH4UgwfRsNvxvKr8L1JDxp0iUDQ3yrxEFYeHZO5fGmJtFpkayF546JIPwtFVkjcdqhFZpszB3BYoY1TWJNRAYeeYruXssH8h0ol4majdNTGIsW1+V0kZZCWKJB8nstScqPkhwMwOjUCFBBZ0GnfozVCvChWrJmxMegUPOBGHaUYn1sJcWiUDbHVaKR2h2JoGe9uZS3VS7CrbGRq8dUSnV9CMqfXFXdrSHhHGEOArCRIhXSutKSgXbN8SyiRwB6ZriUxMr4K5xghr3Ndqu5abmoAhtzm50XkOBTjqu3j0Sc6fUD+MngohpHVZfOmvcH/6ZztlvqBoOpQWP3EJZR+QzEBy6Uh71hpx0LT64gCksPtK9NhBJCG4qxuJOQ4pUiJI2SRx/fXSZ4/RMqXAndaaG3GRiD9sQdb9nNnihJaw9H6GONqLvUomAfVdlfS76EH/IJTwDyxbHtC7FwQ4uoYwOiuU5/ceoQ2gl4OJVsM2jrg+C02awTycufGIyAw1L1QG4grnX4YbS1a2UT6z6OPS5vvHNR3Dui7J/s7ppUqaCEcb8xBgP6ZzrlgCBVsyyPtE/Ag3OociOGkxLp4v7B/Ddou3i4EeXMXwwEy7ur/zK6Dz8ks4VcPhYKujo3FjkVu/yTfxtPlJJkcMjFlRO4ZQR4gfMgVNaVQ4ckzv2cC24VzPC0OCJ2LDEJzlJ0j5kQvQoghBtSJ64JNrrPwrt6KEBodSLpkqv9Dw1LjkVwYkxiCF+KwLcP7kGAPprEv1G1UP+lNjht0TNNv8SLK0+JL5G8hS6XPmpZlTAGOVCYLu1QXsFmaZFYpRowXvo8xs3Ct7/dJMiF1f8yR/SZ3DsoC6Wz7Yt/nDYPJGqoJItaoByGtct/AKVjGN0rM/1JyfUB1wOe6HLwZDg5pm5LxEm0G3pLsxxA4s6I3m4s8/aIS78RKMUr9JXLJ6EwojxWEs7lu7Z50VFYFdMGXTyPbPVun+ZNJJxTETYbzqeeNxVSg+cKhijODhHLEXBfy4K4BsEXEEF8nrKo/LEIYjLD0hmLZjSzDUfQHMBFboeirKBh/iKQxAFuIlFHHtYhUr7j7lMbhSRy8FfAox7iKM6R5lcyRNldxFQo2Ohd6mm6TGOK4s/EX1N/XDc9yNAdLuhCQ7qnaup3IyBXniog3CIgIaD2JcILPdr0JYmJsxEKBVvsx78LsQ+/HK7CV9gm9BilxGMc50ykIGu3niGYrmcHK8DavuuelpaWL+ueWGsfQqetf8GJhzWqJuV5GPtZXuRLWtNQugKDiFDFxYjS/jYvE66sU9Tc6fIYLEHY1+FZHPQlMu53gpVl8my1poJtxPZMnO4zEkbbg0Dedn8AWZlBVdKlRkYkJJyWIE7xZl1V6XCDdZ/mXJAcXQYchCFa31mRumQDynLYasq5q9y3fX97KfhUcCNTwLdaXpQzAtrWfdmuW2JOjPB6cXp5e1o1D9cNgx634rt+/AoPqwq7b7Xig9j9c9D+ei0s/8uugdwQXp+uSweLJJGI2A6fRP9WRBuMvtowXD6FO4RESP3pIZp7/PWM7J9e1RXeQufOwLnxdU7hStIiAiw6722FNEAH6k35hhb8VHiIxqe/LYGv8aXnM6ogiq0OTi1hVREqFfXHtSt4eMivxMbFK3DWPv+w6M56wEP4J7VgWGDZEsizyBv9hNyakEEMiX5p29IoycYQQ49F6uLvPIEepULshwBEXryDsjBmwkRYGUMCYs0PCS0ipcR9WFHFRYN3ZLiKm7o8Tfo0KLp7IXloUKL/pZ1w1Lcn04YB8e0gkq1AozBr/hwBXSfxGqNxZNuFNkrOBpX8meZRnKZedwkZRZEIz/dC5X6Ay7kCxMFX/Ula/lXpM5Pp6dBZge6v9fIGmgEpUGm0KpSrUFH7YcabqG9/ENWlMkCQvgb7SQPMPSnj0o4YrRGNJHaJl0KvQDgeqG9maBeFf3vBOOF7Xvptr3A5HMtSM6dVXSbwH4LsgXQHdA0wbgecmHIeDwLm0rCPfstmy4PgI73I3lBuj0wpTc6AvrKmDohRuNG2cD8E9iuBrqoV3X5fg0/UETxGIrynErgFxtxRirfIo31oR4ieF8YJyeMvhUJkkC+rApsGX8z/JopaOe4ZqyGT3H0u4xgqye/VKpyyilpMSAlhacrn3XxBIvuHS3mRlPXVJCatdhqfpNqLAcruI7QMT1Lj9rHIIiNlLF+kYjwQlnLfUQteCC/iPzL6xmUCZhOQUiJ6N7yamiieBLOwvQclWBzGFfw6Q8HKfpYpelmI4Z+WWRJ2k0OAt16VfY8609tbOLlskyBiW4WyqcPS/f3ojVLruJ9ad+5Fqeub5qxLsKkGncdQJmqCZtdbpT2iHiiBRnZENOHlrmt0necM5D5UzTebWiDwXXLx+votNqdZ2P4k+xaxsQsD3R4NfMeVms5w2kJ4eUthHtzEkuV/US4+IuSjSu5pYKxZnG0mHFjKiGziMj8O3M72A9LxNOcJaNe1culk0U/2an31kYkY/M/oGDoEkceQgIxxc3RfDZJGFD06mBB95mefQuzl3YHk3LAaT2aXePP9i/LFsJSiwkqtQCyHsU7qsgELfR4Nt99PGQ/ezi2/gNseUcw0ksgkOzUHz5DZUD5SHUeks+pxn94vb6D6dTbJ7UAg0O8FeAf4BtkvA7SSiYll1FL+yjqwrfhZ8a32v0E2/b4o6/5Ipwic71Cgr58k9ev6PX3kwo5XLTgRtVU/z/sZSeNuWws7YniKBuX7oil/X+oazv+TbQt42RL+lDg99btX5X2XC4kWtw6DSwy6crdn4i1B7oYIHo+e8L867q3aol3eunJnfde6AwzQPF9T9VZH0y8j73DT8Q5xB+UBCjZFkLOQ7bQP6zvx9BuB6nnHwwdnVLNf3yutbNJBZnfRx6ib8iiZGsJmcg+eOftb+3vUBtNLhZrZlHdIvgubki+AF33qu0Th4OQNkhZAmgEJUjNlUsm1pog6TwCSdsemV0b1IF/yhl8I6l0A3uABaGbcQqJNdXHcUu/VDGiqO0f3JTGRSRbcLAwNx+dGIEOJZDhDOqFl1BE4zJ3gPy6q7UxGnZloFYZZhoIhwv0Bbh3jugbiIqiY5hVbEVOMVA37Yjn/yVLD4d83OheY/Ji8zHlfNp+qZHsnve46/0T9vpSNQuI58oREoRd4e0rZarX4ZdP1xEOWKTrXCU6Fuy/92uu3ywjLtR2sM7z6enp2gKW+MSL/dMwqb5e2lZnG9TJmEr4LidxLQ/4zzpq6/XnOdqAjcEA7wqYKVAQ96siuiHmRLlPi4OFUz3GPe2mfLLhMYLu0KDoOH9YSZv6M4WeVQ5qfq/NpBuSfb0KZcJcup4gxJaFKENu2RP2umVruMyrXqeVPHk8KZsYPOstfgLFPjFwo926MFK8H5nQeL6pVVvqp2/J/Jqekma6khdgsi6Uo4Xa4Y5DNcMZrwtacsi6Y/Pr/2hKg/R5xyEepsj7VVohZXC37EUziM4Arljvro3d0r5fxTJ7lEihjPlK0/xSenf6+HK6Z1PE3inN5nqz/rrPoizjzxcT+SpiULp9Nt6+0x2BXJ2n8YnSfp5wgC/348eX3RlE861FrZZcMJMncWVs3ekmQFV3kxdl+nnQriFUflgHutZP/6kdzU8X/+6WkrlM3VcW8NIYzHHSvkLhHU48QU1OW6zA+cZH8Msb/u//z3MkSJptvd/WGiJE/4FKezcXY3Z8PBNxQUqZ4yyT5i7q9sVUpPJyLbf9DrT1yuM6qtTFDuS0wu+Ui83eBR7ewkbT4tby30X6ypqVblaVuhpZKqa+ioT+Xh9jPrllWeau/xo5i+VazrtZ48h+dMofkap1OM7ZYRAl4DyQlSegvv/q0wjZ5ZUjjOsicQ/Lh/dtY7voqo03Vd2uv+nyQ6NrsQAzGvK8IzPNdcvu9QnZ90sX65iufWYJu3Waf7ur1ZqAKYC6VRCvQZ38pz5hvQD01M6jeuzqjn5CkveHh9SNhyHWnarLli2vCguKIVAm/hTHQc565kpYURxmr5cLYDUsFZXBar6tyHCSmCYUBfQ3FGw3ay6HOczkTwt4E6vMli48Ez6dDlYLyIYiLDi+BZSUvWi1dIbBxsp51+R8MGoAIFyhSgZzi39D7T8u/eAZ/7+FTQe1XcguH2rtx17ETzmD8G4x9JXP2OkDq+upDgLV/cxNPSVBVt33B4Py3n3MX8ZB6ZhLUqyT+FhIbwrE1UxFN/UjHfbtrdWbXx4unPAbwj/ro6f5hLEVhn8KCIhZxPp+QB/RBGWg4KZ2GnVCd0hveyJOdHZ6+9opP2D1TmzFsVz6bTba5ilJ8d0DUjqFQvrVxGydIGcuPUnM94wg9/pUVcfcZoXHt0lFNTM4I9xNaNaiWufiLIZYkjfl/CCySlexLduZpWO1OdOlhrk8E16xf1TxXzq0V8k7+t+Fybmkf//u/ZvY6PBr3KrQtm2rthiYgHoIl6Q1Pr/jEmWXfeOxKpnEPjPp18Bbgiw7Rz/abiSaWajyVfmq85mSAkbpjK2EnPLNMar5Gc2lzotP+DQ2PI6rrP34KT5XzK1PRFIhLhteJ7UAsxVkRkRbqDh3Q4iUH/jgQk7nOV71r7gkk8Ob7NqI4qmj9v1Ml69zbxHhkNKEJ2Im2Nt8bryTwtsklyycpBwX/ThdudLaHx+O92Eoj8iicGNvLAGO13Cg5X3/jkR0uS/TBMhRMY/p9ZvaMweFF5x1PES+qoKACBcSTsm4iCkSFV6wZTjUjmTJhPUq0p0qkpjGn+c11N3mmEEBwr5UhVrhJyD0AE7Twh3ElFGkXXywnP9rV2vJGYL6ChfHGJJ9tG0skxixtUYP5GMrgI0s5ATfg35RINdkD275BflxyRQ+eRjjXDe+BPu8FP74eIy7YSGoSuAUQ2Jf5WlCXo9IpkaSkTGYIOIxXiQoamJ11UlURmhU27bGs5hxf1Gn/xE8/9oP7zae/sRD67QzuUfCN+C3T4XXn8hi8zkKRjag5gyRpUaJKm83T8ZTmXV9FUryYjh0ayH3ygfpyk0wY0MTAIycxBXN/rLfLauO40T+6YpSwwroRhXPFBTwt6Lh4ayD3q/i9yUJtOhR69miTk1DyefU4axtib9Jltkgj/dWijq2fFHkfrULQtnTdj33OzcSsGuDg9/uWjL4V2HU5XoEaZY6rOMWBVxK0Ntwa5MN2mF0I4ZcscbeoCtfC4eVp5j7r1rWp9+Vv+90jjgj3qupFguFyC8CWMUqRpP/+NzYk80T0EVnqHtaSLLzKWcLZcYUYvTqIsJXXK42xJoK8lgvwBlObir3n8C1waGOSlMkEXhf4e7OVa85hYcvHqYOPbdUB/l/jxovgq2LGFjrX43e7LJMBO6CVD00eAf48sqLxKt5lA6FYs/Nfd717e1UlZ+NHoZhoDfc9JcZxOwOwkcHlCgh6P1SFXje7IefKQLnQ740/V0rZ828ibEX2axXwvQG/1mnp0XQsMLeOTUskssRigbA2aUoqsK4qwZfNXLgD/IlhvIdDFIEjtr6OWQkkkCF0JZVXINlhSo2JFmKuCrAwetB89030X9XjEZkloxBZW3EawvlDLMnJxMNuGhLlqt5K2v+7YiK3Es0wpsKSwyLJt3GHRIflU2yYPxQJWMpuAs9bI5RMF1fPyYVPlpIQuLwb9k4/HV5c17xkB6ZDZpe1ViYmhtvQ1Pxm9mu94xuAM0ko4qVTILHFc+cNo1xz7bVygShdpy8Ur2MgwXOtJHqLh/X9LCze7K26zJTNgJzz9gtmT756+cNEpHn97GOy17VIyeaYXkpaLVKH4qKhTo0GXAYzEbLLTbjedJq5HENMIKfEp8po3N/Bz0bUGLml7crwVCeolhzIGp5MZeF55fTIofwVIpgDBDEcgDlbK41bHClsSB28rxDReJ+uKFAldeOlQz2TzGQX4PE5zyx/Ak4JFQtzl8X3kf1jA5ytSNcWr8bS1fjve+KpekKe+Ju5dVBJ1SHEyH9MTsy3RlavfwFRyhLWxi8htUTrkDFCetRL5QrxDJFoodsBXwimS69/DUfC592TfG9Gh0aRGozdZLNROC1x2wAcq4Ycp+zDHF9nvDn1hcaVo63MDSj/Bf6OqdwLX8c3T0AbwIL/UAN6aZKA5x8Kgu93ccAzWypTiynNY4T+vWecEwz2D2QBJf5Z2/dzBLIlzoACIZe/7oqWi+elzoqO+uWKrOMw7LJPtFK+g2jvyp50U53ArObJWctyOhrbsSLvZrJTo8HaHN6TgYnDaH5xe/R69659/vBSnceJ8m4nzdptJZX71hifs87yrJlQgiQMX6p6Du1ZwNTj6tXcWHfcvr+A5WTERFUEJKLBLtzb/HLRwsP5dQ7216S019gJvDVP+VhzyM0FfWdwebRYi0PRtZ8sCvK8qmya8dNjUBcT/bZXoV5R0Aao68t017xET1DgWFer+u9+gnkR5lt1p/7YJt+U1dojvJ9RTZG3N1tYX4f+EbIDb6lm+AKkADGM+Z0vB4HO2lGr2nqoqSOqtsCm0/i0bDA14I+s1ruhL4rkyy7C09uXSPTKbi0cB6F4k0l06h+Aeia/b4/UBFbMV+OJPHsuksRiGOi9WUEsej8P6krSC/SqUBvoIEx+qkaya35QTyeDsx5bWJaS42+wjoz0GUZVRtzKbuoGWbNMZlaVTJ6d7XOyq5k9XVRS9XNGImVAFVt6dTu+Rh54tCvX46Pqm4Ck88dy4IRQAbTwZmqt/X7WSE/mmS2y1++sObai8ToGz5Rrbo3AtyQPnfz8BbH1nM4uAxNXq91Zdew9Isf80/8ZKl4TF01WQnZCyx1rJBHju5cvJtyeg9NbK6llZscf6lVHHxWXS33ra1AxWm0HjoDKHOPqEZQ8+IlseL4xNVpvuE+wavF/AEUN4yjWh+lDg/ZDVzAy5yTjy610Kx9Zh1Qj9RJdMXBag618L5XkF1nxZbo0YNtOBaPsA8QVR6e/D5+hKA9r8wW2yTWWTigQKyHelz5hWNPYj+ceS7VBcPYQzDpPc3KEZ6qn2w+CPpHEw5fxaybMW28krnlzaSOBr8KGs6tyMF8AxYaDoBzvioTQYRqpN9HZz9T0+ohkQZ4X96JZSB4a6/ggQ7TxFv6Gq1lY8mTS0Jta09CtLQKqQr5oBS6jMCIvWb5rVtENyokJgZEUawUhebRJhYvsKHaZo6ZcG+M4ryU/eR+JxZLpVZ5+WEsUX88CrX/I1JQg5/IBrVki9eQoPOcfTaTamz27dpNcNjj8ncBwG13B9JRTzYp2t0aqk+nUYxMxWvqYkiNX7ECKYGNhTP0SJUWNhII2nQ9PiErnXeEv9qKoZRMeVvjWi5ZzIMezciOATXzjVRXgdsfV84QygEkj0NZu0Kg9fvInJJJ2f9igKEXTOAZdY4M4jslZUoHyH9e2hG0wIT2kbUqNkM3YlQ4m2J5ahzj4nUOA9SDMEt9t97+kRxMIu+eOfWiPzvmfEZ32czEUiWXhCU88Fh8LnIgz8Ql0+Bke0Q57EUkItdW0J3uUbB5hM/LepOr5UgDz5jrkcL62AyUgq9aLrPIm/mAaJSWS+ueCA8Ilo/pIEHkhZmjvXT0Rry+rL4dXAgpMY6CpqdUdNniHDALwtortqFXNUc5UPnpZeduTZ19c1AEIvocPqq0NPYIHvmX7v1MvMgxyOfnuH2Qo8ALdgQ44h/GETCStSROrtyjGn6CqpVSzFVTKLPrPoE+bDksdV1kOoDKmVrz0S844ANqy5RzscTscMbLD7jLx7kLgDyPehUOkBTbkjOTHlo9rqyG+PhK+I+37KdqF3hO/aBGo+v58SUbRDUmHtzQHVA2eH8CUrM3y6xrYutnMjW6jH/P3uTWb1fgIoKGmikrSus5NMkqmeh6GZhw03BZO4z7MpWK+oSsm55lnqf/mWsj4nrdh4qtnCmtRV2oJd/bBUcajimKfzo4/XyLHjv1UDgZjB52IWlxg+PnHmWqDgnWlCbDbJvOZKdjgMStj2pQ3Bk1jSnCdMUeHc6FwzDKs0rZLCVz2h3bKzdRhsq6r5OdfkWnWPx3i58BZCJWi8q3ydOZlOI+kyEZS6jb8mJEZQpDhTJ3shJBXG5+TEuKoSLstUcQfG07ECK+iIX9OLCzzEl862OI/vkkWSi4tVmOFLbow6GQC2ehEMepe9wa+96OfB0THcXiPrlvT+hvbOIbwQWx8EKDWwXgvfAMBbMzUVoiWzbsMDaFtvdr1w2lu7u5zg2XzOKDNbRMVyPp/irTL16Wta8KRTWKTHxkjOJuXPbHYobuotsvtZNEnuYgzuicjPiLEnPEZACFNTqoLZ81sKpmL0b7ph0Nlqs8EzSnQgOWfDBtWioJoy1DX7J6bwjHy6Djy1UwmFq2BLSL2lHjrjf0SxXLKeLpD5BJFu2VBuM0zlIgblTtoBzpmQs5iykDAChQC0UL8hYG1vD+JACYJCbSTu0jZRSbUU0Y5QzeBSctgjpOSS7fgQgd0UPvabRaoLdZNeqh3WFRxn2tUpx2s3i2PXDE00+H7vtSl3qNOYUU7jLvYDGDnkdKW9NptVgtdeaDuO2C0lHmEKch8ZMNmDmdwGngZh0SyTyNZoamUu8tLtzzdcuKjr3fuoTOwamdLHaifXl2wQtJWhXwkrqpOXTwnb2rlM512YKf21v1zuCGweILSu4lbrZi/aisz61Zdc1UOlhfZ6PuE1b4etKp7OdlMNYafuI+Cl72p7Q48RDZ2F1wNPJOX1H4FAaL/bxP9a+sqAIhvy2o+AWwsEJ0apVzOVH5uoVh3/MQAcdyg3BjcKWypQgj6j5x5Vw/TzMmXa8p5Nl4pMzsAPQtBl8Fycu+nN8MlS3aEC5XvzG+KH9Fk+0N2ZMJEvMHieSsZZK8i0vbRf1i4pJG+4+4qdl+qpl32aZXC1stOmuZjeCNcgwdhKDYTPNosnKyAW/DtffiYYCc1IRZSa8rOhJuWlxgG0vDdbbVMSYopyLi4/HP0WvT86P7nkGxqOOTQ7AtErWE+orWunrRAuyn/56yd4RUglii+/OvR8oYPc7iDOWtP36XPj0keCDH+ugRCZfkF9qoXifX99u3iNC12qtXunS/byLcryCY/JdGnBivmpYx/rXCT5FWMpfNHot4if+0f9wUlvcNl8omzBZ+Y5zdZ8dp7bIxLnoX5cV+a9+4mnSELOMixLFTvuSXVhWsvrSbWrnPi5mjVP+P2s0RF3KY2+yW5wm+aJpD3JOpIKIeWuGK2b86YHsj/pdOGkAeexnlzbrYtvaeNhMQJxnBd8SR3wmi+Drsc8N0+j9JxIh/Sw/v500LNDqzlgiKthkM18iDgaO4eWD30h13jRcF+PUrBCNh4v50xJeRBRTyXntRUPSLkvU2mYls5qLF+5w4lklOmcGQQz49aktQf4dguRfB/y89VH1ptWOguK3K/svl4E4NyJLq/6x7+goBSW9wJeriSZT/S7YI6P56U3v0CVW6j8RMUvI2skLhxcNdY1J/t8bp9cJfya2AMw3XI122NNHh4h5rLljhRwtUkheIbSrUUI3zT6wdZrHKHD5Soqz20Rr4bjv3Ul5O0pTydowBKaPFdM5D0uHannM6YsH5njnV3PYea/AFgRNOUQZZ5nY2EfOZEWVX6F7OamwGhu20YGDuKFOs5FIuS9LwoMKKVWQ6Jjh2hJmYCVIX6/kSdfmcxJcBfQkRpl7VHGQGPDqKPymj++7BXYpqy15exl7+zMDSkjqX5Q5r5Efwd/0ZmjqAESgSkYX39qBS7xjMOC1a5yL4x//2Kwe3T97MR9pbQMEVlEI7Yw+sXjD/brJGadjfSSp+omrtW8kY7yZD3lGXSVNfSV9XUWv96yru5Srb+U6DCrlBXMscHfO2RUlXmseZQqElrYcvAOhr4K9Revty+SUzkZULSjSfuXtHtJ6hT73hdj6CNItJ3/KgwtEQtIjT+7n1nhKBJtPuqhSLXCZisSvIxvfYhIc8XevJqpZThDJpEokog8SDLK2JyCow4tDbxI9K7fv7q8Ghxd8KtEHy76H89P6o+m40tYJuhjO708vawbLu8K6964kWHUXStnBmPhn3b97WEk+m0P6Out1ZWlDupElqUqaAnkmuHYbdgJ4yltax5XHJ+U6OLj4Pj90WUvOju6vAJ11KgrHyPcsw8C5st8DNTnSwU0A35hznou+Lj/CSaQWei9i7rlkyZh85LpXFkaTVPI7M007QjvYfq9nqW5lasywyDvSwb3V9EnaOaqB+Fb3tIb8kbWoFxTrVUoiBWEDA6ErFX0QnWdspQ9ekWa0+e+ibme0uMX1k5OwnUG79q3Aoo/2NIbwyJO0GefI31UyzN7g9qK3Wn7ka234/6gF73vDU7EKZSVM8S6nlDzXt6lic/kf93S0xr79qrS4UriGzZ5/KX0EjmuRLErujQqvz/y6pV6TUJMEpODWbEouZPTXLkQnDlWFPaGYNgb/ruPv0fqzVmZXcujTDuKgOrmxdoD0sr1S3Kxo1KCGDX/g016clk1uk1y+ai3tTyEf7tIc5JB1WBJA4bj09Azv0euvPLKOqt+WLaHmT2/paS09ie/hmpUsju397K17BxB/h9m6iR3CROQjOOI8JKyq0yNQZgkE5ITIUBUc/1hmt6l3rxyJtFblOhh7YnSzORS2+hHVKwTPzjCNkIyFONrorYkpBbcm3H8C8rUaPHRtzjQA5fI6+xn3Fo5RBj+DcJ2iZbLNZ9MEz400JPkSEukWrlp45dXWCX5tsjjaMojemiWYGUiLmfTDKLTwNU0yfFgDwKjzz8x86qJp/3uAl2tqhtLkCBxgCcTvd9YC6Zznp84uilTOOB71L/onaMsGerWI6922umuLxRofjm+MrCzi8Hpcc/fk16P2zwLLqXooZRlNJ+Y4r+3h9LW9fKbjxsAm2qXvBGwsF7UhE7/bWtGwhgQL9p0dtu6qtTsyXlW/UPvrH9eD5u2A8UwqBCUqqu5ug5c8uldbzD4HVnlaDDo4/er/oejq369aV2p4hccZwSb9TxzeIPKPeRfJ6Uf5wgepYD3JeW70LV1tEIerIDtrASbu2HZ/TSlmdHpaYLyBcg0vaJ0YxFzyd9H5qRYJVyIuqT5TaaTOjDihGnoA8YLrhcK8fxHawKFH3HABnPVZZO1LejR/JcetNVWeah4IMAkGaeThAYA0DhZIC2QWZ1G4wd6Y0ZqhQ+Jc9bPv9oTIa6kNNrAEaLhgdpXCnrpQ0bm6Hu56vKtvnQ7HOFZNh6Sw69Hhbh4XqMY8l4Eqpyh6JjEJye0S0axYHAIhhtpzpRvYfBn+TZ9A2TV9WP7zqt71di8A0sqKEo8OcQjNIKUYqb1LMrYY5E/OLPl4aimUCfgkkHQw3+M60iVXIZncEW8WOQcZB2S3TXFXewRNSqqWbAaUpscQ84mKoNyw0lvbHBU2TMitkQo43Unz4c3fbTDKCazuBfSDU7xPL3BVWg90KbdliynkNyt+/8GWR+OI/IAAA==', 'Frontier V12': 'H4sIANyXb2oC/9V9bVMbubLwd/+KOT5169phTGwHEpaCVBHwbqgQzGPIZve4XFODPYTZGI93xg5hU/z3R916a73M2BCye8/WqRM8klqtVqvV3Wq1/h0cJXn6JZkEV3l2Eyyuk2C+vJym4+Bd/OnTNAlm2SK5zLLPQf18nMziPM1aB7dxngS9cTbLbljFs4xVv6vX/h1c3gVn6fTzbTz7FLxLb3aD68ViXuw+f357e7v5GeFtjrOb5+Nskjyfi5rPWUMoy9PxcrpY5kmrkB3F0FErER215thRrV6vv6P1g8P2dpB8nbNx3CSzRXCbLq6Dq7hYJHlwk05aRRIX2Sy4TvIJVItnRZrNNmu1i+u0COZ5UiT5l6RgULb+twiu0qtFksxa8Sy9iafs54z9/yLOPyUM8HXKCJLHaZGyAQKp0hnr5CaZpPEiqWEHoioSM0k/XS+CRRYsklnAARbBZbK4ZR0Ek/iuCIrkCxZNoMomjKxWw6bjbDpNxguGaBGkN/MsXwST5M9lUhM/buLFda3GKHeU3cTpjNWfFYt4tihqtcNB/+w82A++1QL2X/3j297BRX1X/MRPRZJM2JdOO9TfrtK8WLCPXfLtJv4aMTTZ1y3r612aTAHES/I9m33KGF3Y15/ZOBNSkqfzxAIyZdMTzacMY+hTlNzzf+qHB4NB349zd32cX5TgvPUQnF+U4rxt4nzRf39w0ffivO3DeceL886DcL7Ilx6Ud8pQ7vxkonx+MTj4+KY3GPxewh4+vA2m0Yg7nx+DuQHERH3HRP1976R/6sV65yFId5+CrdfC+r5WOzg9fn9wQlbmL/3+ec8cxDhDfF8YpC8W+XIMgo6V1A/7/bM6KZzn2YSVQlHvl1/qnsHTGUB59SWeAnLW2K+N+VJLsf/Ri+FWOYZnB+cXHwa9EiTfH5+8q69YDwTLrh/LlxYnv+31zrx4bj8Wz4/9/okPz5d+PF+swpMxwNmgf/Th8AI4oEFlsyn1THniLlVjBYgfeuYpfekY6j/3BhfHJ8f/6Q3Yl2at9v5g8K534W4TDSbYcHYZtf7MF/UwaG/uwK9p9gl/dNtNW0w3XkCbbbMWBfBKtVFisvGS1ehiP9N0lsQ5Vtwymr1UzQxR1egA9E7b6oJC6pC2UlawkbE6L9oeNNnfL0gLICerj9VfdD0YurRAsgNqMKxOt2sOoww1nCDArI0D2i7BTHdDZhE6a5fS0PjdRPY7f2voBW8O3vU4OTn7SBaQXZ0d/+c/BxE0wjqcrzRf2tXfDD6cHr6Nzs84R5gwQ2MGZZPfDwan0flFf9DDFpxbZeHxYS86HPQO3msU6CIIFEIW1r2L6PDgZw5RrijFRe/7/Yu3x71KkIrUB4P3vcF5xBcKx1AORgKm5DAHyOg96J33Br/2op8HB4cXx/1Td60x1tix11J7c3vbXivsW9u3ENjc7lhMzqruGFzM+tg2eZS16pr8hzzjYbD2JuxgbCjv+7/2tNhq1E/7g4u3wFxh0OoIgjHy9M6BJq0OK1Afz/sfZFVds3eANUVFJo1ODk6PorMBm3TshjE25+w2yqJ2U4ir6LjNiqG0XbvoXxycREcHv0OLF21W4zcxV1F/cMRmDmvi57cMOv7s1g4Zt0Vve4Mj9nOr9v74SP7odGoXB4NfWGv5YbsGf0W9384OTs/Z/EFf7Psr/vnn41PePW/LN/fo7MPg8O3BeS86YSOUpTuy9Pykj9L/W/30I261Qf0UOPUVsI/8gipB+x7V+Oj98W+abbDRNzXVnXaoWWkrJDzUlfsiQv/mr9RRm+fHdSqZkLbt7u5riFbEmKfH1swxcPuO8e0NI4uYElrz4Dc+MTBPvd8Yd0fAC+wbm6uD87eRWEXwYbtdOzn+fx+Ojw5gOUVs8z4Fana7TLT1jtiyPzs4PL74nXNIjYH6tXcSHfbPYZfb2WzXfu6xWkzeHL5TbMMZr3/WO5WfGmxoPzXZXn3cHzBg0Zv+6QctNVsd0BrbEWPATbFk2qgp0y9QJ9rWv7tshrfVrxdgO6hfW7DKxN/buww+b3aPtt1hNrtKPy3zGOzAkJmq6ZhpoiHai3m2XIAVyhQNZvgys2+SXAXR+OpTY4ytwuBzchcyo/EqZlZycxd7SK8CXhowy/c0myW7SmvJE6YXzWR9WRssXbArx4kCO0nHEhxpx0s3menbMPqtkTqsMF4scj+CNTGC4jqeJ41ZfJOEAVOtlonoC/9m88CUqwaQKLiaZvGiwas0Jb7QLtjfV/ufgyfWd2qzfdZfM3hW2oJt7k4bsMo3oUjg5aLFdnd/K1bSmVc067QrGrYbnc12sBGQ5sZ4BW1v4vxzwsyTOGeAmepaNLLLIgzSRXIjqHwZMwsnWFwz7vp0PV8uwuAymWa30dVM/nWTfWE14kv2D37lf8FXNjlc/g4B4AjhjZfFIruBZdWAvoIsD77dN5FN6hwbtg2wL0YB4le4BWr4AB9K8YMxXF2Hswfvn4OFsTGg8E+zGVZUhK1Jk4DWLYB7KUhOnOVsXNeUqgbOa3EvUZ0StaofQW7ej6R9dT+8lupHT1N1s+M2q6w2W1lXrU4QQUnE2vApSGdfktkiy9k6ZrO7DxJFMJKeCGQp9Yuwlq4hWcz6gqymvimWs76YtZI/l+k0vczT5Y1AnTFfJeMrUSfHEuxRKHrRxTfzabpYToDRNYJMQsAIg+comTpJ66dQCjG9dCg3KXhSomHzDQL+mQuAIBS0NKocWDItEj+aZHGWo6nX8mo0Wz40NQBNwxZF2Vii2D9UXTRYb7OJFOGKxxbZ7SyaJDdsl4vY1ga+GkdOsW8MJYDhiBVw7YBeCR/bvOdxAoY6a9DGiWZgUKoSXRdpGHTZmDTjbkFl6Og10zHasob+1hHfOkQMFdfZHPrBH1cMAxTh6cwn/2CcFUKuvpxNs/HnZBIBUBCHw5HgaFZ9OCJ7sHBbFKxnNPIQEN9CG2Qm5dgZOrKFhqGw3wB1DOpOk1lD1msCwTp8vC/pbArSbmBbOYPZfM4kwWwRfWH6w+U0iYrlfD4lsxgG11me/pXN9jtiGPNpfIdT5J1TXmpP68OYgPvfI5gU0MC5dSR8cKEyjtDhFSrLSHiW+E6zyBbx1JjddDZJvjJVJM6RqslseZMwbS1xEYIqfA7d6UPhwwABjflIzWlh+hJT9YQSIrvOs1voEuAKdkqnidGDCQTaQBVoxNqahVLfyBZU44Pqjr5XiRiB1fC2kPyHZxTeGngWwjrmg/rMCFNH5qufMVX9or5Gm3GezXkb6M1pUDKWfDkFKYeIEQXG/g9du8mE1dQdim/oVG5WtYo4x8L/MwpJUEIV54sLuFkU+EHFnxIBokXBeuuCE1KsEI0t+rij5SxdFPYS8UwjgnjNBCfQGPpmgg9INRQO0ZGfmnq5MHECMLy1kinrAYBuSHmgwaNzfR3oYj+5SWcN3lI78kchx1+Bb7rjRBwkWxIZYfIUL+BcpStxPvHjqPD7LupLqjArcw1KMBLAFiZahYE4cTD3FS8IZ6BsXzJOBe5L2zeHuuLI2OkRMbknTFIhVWKm9Io1KOrFl0UDqBQP20x2tZBil/B3E9QiVdghhexvYjGyJYDSr3GZxTnbL9O/wIiBL1QlvY6nV6DHqDrB8+dBlysIbLbTCRPchXJyoRKLTVrgp5J/EeUZS8tLdDt/G6Vai7lGfEvtco0h38zG46TAvRUk0XI+TTTW86xIwWFgbBfyI3C5BYv2P5T1GIVH+kd7NAr+xWTwSf/wXe+o7tpdBB/Wme5guNsZyXkCgzwpFjhfDQk6DCrnTOqMjLMVrpUTTmj9Obnbn8Y3l5NYHMrvBg3NhhoBXtiUf7CRqz8ZD1oG0OVVEaHki4pFMseFzXSaIlvm40SB4rjzj8BQwLP8F0IMyG9kZFyCPG5A1Cb9B+S3qs3mS4Lfl6OzWWYIJ1zndb6PEa7fR82OUwvL/lwmqOFjgEFjyAGLpcxMJgipYBoT/7yL7HlPyqTtL8xxHiOBIDVG42UuwGDB5jxjHHu1aBi6qarkjghHkCfxZ4OnuYY7YQrY5A74Gr3UZqPZVySpAM3oySQKNJAfOvjhzhTAQhNqtIO9fYAAzfYoBWFvUIUds9CjXnjVJOiEIQe6OCex0M+KIQc6GvKeR6j8iIW3JmwOEQCw1nKsnho4dbIaUNOow6cqns8TZqSxSlpScVZFbVGiX819ev55W8IpAldRA8WM4DQP91iVfdzjCNAynOQ3SgoJWMmsG2aBz4QXBXR6aa2AGwT3Cvi4yp2FlY0CDUSoZ7wOry9OdqUzTfwe4ebImns+IzhldPGNRJophmGlkPUbVMSkyRbXSW7aNKrxj7JlsM+1jBlrrEz3KZY3rorjt2Vq32tylJkbYkqc+pbpZZSrRcW54TUcZXFl2xojkzbbziKzTlJMWKxF51UbyGeBAt9FtxrW8amE5bb9qbIpHvO0hFOkpJx4E78wzopAjY5QW2yIT4azR9rdaOmLCpz2oAm4K457GDycLZ1T6AAhYGRJWsZzhjLf0ICqO/WpwbB2ImFBjLPlbFE04JMYKf8Cmymvssu2GW6MYDAiw1uo9Pe1x/gBKn0A5mkPWS9lphBGHQK+vl0Jvg+h0VA2YDsZuJYM5xFWc7xGJnmI1H2AKHsEJUudOT9G8AnfGLiny5jCmDheHvKRAYICwCZwXtFo2t3hDPBGSHn8Uk79afqFKefZ+HNUjLM84b462Sd4ZLF6qOUB/pZqOneeCJLKXmvENxlIu16GWI2efiPlpxlkKxV9qa1U/La3UvoZAc6ynOHPlLlJJGHzf5+LPsoaCq81uDshSJexOx7LtTfbHTDMy13bApLgXbY2J3C2q9t3dlhzk/QcLn6Xs0Nn1xnDMxu756ojyQN/LuNJHrMOsiuvgSZmmy0ScGMZdmaFcS2P5uqndVglcL6CtdHfVT+vg5XfqH/Ewq9mYa+uDC8Rpj2R66SYZqbslGpPhSRcaQlJlzsrKRgEDUr54iWJKFwIzBhJ/38yASSqbVReFQbAqkq07hRsbUzBuldfU6QheqdP0WV9jn+fMyIZYkg1EAq6xH2FPBon0ynBR3Z/F8otRgtEPgrXv/w1VNuLrsxaN73eZqxKjBugpcGCDWC0psGAYB7IKg/0SUvLP+COMu0JkN3AlDVxHPAXDAJn09UXkVLSMGpIOCGQ6ivxMPJqBVtqxM7lS5ejQKOChnJUoxCZEhsTYEUC1wGQM4ccYcQ04r3iXgwthrvYwUjPIeEr3YeKZ9GxzGJ51XdVV6FZA/TcghVXdo+9745Guq12HCLDbyZfF0A22YmhKWGNkKIsVz9crRBmmDi7hbUZBkpdHMfzeJwu7pzzQbWteI6EwAfBqpFIslYgfelsDx8juat3ZyZ7sZYmp9Q3OAC9A1eru7yab8fn614cS/HjKF7tXqroMNi9wA1W2yXxBPEnGQwAhxwyCK6mPOAEhopsK20vw+Y8p89WTRJSR7F9HZTGy7Fx47TsBTulUMHYQ8Jv4tlx0TB8aJG0F2CNSb4I8ZzA0zA0gINXWUMqjIkVsfAA54V2g4jeXhOq8s2rHXLmGGK7EXFMqnB1PLp4ECjeUgK7Nwx/3HcqlGjidAE6iCFqSjCSUzx2LY4WsVeGULD50hQZxAELzqVdz5mgo3Z6ZToseG8BegD9xy98dEOoMfJXUXQrq9R0P7VWwFURW0iXWgk0vcwlOKKnGxaSKFdKUDZN8HaJloHaIwWChE2SNoz4J6ovP0hRoopWGPzFaIUcVqGIWUtntdSGnY6CaUo1ukhz1UMhlrJTOfTzsTjuHy8Ya0VSy5qyXdpoPdy1ehlpTwE9LKFt9JLwqcCStuJsuBjejYZfDR9lhX0trWToEoHAR4WH0OTomEzth5ZIzUS2FiItLor0k9ishChrh1oWtTlzAIcVWgGFNRkVcC7ANSe14M1/6LZETGlKR60QUlyb30VSBmmJgsnnnSglN4pU8BHTqRGggMiCThaBpQzkFWvm8TEa/ntccaMU43MrIQ6NsiG2Go2U2C2GloLuVlYVRiXYVTYyd+pqyb5auq8p4R1hDDFkkiAV0rlSW4J2zfItoUQCe2S6lsTI+OrEc4S8zpWUuqudqQEYchuL6YkBhh7aJwiPNOxA/jJ4KIaR1WVz4wwNBMVfzvFHqa0HVYdCsydmX/oVxQQsl4a8hqCNh6bX2ASSwwUF0ygESOqoQshxSpESY2eRx7eXSZ7fRcpegrBvWpuxEUh/7MGW/dxgUkJrOFofY9lM2ika+Wd8NOYZTdvTo25KLutAe3tYVU35RQurV3n/qLpXcS/DaquuLLWb/gn81xOTg26kIbia5sDof6VzPoGhgm15B3wLEZiwzoEYBiPWxasS/UvQSvCiBPDFTQzOfJS+f2SXwadklvBbFEKRUi58IVlIIHPydTxdTpLJPmMnIwjBOGIC05JE26uND08BeMgsLGvnqEA4a52YTLHDy84Rc7J/IcQQYwPEzYcm31t4V6/FYZ+OiVkyFe2uYalbSC4MrwjBDNtvy0gFJNidae0ItQjVBBqUeoVOAvotXkR5WnyO/C1kqfQf0LKMKSqRupRrl+oCNkuTzCrF4LfC9zG+nCbW99skmZC6P+b45CHhk2UxAbYe6LYUkzVUE0SsBg9CWjW6Ao9kxt0geJW9JBJSdcDnuhy8GdkGGSiS8RJ1O95Se1vwyHFFbzYXefpFZcs5qWWU+iauxetL3fcVhLO5bu2e9JlwBXTBl48j2y1bp/mjSSc28ocM52PPG1aiQPOFQxUcBgnliLkupBO1AbDF6S1fJ6yq/4hKKPaw9IZi2Y0sBV/0BzARW6GQqRNJf7SHMQBbiJRRx9VcVa+4+5RGAUgcvBXQrWZEFQ9pighzpM1VXIWCjc6FnqbrJIaQtGz8GfWsdY/KHZXCki4EpOvhXLcTeYroRLt6D2SJgNaTCKcpbNebICbGRiwUQ7Uf8y7MPvR+vAJbqUfSGx0Sh3GcM52CoNF+isiCkhmsDDXw6oFeWlrKnH9uqREDnbp2oBcLa1ZLzKoy8rG+ypWwpqV2AQQVM4I5oKL5dVwkXp+SqP+ggwCI5bSrwbc66klM1+1uBRtm8XW2pEEHxk0TnrcpEscLgkNfdX4Cm4VBVTE/RkUmJpzbzU6kTV1W6XGBdJvln5McTLkOQxCsI63JXDMBZEYr1Jy7GVW7b/n+8lr2q+DAoZlvsW6UMgAEEm3XLLEnR3k4OD4/Pq8bBxz7wZZb8U2/fwFW1pldt9vxQO2/P+t/OBXxy/LroHcAd8DqksHiySRiNgOn0R/K9cz4iy3jxV2obyOHxN8Zkpnnf8/Yzsl1bdEdJCHbrwufxBSioxcRcNF+90VYE0SA/qT/TuFvHdVJTOq7MnQQf1qejTqiyOrQe9JWFXE7dFdEkMtAaLMSHxOrxF2o+MuuM+O5l+Cf0D5XhGHDqeIib/AfdmNCCjEk8qVpnyQqE0cIMR45gbv7DNKtCbUbgk1w8QrCzpgBG2lhAAWMOTvkqI+UGld7RBEXBdb1syJi6v444RHhEEMre2lRoPzSgnFrpuTSsgPy9T6RrEKhMGv8DwGu8hGNULmzbMKrJGcDS/9K8ijPUi47hY2iyIRm+r4T3aiSB0CxMFW/KavfyqIi0pbcOwuwvdl+uqAfQCUqjfyBUhX2Az/smB/1jW/imjQmSHLFUgfo0lRKKhyXfNRwhWgsqUO0DHqry+FAdblMsyD8yxveCAfZ2teMjItuSIaaMb06kHWXerk/iWC1bZCugO4e3oBF/zbHYS9w7l/pKIRstiw4PsIL2A3lxui0wlvG6CBr6gNqhRvNgONDcIci+JJq4d2XJfh0PQfZBOJLCrFrQNwuhVgjNriEVboixE8K4xnl8JbDoTLfB9SBTYMv5z/IopYOVoZqyGT3n0u4kQOye/VKpyyilpMSAlhack/pbwjqe+DSfsjKeuySElY7xr9lU9JGFFhuF7F94F17t59VDgExe+kiHePRjYTzmlroWnAB/5HZNzYTKJOQnALRs/HdxFTxJJCF/SUo2eogpvDPHhJe7rNU0ctSDK2p3JKokxQavOa69EtM/9Le3NpmiwQZwzKcTRWO/veHN5Kg635i3bkfqaZnnpMpwa5yjRlHUqAJmgn4nP6EdqgIEtUZ2YCTN6fZbZI3nHMrOdNkbo0oQMHF61/dfSjV2u4n0aeYlYcQ8O3B4FfMHtYspw1k2oVsvNF1DPl6F/XSozwuivSuJtaKxdlG/oSFjK4DDuPj8O1MzyDTYFOOsFZNO5duFs1Uv+ZnH5mY0c+MvoFDIEkcOcgIB1f3xZpYZOGDk9lNR17m2fduzh1Y3g2LwWSijFdPvxh/LFsJCqzkKtRCCPuULiug0PfR4IX76cFD97OLb+A2x5RzDdzJD/bNQfN7+lQPlIdR6Sz6lGe3i+voNp1NsltQCDQ7wV4B/gG2S0CkOFGxrDqKX1lH1nULC761vlfopt83RZ2/ZYow+7gaZeU8ucEd/+dXHsxo5bITwTXV07z7YCn8wpbCztgeI4G5fuiKX9f6hrO/5OtC3vxAv+WPuFj695qwGDS/H1R62IWzNRt/FmovVPBg9JR393h31Q718s6VM/O7zh1wmObhgrpLJPKXGCksm4Z/iDMoH0ioMZKMhXynbUDfmb/PAFzPMw4+OLua5fpeGUpPA07VSR+nbsKvy2Ckkck5eO7oZ+3vXR9AKx0WZFvWIf0iaE6+CF7wrecajVeWM0BWCGkCKETFmE0l25Ym9LL2bMamV0ZhIl3wh14K61zIecBlnMq4hUCd7OK6o9itH9JQcYzuv0ot80O5XRgYiIsoRoQQv3GKcEbNqiNweovVe1hW3Z2KDDSvuAqzDANFhPsF2jrEcw/ERfQrSY+wIvYVQ8H5YTv+ybPa4d81O62L/5i8zHhcNZ+qZ3okv+s5/kb/vHU1VOE68oVGoBR5vU/barV6I+j64yDKFZ1qhadC3VY5ALvt8sIy7UdrDG8+HJ8coSlvjEg/QzAKm+XtpWZxuUyZhK+C4ncS0P+M86auv15znagI3BD2MOvyyoAHPdkVUQ+yJUp8XJyqGe4xr+2zZZcJDJd2BYfBG0HCzN9SnKzSQfJTdR4eXu7JNrQpV8lyqjhDEpoUoU175E8AptUuo3Ktet7U8aRwZmyhs+wlOMvU+IVCz/ZowUpwfufBonplla+qLf9ncmr6kLXUELsFkXQlnC5XDPIZrhhN+NpjlkXTH0dde0TUnyNOuQh1tsfaKlGLqwU/4ikcRnCFcke99+7ulXL+sZNcIkWMF1fWn+Kj41/q4YppHU+TOKf3jupPOqu+iDNPfNyPpGnJwul023p7DLZF3tkfRudJ+imCwL8fT15fNOWjDrVWdtlwgsydhVWztyRZwVVejN3XaaeCeMVROeBeK9m/fiQ3dfyff3rcCmVzddhbQwjjcccKuUsE9TgxBXW5LvMDJ9kfQ+yv+6//XoYo0XS72z9MlOQJn+J0Ns5u5mw4mA5akeoxk+wj5u7KVqX0dCKy/Qe9/hysOrvNylyrvhyrko9EGmqPamcnzPFpeWuh/2xNTbUqZ84KLZVUXUNHfSwPt59Yt6zyVHuPH8X0rWJdr/XkOTxnCs2XOJ1ibLeMEPAaSE6Q0mt4wmiFafTEksJxlj2C4If9k5Pe4UVEna7r0l73/yjR8bALMRDzuiI8w3PN5fsO1flJF+uXq3huDbZ5m3W6L9sPC1UAc6E0SoG+SFh5znwF+qGJSf3K1Rn1nDwmGbnXh4Qt15GmzZorpg0PiitaIfAWzkTH8MCxEwxKCiOM1fLhbAekgrO4LFbVuQ8TUgTDgCZ2d0bDdrLoE7z9y4O/DdQxCaSFB08lQpeDkdzdRIYXwQtZlqwXCdVtHGynnU4JbgNQgQJlCtATnFt6M87/0zvgUx+fCnqvilsw3N6Vu46dMxfzfGD8I4mr3xJSx1cXXirJF1fxtDSlQNs3HN5Pyzl3MT+ZRyZlo+WSH9/Ohgz9URGj6HAx8e2m3a1VGy+e/uzBk6gva5Vbv0sRWGeQG91CzqdT8oB+CCMtB4WzsFWqEzrD2yjJzdDZaa/opP0DlTnzVsWT6XQPVzHKzw7omhFUqpdWLqNkaQO5cWrOZzzhh7/SIq4+YzSuPTrKqakZwR5i60a1Elc/EeSyxBG/G5BMvXRPojtX02pnqlN7a20yuGb9ov6xYn61iG/yZ6KealPz6N//PbvX4cGgV7l1wUx7NywR8QA0Uc+Bad0/xoS3ztMNIq1maNynkw8aVmT7dK7fVLwOUfOx5Ib5MIUJQuKGaSWdVJkyxeQaiULNhU7739s3hqyu+/w7OFrOp0xNXyQiYVkrvgW1EGNFRPaaG3gTgJMY9O9IQOI+V/lEpy+YxJNv1YzqqKL500adrHdvE++R0YAiZCfS1ng2tZ7M0yKbJOesHBT8V1243dkSGo//bieByK94YmAjD4zRfqdgf/WNT360JNkPw1Q4geH/mdU7CoNnlXc8RbykjooCEBhHwr6JKBgZUrVuMNWIpA6E+STVmiLtlcKY5qLV1eSdRgjBsVKOVOUqIfcARNDOI8KdVKRRdLmc8KxMa8cbifkCGsrHI3jiUySdHLO4QQXmbySDiyDtDNSEf1Mu0WAHZP8O+XXJETl0HulYM7wH/rgb/PR+iLhsK6FB6BpAZFPib0VZgk6vSGqVMpEh6DBSIS5kaHrSRVVJZFbYtMs2l3N4HKjxjZ947gb1n497J0f1e6em4hvxW6DD78rjN8ySTZJDqTmAJWtQoUmaztPx5+VcXkVTvZqMHBrJfvCt3XGSThvQxMAgJDMHcX0vN8nDqbrTPLlhlrLAuBKGccUHPS3oubhrIPeo+7/IQW06FXr0apKQU/N49ilpGGNv0hdDSVLil6GNrp4VexytfdG2dN6Mfe+bow8oBjg7Pnz34czjd6rD6QrUKHNM1TkGrIq4teHWIBem2/RCCKdsmaNNXaAWHjdPK+9Rt75VrS9/y//uaVywR12H9VUjimGJBOFLGKVI037JFJsTeaJ7CKz0DmtJF19kLOFsucKMXpxEWUrqlMfZkkBfSwT5AyjNxV/z+Be4NDDIS2WCLgr9PdjLteYxseTi1cHG1+uA/i7x40XxebBlCx1r8bvdl0mArdBLhqaPAP+MLKi8SvcwgdCtWPgvu9+9vKuTsvCj0YdpDPRtDcVxOlGuk8DlEQl6PFaHXDW6I+fBJbrQ7Yw/VUvb8m0jb0Y0Tb7RC9nqNfXouhYYWsYnpZJZYjFA2Ro0pRRZVxRhy+avXAD+RbDeQqCLQZDaX0cthZJIELoSyqqQbbCkRsWKMFcFWRnyRfanue+SFMyISDhXPiAJjdjCiusI1hdqWUYuDmbbkDBX7VbS9tcNGzG1XlkLphRYUlhkQzbusOiQfKptkzfvACuZTcBZa+TyiYLqeYWqqXJSQpdng/7Rh8OL85r3jIB0yOzS9qoEslBb+pofjV7NdzxjcAZpJZxUKmSWOK78YbRrjv06LlCli7Tl4hVsZBiu9SQP0fD+v6WFm90V19mSGbATnn7B7Ml3T1+46BSPv94Pdtp2KZk80wtJy0WqUMha0XZqNOgygJGYTbba1iPF0MT1CGIaISU+Rf7p5gP8XHStgUvanhxvRYJ6yaGMwelkBp5WXh/hHRR/HZkCBDMcgThYKY9bHStsSRy8rRDTeJ2sK1IkdOHVKT2TzScU4PM4zS1/AE8KFglxl8e3kT8BvM9XpGqKB3Bpa/0MrvFVPYZLfU3cu6gk6pDiZD5sJGZboitXv4Gp5AhrYxeR26J0yBmgPGsl8gVyqmqh2AEfPKVIrn8PR8Hn3pNdb0SHRpMajd5ksQl/ix6WHb5HLxN+mLIPc3yR/W6/7Gny0oeYVG50RQ3Bf6OqN5vW8c3T0AbwIG9oAK9NMtCcY2HQfdF84BislSnFleewwn9es84JhnsG8wAk3evgBunXfoy7+YRzoqO+uWKrOMw7LJPtFK+g2jvyp50U53ArObJWctyOhrbsSLvZzMHgOzXekIKzwXF/cHzxe/Smf/rhXJzGifNtJs7bbSaV+dUbnrDPAbAhVSCJAxfqnoO7VnAxOPi1dxId9s8v4Gk/MREVQQkosEu3Nv8ctHCw/l1DvXvmLTX2Am8NU/5WHPLDg+xVxe3Rw0IEmr7tbFmA91Vl04TX/5q6gPi/rRL92o0uQFVHPjzlPWKCGoeiQt1/9xvUkyjPshvt3zbhtrzGDvH9hHqKrK3Z2voi/J+QDXBbPcsXIBWAYcynBSkYfFqQUs3eU1UFSb0VNoXWv2WDoQFvZL2aFH1OPFdmGZbWvly6R2Zz8SgA3YtEukvnENwj8XV7vD6gYrYCX/zJfZk0FsNQ58UKqhNTROpL0gr2q1Aa6GM5fKhGsmp+U04kg7MfxVmXkOJus4+M9hhEVUbdymzqBlqyTWdUlk6dnO5xsauaP15VUfRyRSNmQhVYeXc6vUfue7Yo1OOjy6uCp/DEc+OGUAC08WRorv591UpO5JsusdXurju0ofI6Bc6Wa2yPwrUkD5z/eQLY+s7DLAISVxvpt6PdoTFS7D7Ov7HSJWHxdBVkJ6TsvlYyAZ57+XLy7QkovbWyelZW7LF+ZdRxcZn0t952NIPVZtA4qMwhjj5h2YOPyJbHC2OT1ab7CLsG7xdwxBCeck2oPhR4P2Q1M0NuMo78epfCsbVfNUI/0SUTlwXo+tdCeV6BNV8AWyOGzXQg2j5AfOlR+vvw2bDSgDZ/cJtsU9mkIoEC8l3pc5MVjf1I/rlkOxRXD+GMwyQ3d2iGeqr9MPjrTRxMOb9W8qzFdvKKJ5c2EvgafCirOjfjBXBMGCj6wY54KA2GkWoTvd1cfY+PaAbEWWG/WqXUgaGuPwJEO4/Rb6iqtRlPJg2tiTUt/coSkCrkq2bAEiozwqL1m2Y17ZCcqBAYWZFGMJJXm0SY2K5Chyla+qUBvvNK8pP3kXgcmW7V2aWlRPHFPPDql3xNCUIO3+OaFVJvnsJLtvF0mo3ps1tX6WWD488JHIfBJVxfCcW8WGdrtCqpfhkGMbOVLykJYvU+hAgmBvbUDwZi1Jh+JXrftLhE7jXeUj9+aQbRcaVvjWg5J3IMOzci+MQXTnURXkdsPV84A6gE6pHrmvYDVB2+eBOTSTo/7lEUIuicAy6xwJ3HPq2oQPle5ut9N5iQATOlRslm7EqGEm1PLEOdfU6gwHuQZghut7ve06NQPNlOTq7v/O8Z8VkfJ3ORSBaeOtRzwaHwuQgDv1CXj8ER7ZAnsZRQS11bgnf5xgEmE/9tqo4bCpAn3zGX46UVMBlJpV50mSfxZ9MgMYnMNxccED7ly1+SwAMpS3Pn+olobVl9ObwaWHASA11Fre6oyTNkGIBfiOiuWsUc1Vzlg6ellx159vV1DYDQS+iw+urQI1jge6bfO/Uy8yCHo9/eYbYCD8At2JBjCH94iIQVKSL1duWYU3SV1CqW4iqZRZ9Z9AnzYcnjKushVIbUytceiXlHABvW3L0dDqdjBh6w+4y8e5C4A8j3oVDpAU25Izkx5aPa6shvj4SviPt+zHahd4Tv2gRqPr+fElG0Q1Jh7c0B1QNnh/AlKzN8usa2LrZzI1uox/z97k1m9X4CKChpopK0rrOTTJKpnoehmYcNNwWTuE+zKVivqErJueZZ6n/5lrI+J63YeKrZwprUVdqCXX2/VHGo4pjH86OP18ix4z+qgUDM4FMxi0sMH584cy1Q8M40ITabZF5zJTvsByVsu2FD8CSWNOcJU1Q4NzrXDMMqTauk8FVva7fsbB0G26pqfs41uVbd4zFeLryGUAka7ypfZ06m00i6TASlruMvCYkRFCnO1MleCEmF8Tk5Ma6qhMsyVdye8XSswAo64tf04gIP8aWzLc7jm2SR5OJiFWb4khujTgaArZ4Fg955b/BrL/p5cHAIt9fIuiW9v6K9cwjPxNYHAUoNrNfCNwDw1kxNhWjJrNvwANrmq20vnPbm9jYneDafM8rMFlGxnM+neKtMffqSFjzpFBbpsTGSs0n5K5vti5t6i+x2Fk2SmxiDeyLyM2LsCY8REMLUlKpg9vyagqkY/atuGHQ222zwjBIdSM7ZsEG1KKimDHXN/sAUnpFP14GndiqhcBVsCam31ENn/I8olkvW0wUynyDSNRvKdYapXMSg3EnbwzkTchZTFhJGoBCAFuo3BKzt7EAcKEFQqI3EXdomKqmWItoRqhlcSg57hJRcsh0fIrCbwsd+s0h1oW7SS7XDuoLjTLs65XjpZnHsmqGJBt/vvDTlDnUaM8pp3MV+ACOHnK6012azSvDaC23LEbulxCNMQe4jAyY7MJMvgKdBWDTLJLI1mlqZi7x0+/MNFy7qevc+KhO7Rqb0sdrJ9SUbBG1l6FfCiurk5VPCtnYu03kXZkp/7S+XOwKbBwitq7jV+rAXbUVm/epLruqh0kJ7PR/xmrfDVhVPZ7uphrBT9xHw0ne1vaHHiIbOwuuBJ5Ly+o9AILTfbeJ/LX1lQJENee1HwK0FghOj1KuZyo9NVKuO/xgAjjuUG4MbhS0VKEGf0XOPqmH6eZkybXnPpktFJmfgByHoMngqzn3ozfDJUt2hAuX74TfE9+mzfKC7M2EiX2DwPJWMs1aQaduwX9YuKSRvuPuKnZfqqZd9mmVwtbLTprmYXgnXIMHYSg2EzzaLJysgFvw7X34mGAnNSEWUmvKzoSZlQ+MAWt6rzbYpCTFFOReX7w9+i94enB6d8w0NxxyaHYHoFawn1Na101YIF+Xf/voJXhFSieLLrw49XeggtzuIs9b0ffrcuPSRIMOfayBEpl9Qn2qheN9f3y5e40KXau3e6ZK9fI2yfMJjMl1asGJ+6tjHOmdJfsFYCl80+i3i5/5Rf3DUG5w3Hylb8Jl5TrM1n53n9ojEeagf15V5737iKZKQswzLUsWOe1JdmNbyelLtIid+rmbNE34/a3TEXUqjb7IbXKd5ImlPso6kQki5K0br5rzpnuxPOl04acB5rCfXduviW9p4WIxAHOcFX1J7vOZG0PWY5+ZplJ4T6ZAe1t8eD3p2aDUHDHE1DLKZDxFHY+fQ8qEv5BovGu7qUQpWyMbj5ZwpKXci6qnkvLbiASn3ZSoN09JZjeUrdziRjDKdM4NgZtyatPYA324hku9Dfr76yHrTSmdBkfuV3dezAJw70flF//AdCkpheS/g5UqS+US/C+b4eDa8+QWq3ELlJyp+GVkjceHgqrGuOdnnc7vkKuGXxB6A6Zar2R5r8vAIMZctd6SAq00KwTOUbi1C+KbRD7Ze4wgdLldReW6LeDUc/60rIW+PeTpBA5bQ5LliIu9x6Ug9nzFl+cgc7+x6DjP/BcCKoCmHKPM8Gwv7yIm0qPIrZFdXBUZz2zYycBAv1HEuEiHvfVFgQCm1GhIdO0RLygSsDPH7jTz5wmROgruAjtQoa48yBhobRh2V1/zxZa/ANmWtLWfPeycnbkgZSfWDMncD/R38RWeOogZIBKZgfP2pFbjEMw4LVrvKvTD++cVg9+j62Yn7SmkZIrKIRmxh9IvHH+zXScw6D9JLHqubuFbzg3SUR+spT6CrrKGvrK+z+PWWdXWXav2lRIdZpaxgjg3+3iGjqsxjzaNUkdDCloN3MPRVqG+83q5ITuVkQNGOJu1f0u4lqVPsel+MoY8g0Xb+qzC0RCwgNf7sdmaFo0i0+aiHItUKm61I8DK+9SEizRV782qmluEMmUSiSCLyIMkoY3MKjjq0NPAi0Zt+/+L8YnBwxq8SvT/rfzg9qt+bji9hmaCP7fj8+LxuuLwrrHvjRoZRd62cGYyFf9r2t4eR6Lc9oK/XVleWOqgTWZaqoCWQa4Zjt2EnjKe0rXlccXxSorMPg8O3B+e96OTg/ALUUaOufIxwxz4ImC/zMVCfLxXQDPiFOeu54MP+R5hAZqH3zuqWT5qEzUumc2VpNE0hszfTtCO8h+n3epbmVq7KDIO8LxncX0WfoJmrHoRveUtvyBtZg3JNtVahIFYQMjgQslbRC9V1ylL26BVpTp/7JuZ6So9fWDs5CdcZvGvfCij+YEtvDIs4QZ99ivRRLc/sDWordqftR7beDvuDXvS2NzgSp1BWzhDrekLNe3mXJj6T/3VLT2vs26tKhyuJb3jI4y+ll8hxJYpd0aVR+f2R58/VaxJikpgczIpFyZ2c5sqF4MyxorA3BMPe8N98+D1Sb87K7FoeZdpRBFQ3z9YekFauN8jFjkoJYtT8P2zSk8uq0XWSy0e9reUh/NtFmpMMqgZLGjAcn4ae+R1y5ZVX1ln1w7I9zOz5NSWltT/5NVSjkt25vZetZecI8v8wUye5SZiAZBxHhJeUXWVqDMIkmZCcCAGimusP0/Qm9eaVM4neokQPa4+UZiaX2kY/omKd+MERthGSoRhfE7UlIbXg3ozjX1CmRouPvsWB7rlEXmc/49bKPsLwbxC2S7RcrvlkmvChgZ4kR1oi1cpNG7+8wirJ10UeR1Me0UOzBCsTcTmbZhCdBq6mSY4HexAYffqRmVdNPO13F+hqVd1YggSJPTyZ6P3GWjCd8/TI0U2ZwgHfo/5Z7xRlyVC3Hnm10053faFA88vxlYGdnQ2OD3v+nvR6fMGz4FKK7ktZRvOJKf57vS9tXS+/+bgBsKl2yRsBC+tFTej037ZmJIwB8aJNZ7utq0rNnpxn1d/3Tvqn9bBpO1AMgwpBqbqaq+vAJR/f9AaD35FVDgaDPn6/6L8/uOjXm9aVKn7BcUawWc8zhzeo3EP+dVL6cY7gUQp4X1K+C11bRyvkwQrYzkqwuR2W3U9TmhmdniYoX4BM0ytKHyxizvn7yJwUq4QLUZc0v8l0UntGnDANfcB4wfVCIZ7+aE2g8CMO2GCuumyyXgh6NP/Wg7baKg8VDwSYJON0ktAAABonC6QFMqvTaPxAb8xIrfAucc76+Vd7IsSVlEYbOEI03FP7SkEvfcjIHH0vV12+1ZduhyM8y8ZDcvh1rxAXz2sUQ96LQJUzFB2T+OSEdskoFgwOwXAjzZnyLQz+LN9D3wBZdf3YvvPqXjU278CSCooSjw7xCI0gpZhpPYsy9ljkd85seTiqKdQJuGQQ9PAf4zpSJZfhGVwRLxY5B1mHZHdNcRd7RI2KahashtQmx5Czicqg3HDSGxscVfaMiC0RynjdyfPhTR/tMIrJLO6FdINTPE9vcBVaD7RptyXLKSR36/4//IfO6+7qAAA='}
DEFAULT_NAME = 'V24 Clone Cash Shield'

def unpack(payload):
    return gzip.decompress(
        base64.b64decode(payload.encode("ascii"))
    ).decode("utf-8")

ANCHOR_SOURCE = unpack(ANCHOR_BLOB)
WRAPPER_TEMPLATE = unpack(WRAPPER_BLOB)
ANCHOR_PREFIX = ANCHOR_SOURCE.rsplit(
    "\ndef agent(obs, config=None):",
    1,
)[0]

tree = ast.parse(ANCHOR_SOURCE)
TRACE_ACTIONS = None
for node in tree.body:
    if (
        isinstance(node, ast.Assign)
        and len(node.targets) == 1
        and isinstance(node.targets[0], ast.Name)
        and node.targets[0].id == "TRACE_ACTIONS"
    ):
        TRACE_ACTIONS = ast.literal_eval(node.value)
        break
assert isinstance(TRACE_ACTIONS, list)
assert len(TRACE_ACTIONS) == 720

def build_source(spec):
    wrapper = (
        WRAPPER_TEMPLATE
        .replace("__CORE_ACTIONS__", repr(CORE_ACTIONS))
        .replace(
            "__ADAPTIVE_TRIAD__",
            repr(bool(spec.get("adaptive_triad", False))),
        )
        .replace(
            "__TREASURY_START__",
            str(int(spec.get("treasury_start", -1))),
        )
        .replace(
            "__TREASURY_FLUSH__",
            str(int(spec.get("treasury_flush", 718))),
        )
        .replace(
            "__TERMINAL_WORK_START__",
            str(int(spec.get("terminal_work_start", -1))),
        )
        .replace(
            "__CLONE_FRONT_RUN__",
            repr(bool(spec.get("clone_front_run", False))),
        )
        .replace(
            "__FRONT_RUN_HORIZON__",
            str(int(spec.get("front_run_horizon", 0))),
        )
        .replace(
            "__FRONT_RUN_ITEMS__",
            repr(tuple(spec.get("front_run_items", ()))),
        )
    )
    return ANCHOR_PREFIX + "\n" + wrapper

core_trace = copy.deepcopy(TRACE_ACTIONS)
for step, action in CORE_ACTIONS.items():
    core_trace[int(step)] = copy.deepcopy(action)
CORE_SOURCE = ANCHOR_SOURCE.replace(
    repr(TRACE_ACTIONS),
    repr(core_trace),
    1,
)

CANDIDATE_SOURCES = {
    "Anchor V6": ANCHOR_SOURCE,
    "Core Hand Schedule": CORE_SOURCE,
}
for name, spec in CANDIDATE_SPECS.items():
    if name not in CANDIDATE_SOURCES and spec is not None:
        CANDIDATE_SOURCES[name] = build_source(spec)

FALLBACK_SOURCES = {
    name: unpack(payload)
    for name, payload in FALLBACK_BLOBS.items()
}

required = (1, 32, 2)
ENGINE_READY = False
ENGINE_ERROR = None
installed_text = "missing"

try:
    installed_text = importlib.metadata.version("kaggle-environments")
    installed = tuple(int(x) for x in installed_text.split(".")[:3])
except Exception:
    installed = (0, 0, 0)

if installed < required:
    try:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--no-deps",
                "kaggle-environments==1.32.2",
            ],
            check=True,
        )
        importlib.invalidate_caches()
    except Exception as exc:
        ENGINE_ERROR = repr(exc)

try:
    import kaggle_environments
    from kaggle_environments import make
    current = tuple(
        int(x)
        for x in kaggle_environments.__version__.split(".")[:3]
    )
    ENGINE_READY = current >= required
except Exception as exc:
    ENGINE_ERROR = repr(exc)
    ENGINE_READY = False

AGENT_DIR = WORK / "v24_agents"
AGENT_DIR.mkdir(parents=True, exist_ok=True)
PATHS = {}

def write_source(name, source):
    ast.parse(source)
    compile(source, f"{name}.py", "exec")
    safe = "".join(
        character if character.isalnum() else "_"
        for character in name.lower()
    )
    path = AGENT_DIR / f"{len(PATHS):03d}_{safe}.py"
    path.write_text(source, encoding="utf-8")
    PATHS[name] = path
    return path

for name, source in CANDIDATE_SOURCES.items():
    write_source(name, source)
for name, source in FALLBACK_SOURCES.items():
    write_source(name, source)

MAIN_PATH = WORK / "main.py"
ARCHIVE_PATH = WORK / "submission.tar.gz"

print({
    "engine_ready": ENGINE_READY,
    "engine_version": (
        kaggle_environments.__version__
        if ENGINE_READY
        else installed_text
    ),
    "engine_error": ENGINE_ERROR,
    "candidate_count": len(CANDIDATE_SOURCES),
    "fallback_controls": list(FALLBACK_SOURCES),
    "anchor_sha256": hashlib.sha256(
        ANCHOR_SOURCE.encode("utf-8")
    ).hexdigest(),
})


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 26.0 MB/s eta 0:00:00


OpenSpiel exception: Unknown game 'capture_the_flag'. Available games are:
2048
add_noise
amazons
amazons_proxy
ant_foraging_arena
antichess
backgammon
backgammon_proxy
banqi
bargaining
bargaining_proxy
battleship
blackjack
blotto
breakthrough
bridge
bridge_arena
bridge_uncontested_bidding
cached_tree
catch
checkers
chess
chinese_checkers
cliff_walking
clobber
clobber_proxy
coin_game
coin_game_arena
colored_trails
connect_four
coop_box_pushing
coop_to_1p
coordinated_mp
crazy_eights
crazyhouse
cribbage
cursor_go
dark_chess
dark_hex
dark_hex_ir
deep_sea
dots_and_boxes
dots_and_boxes_proxy
dou_dizhu
efg_game
einstein_wurfelt_nicht
euchre
first_sealed_auction
gin_rummy
gin_rummy_proxy
go
gomoku
goofspiel
hanabi
havannah
havannah_proxy
hearts
hex
hive
kriegspiel
kuhn_poker
laser_tag
latent_ttt
leduc_poker
lewis_signaling
liars_dice
liars_dice_ir
lines_of_action
lines_of_action_proxy
maedn
mancala
mancala_proxy
markov_soccer
matching_pennies_3p
matrix_bos
matrix_brps
matrix_cd
matrix_coordin

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 40.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: OpenSpiel games skipped: 1.


OpenSpiel exception: Unknown game 'capture_the_flag'. Available games are:
2048
add_noise
amazons
amazons_proxy
ant_foraging_arena
antichess
backgammon
backgammon_proxy
banqi
bargaining
bargaining_proxy
battleship
blackjack
blotto
breakthrough
breakthrough_proxy
bridge
bridge_arena
bridge_uncontested_bidding
cached_tree
catch
chat_game
checkers
checkers_proxy
chess
chinese_checkers
cliff_walking
clobber
clobber_proxy
coin_game
coin_game_arena
coin_game_proxy
colored_trails
connect_four
connect_four_proxy
coop_box_pushing
coop_to_1p
coordinated_mp
crazy_eights
crazyhouse
cribbage
cursor_go
dark_chess
dark_hex
dark_hex_ir
dark_hex_proxy
deep_sea
dots_and_boxes
dots_and_boxes_proxy
dou_dizhu
efg_game
einstein_wurfelt_nicht
euchre
first_sealed_auction
gin_rummy
gin_rummy_proxy
go
go_proxy
gomoku
goofspiel
hanabi
havannah
havannah_proxy
hearts
hex
hive
hive_proxy
kriegspiel
kuhn_poker
laser_tag
latent_ttt
leduc_poker
lewis_signaling
liars_dice
liars_dice_ir
lines_of_action
lines_of_action_p

{'engine_ready': True, 'engine_version': '1.32.2', 'engine_error': None, 'candidate_count': 10, 'fallback_controls': ['Kaito V21', 'Kaito V20', 'Replay Shield V15', 'Scenario V14', 'Frontier V12'], 'anchor_sha256': 'c8aef4f69fe214dc03929b3881af31ab881c9202572b2440e86ee79fadb697e6'}


## 1. Static source and branch checks


In [2]:
for name, source in CANDIDATE_SOURCES.items():
    namespace = {}
    exec(compile(source, f"{name}.py", "exec"), namespace)

    assert callable(namespace["agent"])
    candidate_trace = namespace["TRACE_ACTIONS"]

    assert isinstance(candidate_trace, list)
    assert len(candidate_trace) == 720

    changed_steps = [
        step
        for step, (candidate_action, anchor_action) in enumerate(
            zip(candidate_trace, TRACE_ACTIONS)
        )
        if candidate_action != anchor_action
    ]

    if name == "Core Hand Schedule":
        assert changed_steps == sorted(CORE_ACTIONS)
        for step in changed_steps:
            assert candidate_trace[step] == CORE_ACTIONS[step]
    else:
        assert changed_steps == []

assert CANDIDATE_SOURCES["Anchor V6"] == ANCHOR_SOURCE
assert CANDIDATE_SOURCES["Core Hand Schedule"] != ANCHOR_SOURCE
assert "ADAPTIVE_TRIAD = True" in CANDIDATE_SOURCES["Adaptive Triad"]
assert "CLONE_FRONT_RUN = True" in CANDIDATE_SOURCES["Clone Quad H1"]
assert "TREASURY_START = 710" in CANDIDATE_SOURCES["Treasury 718"]
assert "TERMINAL_WORK_START = 704" in CANDIDATE_SOURCES["Terminal Cash Work"]
assert "FRONT_RUN_HORIZON = 2" in CANDIDATE_SOURCES["V24 Clone Cash H2"]

print({
    "contract": "OK",
    "trace_steps": len(TRACE_ACTIONS),
    "anchor_preserving_candidates": [
        name
        for name in CANDIDATE_SOURCES
        if name != "Core Hand Schedule"
    ],
    "natural_branch_steps": sorted(CORE_ACTIONS),
    "core_branch_diff_count": len(CORE_ACTIONS),
    "default_candidate": DEFAULT_NAME,
})


{'contract': 'OK', 'trace_steps': 720, 'anchor_preserving_candidates': ['Anchor V6', 'Adaptive Triad', 'Treasury 718', 'Terminal Cash Work', 'Clone Quad H1', 'Clone Premium H1', 'V24 Adaptive Cash', 'V24 Clone Cash Shield', 'V24 Clone Cash H2'], 'natural_branch_steps': [333, 334, 335], 'core_branch_diff_count': 3, 'default_candidate': 'V24 Clone Cash Shield'}


## 2. Download and reconstruct public top replay controls


In [3]:
EXTRA_EPISODE_IDS = ()
DOWNLOAD_TOP_REPLAYS = True

def public_replay_url(episode_id):
    return (
        "https://www.kaggleusercontent.com/episodes/"
        f"{int(episode_id)}.json"
    )

def fetch_replay(episode_id):
    with urllib.request.urlopen(
        public_replay_url(episode_id),
        timeout=60,
    ) as response:
        return json.loads(response.read())

def state_value(state, key, default=None):
    if isinstance(state, dict):
        return state.get(key, default)
    return getattr(state, key, default)

def replay_actions(replay, seat):
    result = []
    steps = replay.get("steps", [])
    for states in steps[1:]:
        if seat >= len(states):
            break
        action = state_value(states[seat], "action", None)
        if isinstance(action, dict):
            result.append(copy.deepcopy(action))
        else:
            result.append({
                "farmer": ["PASS"],
                "hands": [],
                "market": [],
            })
    while len(result) < 720:
        result.append({
            "farmer": ["PASS"],
            "hands": [],
            "market": [],
        })
    return result[:720]

def tape_source(actions, label):
    return (
        "import copy\n"
        f"TRACE_ACTIONS = {actions!r}\n"
        "def agent(obs, config=None):\n"
        "    step = int(getattr(obs, 'step', 0) or 0)\n"
        "    step = max(0, min(step, len(TRACE_ACTIONS)-1))\n"
        "    return copy.deepcopy(TRACE_ACTIONS[step])\n"
    )

download_rows = []
replay_controls = {}

if DOWNLOAD_TOP_REPLAYS:
    episode_ids = tuple(PUBLIC_REPLAYS) + tuple(EXTRA_EPISODE_IDS)
    for episode_id in episode_ids:
        metadata = PUBLIC_REPLAYS.get(int(episode_id), {})
        try:
            replay = fetch_replay(episode_id)
            steps = replay.get("steps", [])
            if len(steps) < 2:
                raise RuntimeError("Replay contains no action steps.")

            if metadata:
                zero_seat = int(metadata["zero_seat"])
                opponent_seat = 1 - zero_seat
                zero_actions = replay_actions(replay, zero_seat)
                opponent_actions = replay_actions(replay, opponent_seat)

                zero_name = f"replay_{episode_id}_zero"
                opponent_name = f"replay_{episode_id}_opponent"
                write_source(
                    zero_name,
                    tape_source(zero_actions, zero_name),
                )
                write_source(
                    opponent_name,
                    tape_source(opponent_actions, opponent_name),
                )
                replay_controls[int(episode_id)] = {
                    "zero_name": zero_name,
                    "opponent_name": opponent_name,
                    **metadata,
                }

            download_rows.append({
                "episode_id": int(episode_id),
                "steps": len(steps),
                "status": "OK",
            })
        except Exception as exc:
            download_rows.append({
                "episode_id": int(episode_id),
                "steps": 0,
                "status": repr(exc),
            })

print(json.dumps(download_rows, indent=2))
print({
    "downloaded_replay_controls": len(replay_controls),
    "requested": len(tuple(PUBLIC_REPLAYS) + tuple(EXTRA_EPISODE_IDS)),
})


[
  {
    "episode_id": 89548972,
    "steps": 720,
    "status": "OK"
  },
  {
    "episode_id": 89549522,
    "steps": 720,
    "status": "OK"
  },
  {
    "episode_id": 89550073,
    "steps": 720,
    "status": "OK"
  },
  {
    "episode_id": 89550629,
    "steps": 720,
    "status": "OK"
  },
  {
    "episode_id": 89551188,
    "steps": 720,
    "status": "OK"
  },
  {
    "episode_id": 89551747,
    "steps": 720,
    "status": "OK"
  },
  {
    "episode_id": 89552297,
    "steps": 720,
    "status": "OK"
  },
  {
    "episode_id": 89552851,
    "steps": 720,
    "status": "OK"
  }
]
{'downloaded_replay_controls': 8, 'requested': 8}


## 3. Official simulator harness


In [4]:
def score_from_state(env, seat):
    final = env.steps[-1][seat]
    try:
        reward = float(final.reward)
    except Exception:
        reward = 0.0
    if reward != 0:
        return reward

    for states in reversed(env.steps):
        try:
            return float(states[0].observation.farms[seat].money)
        except Exception:
            continue
    return reward

def play(left_name, right_name, seed):
    env = make(
        "kaggriculture",
        configuration={
            "episodeSteps": 720,
            "seed": int(seed),
        },
        debug=False,
    )
    env.run([
        str(PATHS[left_name]),
        str(PATHS[right_name]),
    ])
    final = env.steps[-1]
    return {
        "left_money": score_from_state(env, 0),
        "right_money": score_from_state(env, 1),
        "left_status": str(final[0].status),
        "right_status": str(final[1].status),
    }

def symmetric_pair(candidate, opponent, seed):
    first = play(candidate, opponent, seed)
    second = play(opponent, candidate, seed)
    rows = [
        {
            "candidate": candidate,
            "opponent": opponent,
            "seed": seed,
            "seat": 0,
            "candidate_money": first["left_money"],
            "opponent_money": first["right_money"],
            "candidate_status": first["left_status"],
            "opponent_status": first["right_status"],
        },
        {
            "candidate": candidate,
            "opponent": opponent,
            "seed": seed,
            "seat": 1,
            "candidate_money": second["right_money"],
            "opponent_money": second["left_money"],
            "candidate_status": second["right_status"],
            "opponent_status": second["left_status"],
        },
    ]
    for row in rows:
        row["margin"] = (
            row["candidate_money"] - row["opponent_money"]
        )
        row["result"] = (
            "win" if row["margin"] > 0
            else "loss" if row["margin"] < 0
            else "tie"
        )
    return rows

EVAL_AVAILABLE = ENGINE_READY

if EVAL_AVAILABLE:
    smoke_opponent = (
        next(iter(FALLBACK_SOURCES))
        if FALLBACK_SOURCES
        else "starter"
    )
    if smoke_opponent == "starter":
        env = make(
            "kaggriculture",
            configuration={"episodeSteps": 720, "seed": 12001},
            debug=False,
        )
        env.run([str(PATHS["Anchor V6"]), "starter"])
        final = env.steps[-1]
        smoke = {
            "left_money": score_from_state(env, 0),
            "right_money": score_from_state(env, 1),
            "left_status": str(final[0].status),
            "right_status": str(final[1].status),
        }
    else:
        smoke = play("Anchor V6", smoke_opponent, 12001)

    EVAL_AVAILABLE = (
        smoke["left_status"] == "DONE"
        and smoke["right_status"] == "DONE"
        and not (
            smoke["left_money"] == 0
            and smoke["right_money"] == 0
        )
    )
    print({"smoke": smoke, "evaluation_available": EVAL_AVAILABLE})
else:
    print({"evaluation_available": False, "reason": ENGINE_ERROR})


{'smoke': {'left_money': 155961.0, 'right_money': 96478.0, 'left_status': 'DONE', 'right_status': 'DONE'}, 'evaluation_available': True}


## 4. Verify downloaded replay controls exactly


In [5]:
replay_validation_rows = []

if EVAL_AVAILABLE and replay_controls:
    for episode_id, control in replay_controls.items():
        result = play(
            control["zero_name"]
            if control["zero_seat"] == 0
            else control["opponent_name"],
            control["opponent_name"]
            if control["zero_seat"] == 0
            else control["zero_name"],
            control["seed"],
        )
        reproduced = [
            result["left_money"],
            result["right_money"],
        ]
        expected = list(control["rewards"])
        exact = reproduced == expected

        replay_validation_rows.append({
            "episode_id": episode_id,
            "expected": expected,
            "reproduced": reproduced,
            "exact": exact,
            "statuses": [
                result["left_status"],
                result["right_status"],
            ],
        })

    if replay_validation_rows:
        print(json.dumps(replay_validation_rows, indent=2))
        replay_controls_valid = all(
            row["exact"]
            and row["statuses"] == ["DONE", "DONE"]
            for row in replay_validation_rows
        )
    else:
        replay_controls_valid = False
else:
    replay_controls_valid = False

print({
    "replay_controls_valid": replay_controls_valid,
    "validated": len(replay_validation_rows),
})


[
  {
    "episode_id": 89548972,
    "expected": [
      90753.0,
      87383.0
    ],
    "reproduced": [
      90753.0,
      87383.0
    ],
    "exact": true,
    "statuses": [
      "DONE",
      "DONE"
    ]
  },
  {
    "episode_id": 89549522,
    "expected": [
      136336.0,
      142069.0
    ],
    "reproduced": [
      136336.0,
      142069.0
    ],
    "exact": true,
    "statuses": [
      "DONE",
      "DONE"
    ]
  },
  {
    "episode_id": 89550073,
    "expected": [
      122505.0,
      132061.0
    ],
    "reproduced": [
      122505.0,
      132061.0
    ],
    "exact": true,
    "statuses": [
      "DONE",
      "DONE"
    ]
  },
  {
    "episode_id": 89550629,
    "expected": [
      136345.0,
      126607.0
    ],
    "reproduced": [
      136345.0,
      126607.0
    ],
    "exact": true,
    "statuses": [
      "DONE",
      "DONE"
    ]
  },
  {
    "episode_id": 89551188,
    "expected": [
      125907.0,
      125788.0
    ],
    "reproduced": [
      1259

## 5. Stage A — direct branch experiments against the anchor


In [6]:
ANCHOR_NAME = "Anchor V6"
STAGE_A_SEEDS = (12011, 12037, 12049)

if EVAL_AVAILABLE:
    stage_a_rows = []
    for candidate in CANDIDATE_SOURCES:
        if candidate == ANCHOR_NAME:
            continue
        for seed in STAGE_A_SEEDS:
            stage_a_rows.extend(
                symmetric_pair(candidate, ANCHOR_NAME, seed)
            )

    stage_a_summary = []
    for candidate in sorted({
        row["candidate"] for row in stage_a_rows
    }):
        group = [
            row for row in stage_a_rows
            if row["candidate"] == candidate
        ]
        margins = [row["margin"] for row in group]
        monies = [row["candidate_money"] for row in group]
        wins = sum(row["result"] == "win" for row in group)
        losses = sum(row["result"] == "loss" for row in group)

        stage_a_summary.append({
            "candidate": candidate,
            "games": len(group),
            "wins": wins,
            "losses": losses,
            "net_wins": wins - losses,
            "mean_money": round(statistics.mean(monies), 2),
            "minimum_money": min(monies),
            "mean_margin": round(statistics.mean(margins), 2),
            "minimum_margin": min(margins),
            "selection_score": round(
                (wins - losses) * 1_000_000
                + statistics.mean(margins),
                2,
            ),
        })

    stage_a_summary.sort(
        key=lambda row: row["selection_score"],
        reverse=True,
    )
    STAGE_A_FINALISTS = [
        row["candidate"] for row in stage_a_summary[:3]
    ]
    print(json.dumps(stage_a_summary, indent=2))
else:
    stage_a_rows = []
    stage_a_summary = []
    STAGE_A_FINALISTS = [DEFAULT_NAME]
    print({"stage_a": "SKIPPED"})


[
  {
    "candidate": "Clone Quad H1",
    "games": 6,
    "wins": 6,
    "losses": 0,
    "net_wins": 6,
    "mean_money": 116998.67,
    "minimum_money": 91211.0,
    "mean_margin": 1865.67,
    "minimum_margin": 1200.0,
    "selection_score": 6001865.67
  },
  {
    "candidate": "V24 Clone Cash H2",
    "games": 6,
    "wins": 6,
    "losses": 0,
    "net_wins": 6,
    "mean_money": 116763.67,
    "minimum_money": 91040.0,
    "mean_margin": 1573.33,
    "minimum_margin": 957.0,
    "selection_score": 6001573.33
  },
  {
    "candidate": "V24 Clone Cash Shield",
    "games": 6,
    "wins": 6,
    "losses": 0,
    "net_wins": 6,
    "mean_money": 116849.0,
    "minimum_money": 91066.0,
    "mean_margin": 1564.67,
    "minimum_margin": 868.0,
    "selection_score": 6001564.67
  },
  {
    "candidate": "Clone Premium H1",
    "games": 6,
    "wins": 6,
    "losses": 0,
    "net_wins": 6,
    "mean_money": 117367.0,
    "minimum_money": 91506.0,
    "mean_margin": 1287.0,
    "minimum_

## 6. Stage B — broad strong-agent preservation gate


In [7]:
CONTROL_NAMES = tuple(FALLBACK_SOURCES)
STAGE_B_SEEDS = (12101, 12119)

if EVAL_AVAILABLE and CONTROL_NAMES:
    tested = [ANCHOR_NAME, *STAGE_A_FINALISTS]
    stage_b_rows = []

    for candidate in tested:
        for opponent in CONTROL_NAMES:
            for seed in STAGE_B_SEEDS:
                stage_b_rows.extend(
                    symmetric_pair(candidate, opponent, seed)
                )

    stage_b_summary = []
    for candidate in tested:
        group = [
            row for row in stage_b_rows
            if row["candidate"] == candidate
        ]
        monies = [row["candidate_money"] for row in group]
        margins = [row["margin"] for row in group]
        stage_b_summary.append({
            "candidate": candidate,
            "games": len(group),
            "wins": sum(row["result"] == "win" for row in group),
            "losses": sum(row["result"] == "loss" for row in group),
            "mean_money": round(statistics.mean(monies), 2),
            "minimum_money": min(monies),
            "mean_margin": round(statistics.mean(margins), 2),
            "minimum_margin": min(margins),
        })
    print(json.dumps(stage_b_summary, indent=2))
else:
    stage_b_rows = []
    stage_b_summary = []


[
  {
    "candidate": "Anchor V6",
    "games": 20,
    "wins": 20,
    "losses": 0,
    "mean_money": 144531.55,
    "minimum_money": 117693.0,
    "mean_margin": 32632.75,
    "minimum_margin": 1775.0
  },
  {
    "candidate": "Clone Quad H1",
    "games": 20,
    "wins": 20,
    "losses": 0,
    "mean_money": 144531.55,
    "minimum_money": 117693.0,
    "mean_margin": 32632.75,
    "minimum_margin": 1775.0
  },
  {
    "candidate": "V24 Clone Cash H2",
    "games": 20,
    "wins": 20,
    "losses": 0,
    "mean_money": 144710.6,
    "minimum_money": 118008.0,
    "mean_margin": 32775.9,
    "minimum_margin": 2041.0
  },
  {
    "candidate": "V24 Clone Cash Shield",
    "games": 20,
    "wins": 20,
    "losses": 0,
    "mean_money": 144710.6,
    "minimum_money": 118008.0,
    "mean_margin": 32775.9,
    "minimum_margin": 2041.0
  }
]


## 7. Stage C — original public replay seeds and opponents


In [8]:
replay_candidate_rows = []

if EVAL_AVAILABLE and replay_controls_valid:
    tested = [ANCHOR_NAME, *STAGE_A_FINALISTS]

    for candidate in tested:
        for episode_id, control in replay_controls.items():
            if control["zero_seat"] == 0:
                result = play(
                    candidate,
                    control["opponent_name"],
                    control["seed"],
                )
                candidate_money = result["left_money"]
                opponent_money = result["right_money"]
            else:
                result = play(
                    control["opponent_name"],
                    candidate,
                    control["seed"],
                )
                candidate_money = result["right_money"]
                opponent_money = result["left_money"]

            replay_candidate_rows.append({
                "candidate": candidate,
                "episode_id": episode_id,
                "candidate_money": candidate_money,
                "opponent_money": opponent_money,
                "margin": candidate_money - opponent_money,
                "win": candidate_money > opponent_money,
                "status": (
                    result["left_status"],
                    result["right_status"],
                ),
            })

    replay_candidate_summary = []
    for candidate in tested:
        group = [
            row for row in replay_candidate_rows
            if row["candidate"] == candidate
        ]
        replay_candidate_summary.append({
            "candidate": candidate,
            "games": len(group),
            "wins": sum(row["win"] for row in group),
            "mean_money": round(
                statistics.mean(
                    row["candidate_money"] for row in group
                ),
                2,
            ),
            "minimum_money": min(
                row["candidate_money"] for row in group
            ),
            "mean_margin": round(
                statistics.mean(row["margin"] for row in group),
                2,
            ),
            "minimum_margin": min(row["margin"] for row in group),
        })
    print(json.dumps(replay_candidate_summary, indent=2))
else:
    replay_candidate_summary = []


[
  {
    "candidate": "Anchor V6",
    "games": 8,
    "wins": 7,
    "mean_money": 125127.38,
    "minimum_money": 90753.0,
    "mean_margin": 4202.5,
    "minimum_margin": -8039.0
  },
  {
    "candidate": "Clone Quad H1",
    "games": 8,
    "wins": 7,
    "mean_money": 125198.75,
    "minimum_money": 90753.0,
    "mean_margin": 4528.5,
    "minimum_margin": -7837.0
  },
  {
    "candidate": "V24 Clone Cash H2",
    "games": 8,
    "wins": 7,
    "mean_money": 125130.75,
    "minimum_money": 90738.0,
    "mean_margin": 4356.38,
    "minimum_margin": -7676.0
  },
  {
    "candidate": "V24 Clone Cash Shield",
    "games": 8,
    "wins": 7,
    "mean_money": 125134.75,
    "minimum_money": 90738.0,
    "mean_margin": 4338.75,
    "minimum_margin": -7729.0
  }
]


## 8. Safe promotion decision


In [9]:
def summary_by_name(rows, name):
    return next(
        (row for row in rows if row["candidate"] == name),
        None,
    )

if EVAL_AVAILABLE:
    anchor_a = {
        "wins": 0,
        "mean_money": 0,
        "minimum_money": 0,
        "mean_margin": 0,
    }
    anchor_b = summary_by_name(stage_b_summary, ANCHOR_NAME)
    anchor_c = summary_by_name(
        replay_candidate_summary,
        ANCHOR_NAME,
    )

    eligible = []
    for name in STAGE_A_FINALISTS:
        direct = summary_by_name(stage_a_summary, name)
        broad = summary_by_name(stage_b_summary, name)
        replay = summary_by_name(replay_candidate_summary, name)

        direct_ok = (
            direct is not None
            and direct["net_wins"] > 0
            and direct["mean_margin"] > 0
        )

        broad_ok = (
            broad is None
            or anchor_b is None
            or (
                broad["wins"] >= anchor_b["wins"]
                and broad["mean_money"] >= anchor_b["mean_money"]
                and broad["minimum_money"]
                    >= anchor_b["minimum_money"] * 0.97
            )
        )

        replay_ok = (
            replay is None
            or anchor_c is None
            or (
                replay["wins"] >= anchor_c["wins"]
                and replay["mean_money"] >= anchor_c["mean_money"]
                and replay["minimum_money"]
                    >= anchor_c["minimum_money"] * 0.98
            )
        )

        if direct_ok and broad_ok and replay_ok:
            score = (
                direct["net_wins"] * 1_000_000
                + direct["mean_margin"]
                + (broad["mean_money"] if broad else 0)
                + (replay["mean_money"] if replay else 0)
            )
            eligible.append((score, name))

    if eligible:
        eligible.sort(reverse=True)
        SELECTED_NAME = eligible[0][1]
    else:
        SELECTED_NAME = ANCHOR_NAME
else:
    SELECTED_NAME = ANCHOR_NAME

SELECTED_SOURCE = CANDIDATE_SOURCES[SELECTED_NAME]

print({
    "selected": SELECTED_NAME,
    "fallback_to_anchor": SELECTED_NAME == ANCHOR_NAME,
})


{'selected': 'Clone Quad H1', 'fallback_to_anchor': False}


## 9. Package the verified winner


In [10]:
MAIN_PATH.write_text(SELECTED_SOURCE, encoding="utf-8")
ast.parse(SELECTED_SOURCE)
compile(SELECTED_SOURCE, str(MAIN_PATH), "exec")

raw = io.BytesIO()
with gzip.GzipFile(
    fileobj=raw,
    mode="wb",
    filename="",
    mtime=0,
) as zipped:
    with tarfile.open(fileobj=zipped, mode="w") as archive:
        payload = SELECTED_SOURCE.encode("utf-8")
        info = tarfile.TarInfo("main.py")
        info.size = len(payload)
        info.mode = 0o644
        info.mtime = 0
        info.uid = info.gid = 0
        info.uname = info.gname = ""
        archive.addfile(info, io.BytesIO(payload))

ARCHIVE_PATH.write_bytes(raw.getvalue())

with tarfile.open(ARCHIVE_PATH, "r:gz") as archive:
    assert archive.getnames() == ["main.py"]
    archived = archive.extractfile("main.py").read().decode("utf-8")
assert archived == SELECTED_SOURCE

print({
    "selected": SELECTED_NAME,
    "main_path": str(MAIN_PATH),
    "submission": str(ARCHIVE_PATH),
    "submission_bytes": ARCHIVE_PATH.stat().st_size,
    "main_sha256": hashlib.sha256(
        SELECTED_SOURCE.encode("utf-8")
    ).hexdigest(),
    "archive_sha256": hashlib.sha256(
        ARCHIVE_PATH.read_bytes()
    ).hexdigest(),
})


{'selected': 'Clone Quad H1', 'main_path': '/kaggle/working/main.py', 'submission': '/kaggle/working/submission.tar.gz', 'submission_bytes': 13002, 'main_sha256': '74ee398bdbb4e06837b2effc7da0c0c53da31f802feea3fba42bc5ed15c76ab0', 'archive_sha256': '3206e07ead9539dc08d58fc6750cf37028b4b385cde93402cef5049efd4873c1'}


## Output

Run all cells and submit:

`/kaggle/working/submission.tar.gz`

The notebook downloads the eight known rank-one replay controls when the public
CDN is reachable. Add newer public episode IDs to `EXTRA_EPISODE_IDS`.

A branch is packaged only if it beats the exact uploaded anchor head-to-head and
does not reduce broad wins, average money, replay-control money, or robust
minimums. Otherwise the exact uploaded strategy is packaged unchanged.
